# Deep-Live-Cam Remote — Colab batch processor

Self-contained, path-based video batch face swap. No upload widget, Gradio, FastRTC, ZMQ, or Tailscale.

## 1. Install dependencies and restore the bundled engine

In [ ]:
# Runtime source bundle: generated from the sibling project files.
_RUNTIME_FILES = {
    'colab_batch.py': '"""Colab-native folder batch processor for the modern Deep-Live-Cam engine.\n\nAll media paths are paths already visible to the Colab runtime.  FFmpeg handles\nseek, FPS capping, resize, raw-frame transport, audio muxing, and final encode;\nPython only performs face analysis and inference.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport queue\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom dataclasses import asdict, dataclass\nfrom fractions import Fraction\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport cv2\nimport numpy as np\n\nVIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v", ".mpeg", ".mpg"}\nMANIFEST_NAME = ".deep_live_cam_processed.json"\nREPORT_NAME = "batch_report.json"\nENGINE_VERSION = "deep-live-cam-remote-v1"\n\n\n@dataclass(frozen=True)\nclass ProcessConfig:\n    input_dir: Path\n    output_dir: Path\n    source_face: Path | None\n    map_config: Path | None\n    ss: float = 0.0\n    duration: float | None = None\n    max_fps: float = 30.0\n    max_width: int = 420\n    decode_queue: int = 6\n    encode_queue: int = 6\n    recursive: bool = True\n    overwrite: bool = False\n    skip_processed: bool = True\n    short_video_policy: str = "start"\n    cuda_decode: bool = True\n    encoder: str = "auto"\n    preset: str = "p4"\n    quality: int = 18\n    many_faces: bool = False\n    opacity: float = 1.0\n    sharpness: float = 0.0\n    mouth_mask_size: float = 0.0\n    poisson_blend: bool = False\n    color_correction: bool = False\n    interpolation_weight: float = 0.0\n    enhancer: str = "none"\n\n\ndef run(command: list[str], **kwargs: Any) -> subprocess.CompletedProcess:\n    return subprocess.run(command, check=False, text=True, **kwargs)\n\n\ndef parse_fraction(value: str | None) -> float:\n    if not value or value in {"0/0", "N/A"}:\n        return 0.0\n    try:\n        return float(Fraction(value))\n    except (ValueError, ZeroDivisionError):\n        return 0.0\n\n\ndef probe_video(path: Path) -> dict[str, Any]:\n    result = run([\n        "ffprobe", "-v", "error", "-select_streams", "v:0",\n        "-show_entries", "stream=width,height,avg_frame_rate,r_frame_rate,nb_frames,duration",\n        "-show_entries", "format=duration", "-of", "json", str(path),\n    ], capture_output=True)\n    if result.returncode:\n        raise RuntimeError(f"ffprobe failed for {path}:\\n{result.stderr[-4000:]}")\n    payload = json.loads(result.stdout)\n    if not payload.get("streams"):\n        raise RuntimeError(f"No video stream found: {path}")\n    stream = payload["streams"][0]\n    fps = parse_fraction(stream.get("avg_frame_rate")) or parse_fraction(stream.get("r_frame_rate")) or 25.0\n    duration_value = stream.get("duration") or payload.get("format", {}).get("duration")\n    try:\n        duration = float(duration_value)\n    except (TypeError, ValueError):\n        duration = None\n    return {\n        "width": int(stream.get("width") or 0),\n        "height": int(stream.get("height") or 0),\n        "fps": fps,\n        "duration": duration,\n        "frames": int(stream["nb_frames"]) if str(stream.get("nb_frames", "")).isdigit() else None,\n    }\n\n\ndef processing_geometry(width: int, height: int, source_fps: float, max_width: int, max_fps: float) -> tuple[int, int, float]:\n    if width <= 0 or height <= 0:\n        raise ValueError(f"Invalid video geometry: {width}x{height}")\n    scale = min(1.0, max_width / width)\n    out_width = max(2, int(width * scale) // 2 * 2)\n    out_height = max(2, int(round(height * out_width / width / 2.0)) * 2)\n    return out_width, out_height, min(source_fps, max_fps)\n\n\ndef discover_videos(root: Path, recursive: bool = True) -> list[Path]:\n    iterator = root.rglob("*") if recursive else root.glob("*")\n    return sorted(path for path in iterator if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS)\n\n\ndef read_exact(stream: Any, size: int) -> bytes:\n    data = bytearray()\n    while len(data) < size:\n        chunk = stream.read(size - len(data))\n        if not chunk:\n            return b""\n        data.extend(chunk)\n    return bytes(data)\n\n\ndef file_hash(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef input_fingerprint(path: Path, root: Path) -> dict[str, Any]:\n    stat = path.stat()\n    return {"path": path.relative_to(root).as_posix(), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}\n\n\ndef config_signature(config: ProcessConfig) -> str:\n    ignored = {"input_dir", "output_dir", "overwrite", "skip_processed", "decode_queue", "encode_queue"}\n    payload = {key: str(value) if isinstance(value, Path) else value for key, value in asdict(config).items() if key not in ignored}\n    payload["engine"] = ENGINE_VERSION\n    if config.source_face and config.source_face.is_file():\n        payload["source_face_sha256"] = file_hash(config.source_face)\n    if config.map_config and config.map_config.is_file():\n        payload["map_config_sha256"] = file_hash(config.map_config)\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n\ndef manifest_key(path: Path, root: Path, signature: str) -> str:\n    return hashlib.sha256(json.dumps({"input": input_fingerprint(path, root), "config": signature}, sort_keys=True).encode()).hexdigest()\n\n\ndef load_json(path: Path, default: Any) -> Any:\n    if not path.is_file():\n        return default\n    try:\n        return json.loads(path.read_text(encoding="utf-8"))\n    except (OSError, json.JSONDecodeError):\n        return default\n\n\ndef atomic_json(path: Path, payload: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef ffmpeg_has_encoder(name: str) -> bool:\n    result = run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True)\n    return result.returncode == 0 and name in result.stdout\n\n\ndef decoder_command(path: Path, cuda: bool, start: float, duration: float | None, fps: float, width: int, height: int) -> list[str]:\n    command = ["ffmpeg", "-hide_banner", "-loglevel", "error"]\n    if cuda:\n        command += ["-hwaccel", "cuda"]\n    if start > 0:\n        command += ["-ss", f"{start:.6f}"]\n    command += ["-i", str(path)]\n    if duration is not None:\n        command += ["-t", f"{duration:.6f}"]\n    command += [\n        "-map", "0:v:0", "-an", "-sn", "-dn",\n        "-vf", f"fps={fps:.12g},scale={width}:{height}",\n        "-vsync", "0", "-f", "rawvideo", "-pix_fmt", "bgr24", "pipe:1",\n    ]\n    return command\n\n\ndef encoder_command(path: Path, output: Path, start: float, duration: float, fps: float, width: int, height: int, encoder: str, preset: str, quality: int) -> list[str]:\n    command = [\n        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",\n        "-f", "rawvideo", "-pix_fmt", "bgr24", "-video_size", f"{width}x{height}",\n        "-framerate", f"{fps:.12g}", "-i", "pipe:0",\n    ]\n    if start > 0:\n        command += ["-ss", f"{start:.6f}"]\n    command += ["-t", f"{duration:.6f}", "-i", str(path), "-map", "0:v:0", "-map", "1:a:0?", "-map_metadata", "1"]\n    if encoder == "h264_nvenc":\n        command += ["-c:v", encoder, "-preset", preset, "-cq", str(quality)]\n    else:\n        command += ["-c:v", "libx264", "-preset", "medium", "-crf", str(quality)]\n    command += ["-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k", "-shortest", "-movflags", "+faststart", str(output)]\n    return command\n\n\nclass ModernEngine:\n    def __init__(self, config: ProcessConfig):\n        import modules.globals as globals_module\n        from modules.face_analyser import get_many_faces, get_one_face\n        from modules.processors.frame import face_swapper\n\n        self.globals = globals_module\n        self.get_one_face = get_one_face\n        self.get_many_faces = get_many_faces\n        self.swapper = face_swapper\n        self.source_cache: dict[str, Any] = {}\n        self.mapping = self._load_mapping(config.map_config)\n        self.default_source = self._source(config.source_face) if config.source_face else None\n        self.enhancer = self._load_enhancer(config.enhancer)\n\n        globals_module.execution_providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]\n        globals_module.many_faces = config.many_faces and not self.mapping\n        globals_module.map_faces = bool(self.mapping)\n        globals_module.opacity = config.opacity\n        globals_module.sharpness = config.sharpness\n        globals_module.mouth_mask_size = config.mouth_mask_size\n        globals_module.mouth_mask = config.mouth_mask_size > 0\n        globals_module.poisson_blend = config.poisson_blend\n        globals_module.color_correction = config.color_correction\n        globals_module.enable_interpolation = 0 < config.interpolation_weight < 1\n        globals_module.interpolation_weight = config.interpolation_weight\n\n        if self.default_source is None and not self.mapping:\n            raise ValueError("--source-face is required when --map-config is not supplied")\n\n    def _source(self, path: Path | None) -> Any:\n        if path is None:\n            return None\n        key = str(path.resolve())\n        if key not in self.source_cache:\n            image = cv2.imread(str(path))\n            if image is None:\n                raise ValueError(f"Could not read source image: {path}")\n            face = self.get_one_face(image)\n            if face is None:\n                raise ValueError(f"No face detected in source image: {path}")\n            self.source_cache[key] = face\n        return self.source_cache[key]\n\n    def _load_mapping(self, path: Path | None) -> dict[str, list[dict[str, Any]]]:\n        if path is None:\n            return {}\n        payload = load_json(path, {})\n        if payload.get("version") != 1 or not isinstance(payload.get("videos"), dict):\n            raise ValueError("Mapping JSON must contain version=1 and a videos object")\n        base = path.parent\n        mapping: dict[str, list[dict[str, Any]]] = {}\n        for video, identities in payload["videos"].items():\n            mapping[video] = []\n            for identity in identities:\n                source = identity.get("source_path")\n                centroid = np.asarray(identity.get("centroid", []), dtype=np.float32)\n                if source and centroid.size:\n                    source_path = Path(source)\n                    if not source_path.is_absolute():\n                        source_path = base / source_path\n                    centroid /= max(float(np.linalg.norm(centroid)), 1e-8)\n                    mapping[video].append({**identity, "source_path": source_path, "centroid_array": centroid})\n        return mapping\n\n    @staticmethod\n    def _load_enhancer(name: str) -> Any:\n        if name == "none":\n            return None\n        module_names = {\n            "gfpgan": "modules.processors.frame.face_enhancer",\n            "gpen256": "modules.processors.frame.face_enhancer_gpen256",\n            "gpen512": "modules.processors.frame.face_enhancer_gpen512",\n        }\n        module = __import__(module_names[name], fromlist=["process_frame"])\n        if hasattr(module, "pre_check") and not module.pre_check():\n            raise RuntimeError(f"Enhancer pre-check failed: {name}")\n        return module\n\n    def reset_video_state(self) -> None:\n        if hasattr(self.swapper, "PREVIOUS_FRAME_RESULT"):\n            self.swapper.PREVIOUS_FRAME_RESULT = None\n        if hasattr(self.swapper, "FACE_DETECTION_CACHE"):\n            self.swapper.FACE_DETECTION_CACHE.clear()\n\n    def process(self, frame: np.ndarray, relative_video: str) -> np.ndarray:\n        if self.mapping:\n            output = frame.copy()\n            faces = self.get_many_faces(frame) or []\n            entries = self.mapping.get(relative_video, [])\n            bboxes = []\n            for target in faces:\n                embedding = np.asarray(getattr(target, "normed_embedding", target.embedding), dtype=np.float32)\n                embedding /= max(float(np.linalg.norm(embedding)), 1e-8)\n                match = max(entries, key=lambda item: float(np.dot(embedding, item["centroid_array"])), default=None)\n                if match and float(np.dot(embedding, match["centroid_array"])) >= float(match.get("threshold", 0.35)):\n                    output = self.swapper.swap_face(self._source(match["source_path"]), target, output)\n                    bboxes.append(target.bbox.astype(int))\n                elif self.default_source is not None:\n                    output = self.swapper.swap_face(self.default_source, target, output)\n                    bboxes.append(target.bbox.astype(int))\n            output = self.swapper.apply_post_processing(output, bboxes)\n            detected = faces\n        else:\n            detected = self.get_many_faces(frame) if self.globals.many_faces else None\n            output = self.swapper.process_frame(self.default_source, frame)\n        if self.enhancer:\n            output = self.enhancer.process_frame(None, output, detected_faces=detected)\n        return output\n\n\ndef effective_segment(info: dict[str, Any], config: ProcessConfig, path: Path) -> tuple[float, float | None]:\n    start = config.ss\n    duration = info["duration"]\n    if duration is not None and start >= duration:\n        if config.short_video_policy == "start":\n            print(f"  ! shorter than SS={start:g}; using SS=0")\n            start = 0.0\n        else:\n            raise ValueError(f"SS={start} is beyond the end of {path.name}")\n    remaining = None if duration is None else max(0.0, duration - start)\n    clip = remaining if config.duration is None else config.duration if remaining is None else min(config.duration, remaining)\n    if clip is not None and clip <= 0:\n        raise ValueError("No video remains after seek")\n    return start, clip\n\n\ndef _start_decoder(path: Path, config: ProcessConfig, start: float, clip: float | None, fps: float, width: int, height: int, cuda: bool) -> subprocess.Popen:\n    return subprocess.Popen(decoder_command(path, cuda, start, clip, fps, width, height), stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=10**8)\n\n\ndef process_one(path: Path, output: Path, relative: str, config: ProcessConfig, engine: ModernEngine) -> dict[str, Any]:\n    info = probe_video(path)\n    width, height, fps = processing_geometry(info["width"], info["height"], info["fps"], config.max_width, config.max_fps)\n    start, clip = effective_segment(info, config, path)\n    expected = max(1, int(round(clip * fps))) if clip is not None else None\n    encode_duration = clip or max(1 / fps, ((info["frames"] or 86400 * info["fps"]) / info["fps"]) - start)\n    frame_size = width * height * 3\n    engine.reset_video_state()\n    print(f"  {info[\'width\']}x{info[\'height\']} @ {info[\'fps\']:.3f} -> {width}x{height} @ {fps:.3f}")\n\n    decoder = _start_decoder(path, config, start, clip, fps, width, height, config.cuda_decode)\n    first = read_exact(decoder.stdout, frame_size)\n    if not first and config.cuda_decode:\n        error = decoder.stderr.read().decode(errors="replace")\n        decoder.wait()\n        print("  ! CUDA decode unavailable; using software decode")\n        if error.strip():\n            print("    " + error.strip().splitlines()[-1])\n        decoder = _start_decoder(path, config, start, clip, fps, width, height, False)\n        first = read_exact(decoder.stdout, frame_size)\n    if not first:\n        error = decoder.stderr.read().decode(errors="replace")\n        decoder.wait()\n        raise RuntimeError("FFmpeg produced no frames:\\n" + error[-4000:])\n\n    selected_encoder = config.encoder\n    if selected_encoder == "auto":\n        selected_encoder = "h264_nvenc" if ffmpeg_has_encoder("h264_nvenc") else "libx264"\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.unlink(missing_ok=True)\n    encoder = subprocess.Popen(encoder_command(path, output, start, encode_duration, fps, width, height, selected_encoder, config.preset, config.quality), stdin=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=10**8)\n\n    decoded: queue.Queue[Any] = queue.Queue(config.decode_queue)\n    encoded: queue.Queue[Any] = queue.Queue(config.encode_queue)\n    errors: queue.Queue[tuple[str, BaseException]] = queue.Queue()\n    sentinel = object()\n    stop = threading.Event()\n\n    def decoder_worker() -> None:\n        try:\n            raw = first\n            while raw and not stop.is_set():\n                decoded.put(raw)\n                raw = read_exact(decoder.stdout, frame_size)\n            decoded.put(sentinel)\n        except BaseException as exc:\n            errors.put(("decode", exc))\n            try: decoded.put(sentinel, timeout=1)\n            except queue.Full: pass\n\n    def encoder_worker() -> None:\n        try:\n            while True:\n                raw = encoded.get()\n                if raw is sentinel:\n                    return\n                encoder.stdin.write(raw)\n        except BaseException as exc:\n            errors.put(("encode", exc))\n\n    decode_thread = threading.Thread(target=decoder_worker, daemon=True)\n    encode_thread = threading.Thread(target=encoder_worker, daemon=True)\n    decode_thread.start(); encode_thread.start()\n    frames = fallbacks = 0\n    started = time.monotonic()\n    try:\n        while True:\n            if not errors.empty():\n                stage, exc = errors.get()\n                raise RuntimeError(f"{stage} pipeline failed: {exc}") from exc\n            raw = decoded.get(timeout=30)\n            if raw is sentinel:\n                break\n            frame = np.frombuffer(raw, np.uint8).reshape(height, width, 3)\n            try:\n                result = engine.process(frame, relative)\n                if result is None:\n                    result = frame; fallbacks += 1\n            except Exception as exc:\n                result = frame; fallbacks += 1\n                if fallbacks <= 3:\n                    print(f"  ! frame fallback: {exc}")\n            if result.shape[:2] != (height, width):\n                result = cv2.resize(result, (width, height))\n            encoded.put(np.ascontiguousarray(result).tobytes())\n            frames += 1\n            if frames % 30 == 0 or frames == expected:\n                suffix = f"/{expected}" if expected else ""\n                print(f"\\r  frames {frames}{suffix}", end="", flush=True)\n            if expected and frames >= expected:\n                stop.set(); break\n        print()\n        encoded.put(sentinel)\n        encode_thread.join(timeout=60)\n        encoder.stdin.close(); encoder.stdin = None\n        if stop.is_set() and decoder.poll() is None:\n            decoder.terminate()\n        decoder.wait(timeout=20)\n        rc = encoder.wait(timeout=120)\n        error = encoder.stderr.read().decode(errors="replace")\n        if rc:\n            raise RuntimeError("FFmpeg encode failed:\\n" + error[-4000:])\n    finally:\n        stop.set()\n        for process in (decoder, encoder):\n            if process.poll() is None:\n                process.terminate()\n                try: process.wait(timeout=5)\n                except subprocess.TimeoutExpired: process.kill()\n    if not output.is_file() or output.stat().st_size == 0:\n        raise RuntimeError(f"Output not created: {output}")\n    return {"frames": frames, "fallback_frames": fallbacks, "fps": fps, "width": width, "height": height, "seconds": time.monotonic() - started, "size_mb": output.stat().st_size / 1048576}\n\n\ndef scan_identities(args: argparse.Namespace) -> int:\n    import modules.globals as globals_module\n    from modules.face_analyser import get_many_faces\n    globals_module.execution_providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]\n    input_dir, mapping_dir = Path(args.input_dir), Path(args.mapping_dir)\n    mapping_dir.mkdir(parents=True, exist_ok=True)\n    payload: dict[str, Any] = {"version": 1, "instructions": "Set source_path for each identity, then pass this file to process --map-config.", "videos": {}}\n    for video in discover_videos(input_dir, args.recursive):\n        relative = video.relative_to(input_dir).as_posix()\n        capture = cv2.VideoCapture(str(video))\n        fps = capture.get(cv2.CAP_PROP_FPS) or 25.0\n        every = max(1, int(round(fps * args.sample_seconds)))\n        samples: list[dict[str, Any]] = []\n        index = 0\n        while len(samples) < args.max_samples:\n            ok, frame = capture.read()\n            if not ok: break\n            if index % every == 0:\n                for face in get_many_faces(frame) or []:\n                    vector = np.asarray(getattr(face, "normed_embedding", face.embedding), np.float32)\n                    vector /= max(float(np.linalg.norm(vector)), 1e-8)\n                    bbox = np.asarray(face.bbox, int)\n                    x1, y1, x2, y2 = np.maximum(bbox, 0)\n                    crop = frame[y1:y2, x1:x2].copy()\n                    if crop.size: samples.append({"embedding": vector, "crop": crop})\n            index += 1\n        capture.release()\n        clusters: list[dict[str, Any]] = []\n        for sample in samples:\n            match = max(clusters, key=lambda item: float(np.dot(sample["embedding"], item["centroid"])), default=None)\n            if match is None or float(np.dot(sample["embedding"], match["centroid"])) < args.cluster_threshold:\n                clusters.append({"centroid": sample["embedding"].copy(), "count": 1, "crop": sample["crop"]})\n            else:\n                match["count"] += 1\n                match["centroid"] += (sample["embedding"] - match["centroid"]) / match["count"]\n                match["centroid"] /= max(float(np.linalg.norm(match["centroid"])), 1e-8)\n        identities = []\n        thumbs = []\n        stem = hashlib.sha1(relative.encode()).hexdigest()[:10]\n        for number, cluster in enumerate(sorted(clusters, key=lambda item: item["count"], reverse=True), 1):\n            thumb_name = f"{stem}_identity_{number:02d}.jpg"\n            cv2.imwrite(str(mapping_dir / thumb_name), cluster["crop"])\n            identities.append({"target_id": number, "samples": cluster["count"], "thumbnail": thumb_name, "source_path": "", "threshold": args.match_threshold, "centroid": cluster["centroid"].tolist()})\n            thumb = cv2.resize(cluster["crop"], (180, 180)); cv2.putText(thumb, f"ID {number}", (8, 24), cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 255, 0), 2); thumbs.append(thumb)\n        if thumbs:\n            columns = min(4, len(thumbs)); rows = math.ceil(len(thumbs) / columns)\n            sheet = np.zeros((rows * 180, columns * 180, 3), np.uint8)\n            for i, thumb in enumerate(thumbs): sheet[(i // columns)*180:(i // columns+1)*180, (i % columns)*180:(i % columns+1)*180] = thumb\n            cv2.imwrite(str(mapping_dir / f"{stem}_contact_sheet.jpg"), sheet)\n        payload["videos"][relative] = identities\n        print(f"{relative}: {len(identities)} identities from {len(samples)} samples")\n    output = mapping_dir / "face_mapping.json"\n    atomic_json(output, payload)\n    print(f"Mapping template: {output}")\n    return 0\n\n\ndef process_batch(args: argparse.Namespace) -> int:\n    config = ProcessConfig(\n        input_dir=Path(args.input_dir), output_dir=Path(args.output_dir),\n        source_face=Path(args.source_face) if args.source_face else None,\n        map_config=Path(args.map_config) if args.map_config else None,\n        ss=args.ss, duration=args.duration, max_fps=args.max_fps, max_width=args.max_width,\n        decode_queue=args.decode_queue, encode_queue=args.encode_queue, recursive=args.recursive,\n        overwrite=args.overwrite, skip_processed=args.skip_processed,\n        short_video_policy=args.short_video_policy, cuda_decode=args.cuda_decode,\n        encoder=args.encoder, preset=args.preset, quality=args.quality, many_faces=args.many_faces,\n        opacity=args.opacity, sharpness=args.sharpness, mouth_mask_size=args.mouth_mask_size,\n        poisson_blend=args.poisson_blend, color_correction=args.color_correction,\n        interpolation_weight=args.interpolation_weight, enhancer=args.enhancer,\n    )\n    if not config.input_dir.is_dir(): raise NotADirectoryError(config.input_dir)\n    if config.source_face and not config.source_face.is_file(): raise FileNotFoundError(config.source_face)\n    if config.map_config and not config.map_config.is_file(): raise FileNotFoundError(config.map_config)\n    config.output_dir.mkdir(parents=True, exist_ok=True)\n    videos = discover_videos(config.input_dir, config.recursive)\n    if not videos: raise FileNotFoundError(f"No videos found in {config.input_dir}")\n    engine = ModernEngine(config)\n    signature = config_signature(config)\n    manifest_path = config.output_dir / MANIFEST_NAME\n    manifest = load_json(manifest_path, {"version": 1, "items": {}})\n    items = manifest.setdefault("items", {})\n    report: dict[str, Any] = {"engine": ENGINE_VERSION, "config_signature": signature, "completed": [], "skipped": [], "failed": []}\n    suffix = f"_ss{config.ss:g}" if config.ss else ""\n    if config.duration is not None: suffix += f"_dur{config.duration:g}"\n    for number, video in enumerate(videos, 1):\n        relative = video.relative_to(config.input_dir).as_posix()\n        output = config.output_dir / Path(relative).parent / f"{video.stem}_face_swapped{suffix}.mp4"\n        key = manifest_key(video, config.input_dir, signature)\n        print(f"\\n[{number}/{len(videos)}] {relative}")\n        if not config.overwrite and config.skip_processed and key in items and Path(items[key].get("output", "")).is_file():\n            print("  skipped: matching input + source/model/config manifest entry")\n            report["skipped"].append(relative); continue\n        try:\n            result = process_one(video, output, relative, config, engine)\n            record = {"input": relative, "output": str(output), **result}\n            report["completed"].append(record)\n            items[key] = {**record, "completed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n            atomic_json(manifest_path, manifest)\n            print(f"  done: {output} ({result[\'size_mb\']:.1f} MB)")\n        except Exception as exc:\n            output.unlink(missing_ok=True)\n            report["failed"].append({"input": relative, "error": str(exc)})\n            print(f"  FAILED: {exc}")\n    report_path = config.output_dir / REPORT_NAME\n    atomic_json(report_path, report)\n    if args.zip_output:\n        zip_path = Path(args.zip_output)\n        zip_path.parent.mkdir(parents=True, exist_ok=True)\n        created = shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=config.output_dir)\n        print(f"ZIP: {created}")\n    print(f"\\nCompleted {len(report[\'completed\'])}; skipped {len(report[\'skipped\'])}; failed {len(report[\'failed\'])}")\n    return 1 if report["failed"] else 0\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=__doc__)\n    subparsers = parser.add_subparsers(dest="command", required=True)\n    scan = subparsers.add_parser("scan", help="scan target identities and write contact sheets + editable JSON")\n    scan.add_argument("--input-dir", required=True); scan.add_argument("--mapping-dir", required=True)\n    scan.add_argument("--sample-seconds", type=float, default=1.0); scan.add_argument("--max-samples", type=int, default=300)\n    scan.add_argument("--cluster-threshold", type=float, default=0.55); scan.add_argument("--match-threshold", type=float, default=0.35)\n    scan.add_argument("--recursive", action=argparse.BooleanOptionalAction, default=True); scan.set_defaults(func=scan_identities)\n    process = subparsers.add_parser("process", help="process every input video through the modern engine")\n    process.add_argument("--source-face"); process.add_argument("--input-dir", required=True); process.add_argument("--output-dir", required=True)\n    process.add_argument("--map-config"); process.add_argument("--zip-output")\n    process.add_argument("--ss", type=float, default=0.0); process.add_argument("--duration", type=float)\n    process.add_argument("--max-fps", type=float, default=30.0); process.add_argument("--max-width", type=int, default=420)\n    process.add_argument("--decode-queue", type=int, default=6); process.add_argument("--encode-queue", type=int, default=6)\n    process.add_argument("--recursive", action=argparse.BooleanOptionalAction, default=True)\n    process.add_argument("--overwrite", action=argparse.BooleanOptionalAction, default=False)\n    process.add_argument("--skip-processed", action=argparse.BooleanOptionalAction, default=True)\n    process.add_argument("--short-video-policy", choices=["start", "skip"], default="start")\n    process.add_argument("--cuda-decode", action=argparse.BooleanOptionalAction, default=True)\n    process.add_argument("--encoder", choices=["auto", "h264_nvenc", "libx264"], default="auto")\n    process.add_argument("--preset", default="p4"); process.add_argument("--quality", type=int, default=18)\n    process.add_argument("--many-faces", action=argparse.BooleanOptionalAction, default=False)\n    process.add_argument("--opacity", type=float, default=1.0); process.add_argument("--sharpness", type=float, default=0.0)\n    process.add_argument("--mouth-mask-size", type=float, default=0.0)\n    process.add_argument("--poisson-blend", action=argparse.BooleanOptionalAction, default=False)\n    process.add_argument("--color-correction", action=argparse.BooleanOptionalAction, default=False)\n    process.add_argument("--interpolation-weight", type=float, default=0.0)\n    process.add_argument("--enhancer", choices=["none", "gfpgan", "gpen256", "gpen512"], default="none")\n    process.set_defaults(func=process_batch)\n    return parser\n\n\ndef main(argv: list[str] | None = None) -> int:\n    args = build_parser().parse_args(argv)\n    if getattr(args, "ss", 0) < 0: raise ValueError("--ss must be non-negative")\n    if getattr(args, "duration", None) is not None and args.duration <= 0: raise ValueError("--duration must be positive")\n    if getattr(args, "max_fps", 1) <= 0 or getattr(args, "max_width", 1) <= 0: raise ValueError("FPS and width limits must be positive")\n    if not 0 <= getattr(args, "opacity", 1) <= 1: raise ValueError("--opacity must be between 0 and 1")\n    return args.func(args)\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'modules/__init__.py': 'import os\nimport cv2\nimport numpy as np\n\n\n# Utility function to support unicode characters in file paths for reading.\n# OpenCV\'s cv2.imread() encodes the path with the locale ANSI code page on\n# Windows, so it silently returns None for paths containing non-ASCII\n# characters (Chinese, Japanese, Cyrillic, accents, ...). Reading the bytes\n# through NumPy (which uses Python\'s unicode-aware file I/O) and decoding them\n# in memory sidesteps that limitation. Returns None on failure, matching\n# cv2.imread() so it stays a drop-in replacement.\ndef imread_unicode(path, flags=cv2.IMREAD_COLOR):\n    try:\n        data = np.fromfile(path, dtype=np.uint8)\n        if data.size == 0:\n            return None\n        return cv2.imdecode(data, flags)\n    except Exception:\n        return None\n\n\n# Utility function to support unicode characters in file paths for writing.\n# cv2.imwrite() has the same ANSI-path limitation, so we encode the image in\n# memory and write the bytes out with NumPy\'s unicode-aware file I/O. Returns\n# True/False like cv2.imwrite() so it stays a drop-in replacement.\ndef imwrite_unicode(path, img, params=None):\n    try:\n        root, ext = os.path.splitext(path)\n        if not ext:\n            ext = ".png"\n        result, encoded_img = cv2.imencode(ext, img, params if params is not None else [])\n        if not result:\n            return False\n        encoded_img.tofile(path)\n        return True\n    except Exception:\n        return False\n',
    'modules/capturer.py': "from typing import Any\nimport cv2\nimport modules.globals  # Import the globals to check the color correction toggle\nfrom modules.gpu_processing import gpu_cvt_color\n\n\ndef get_video_frame(video_path: str, frame_number: int = 0) -> Any:\n    capture = cv2.VideoCapture(video_path)\n\n    # Set MJPEG format to ensure correct color space handling\n    capture.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))\n    \n    # Only force RGB conversion if color correction is enabled\n    if modules.globals.color_correction:\n        capture.set(cv2.CAP_PROP_CONVERT_RGB, 1)\n    \n    frame_total = capture.get(cv2.CAP_PROP_FRAME_COUNT)\n    capture.set(cv2.CAP_PROP_POS_FRAMES, min(frame_total, frame_number - 1))\n    has_frame, frame = capture.read()\n\n    if has_frame and modules.globals.color_correction:\n        # Convert the frame color if necessary\n        frame = gpu_cvt_color(frame, cv2.COLOR_BGR2RGB)\n\n    capture.release()\n    return frame if has_frame else None\n\n\ndef get_video_frame_total(video_path: str) -> int:\n    capture = cv2.VideoCapture(video_path)\n    video_frame_total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))\n    capture.release()\n    return video_frame_total\n",
    'modules/cluster_analysis.py': 'import numpy as np\nfrom sklearn.cluster import KMeans\nfrom typing import Any\n\n\ndef find_cluster_centroids(embeddings, max_k=10) -> Any:\n    inertia = []\n    cluster_centroids = []\n    K = range(1, max_k+1)\n\n    for k in K:\n        kmeans = KMeans(n_clusters=k, random_state=0)\n        kmeans.fit(embeddings)\n        inertia.append(kmeans.inertia_)\n        cluster_centroids.append({"k": k, "centroids": kmeans.cluster_centers_})\n\n    diffs = [inertia[i] - inertia[i+1] for i in range(len(inertia)-1)]\n    optimal_centroids = cluster_centroids[diffs.index(max(diffs)) + 1][\'centroids\']\n\n    return optimal_centroids\n\ndef find_closest_centroid(centroids: list, normed_face_embedding) -> list:\n    try:\n        centroids = np.array(centroids)\n        normed_face_embedding = np.array(normed_face_embedding)\n        similarities = np.dot(centroids, normed_face_embedding)\n        closest_centroid_index = np.argmax(similarities)\n        \n        return closest_centroid_index, centroids[closest_centroid_index]\n    except ValueError:\n        return None',
    'modules/core.py': 'import os\nimport sys\n# single thread doubles cuda performance - needs to be set before torch import\nif any(arg.startswith(\'--execution-provider\') for arg in sys.argv):\n    os.environ[\'OMP_NUM_THREADS\'] = \'6\'\n# reduce tensorflow log level\nos.environ[\'TF_CPP_MIN_LOG_LEVEL\'] = \'2\'\nimport warnings\nfrom typing import List\nimport platform\nimport signal\nimport shutil\nimport argparse\ntry:\n    import torch\n    HAS_TORCH = True\nexcept ImportError:\n    HAS_TORCH = False\nimport onnxruntime\ntry:\n    import tensorflow\n    HAS_TENSORFLOW = True\nexcept ImportError:\n    HAS_TENSORFLOW = False\n\nimport modules.globals\nimport modules.metadata\nimport modules.ui as ui\nfrom modules.processors.frame.core import get_frame_processors_modules, process_video_in_memory\nfrom modules.utilities import has_image_extension, is_image, is_video, detect_fps, create_video, extract_frames, get_temp_frame_paths, restore_audio, create_temp, move_temp, clean_temp, normalize_output_path\n\nif HAS_TORCH and \'ROCMExecutionProvider\' in modules.globals.execution_providers:\n    del torch\n\nwarnings.filterwarnings(\'ignore\', category=FutureWarning, module=\'insightface\')\nif HAS_TORCH:\n    warnings.filterwarnings(\'ignore\', category=UserWarning, module=\'torchvision\')\n\n\ndef parse_args() -> None:\n    signal.signal(signal.SIGINT, lambda signal_number, frame: destroy())\n    program = argparse.ArgumentParser()\n    program.add_argument(\'-s\', \'--source\', help=\'select an source image\', dest=\'source_path\')\n    program.add_argument(\'-t\', \'--target\', help=\'select an target image or video\', dest=\'target_path\')\n    program.add_argument(\'-o\', \'--output\', help=\'select output file or directory\', dest=\'output_path\')\n    program.add_argument(\'--frame-processor\', help=\'pipeline of frame processors\', dest=\'frame_processor\', default=[\'face_swapper\'], choices=[\'face_swapper\', \'face_enhancer\', \'face_enhancer_gpen256\', \'face_enhancer_gpen512\'], nargs=\'+\')\n    program.add_argument(\'--keep-fps\', help=\'keep original fps\', dest=\'keep_fps\', action=\'store_true\', default=False)\n    program.add_argument(\'--keep-audio\', help=\'keep original audio\', dest=\'keep_audio\', action=\'store_true\', default=True)\n    program.add_argument(\'--keep-frames\', help=\'keep temporary frames\', dest=\'keep_frames\', action=\'store_true\', default=False)\n    program.add_argument(\'--many-faces\', help=\'process every face\', dest=\'many_faces\', action=\'store_true\', default=False)\n    program.add_argument(\'--nsfw-filter\', help=\'filter the NSFW image or video\', dest=\'nsfw_filter\', action=\'store_true\', default=False)\n    program.add_argument(\'--map-faces\', help=\'map source target faces\', dest=\'map_faces\', action=\'store_true\', default=False)\n    program.add_argument(\'--mouth-mask\', help=\'mask the mouth region\', dest=\'mouth_mask\', action=\'store_true\', default=False)\n    program.add_argument(\'--video-encoder\', help=\'adjust output video encoder\', dest=\'video_encoder\', default=\'libx264\', choices=[\'libx264\', \'libx265\', \'libvpx-vp9\'])\n    program.add_argument(\'--video-quality\', help=\'adjust output video quality\', dest=\'video_quality\', type=int, default=18, choices=range(52), metavar=\'[0-51]\')\n    program.add_argument(\'-l\', \'--lang\', help=\'Ui language\', default="en")\n    program.add_argument(\'--live-mirror\', help=\'The live camera display as you see it in the front-facing camera frame\', dest=\'live_mirror\', action=\'store_true\', default=False)\n    program.add_argument(\'--live-resizable\', help=\'The live camera frame is resizable\', dest=\'live_resizable\', action=\'store_true\', default=False)\n    program.add_argument(\'--max-memory\', help=\'maximum amount of RAM in GB\', dest=\'max_memory\', type=int, default=suggest_max_memory())\n    program.add_argument(\'--execution-provider\', help=\'execution provider\', dest=\'execution_provider\', default=[suggest_default_execution_provider()], choices=suggest_execution_providers(), nargs=\'+\')\n    program.add_argument(\'--execution-threads\', help=\'number of execution threads\', dest=\'execution_threads\', type=int, default=suggest_execution_threads())\n    program.add_argument(\'-v\', \'--version\', action=\'version\', version=f\'{modules.metadata.name} {modules.metadata.version}\')\n\n    # register deprecated args\n    program.add_argument(\'-f\', \'--face\', help=argparse.SUPPRESS, dest=\'source_path_deprecated\')\n    program.add_argument(\'--cpu-cores\', help=argparse.SUPPRESS, dest=\'cpu_cores_deprecated\', type=int)\n    program.add_argument(\'--gpu-vendor\', help=argparse.SUPPRESS, dest=\'gpu_vendor_deprecated\')\n    program.add_argument(\'--gpu-threads\', help=argparse.SUPPRESS, dest=\'gpu_threads_deprecated\', type=int)\n\n    args = program.parse_args()\n\n    modules.globals.source_path = args.source_path\n    modules.globals.target_path = args.target_path\n    modules.globals.output_path = normalize_output_path(modules.globals.source_path, modules.globals.target_path, args.output_path)\n    modules.globals.frame_processors = args.frame_processor\n    modules.globals.headless = args.source_path or args.target_path or args.output_path\n    modules.globals.keep_fps = args.keep_fps\n    modules.globals.keep_audio = args.keep_audio\n    modules.globals.keep_frames = args.keep_frames\n    modules.globals.many_faces = args.many_faces\n    modules.globals.mouth_mask = args.mouth_mask\n    modules.globals.nsfw_filter = args.nsfw_filter\n    modules.globals.map_faces = args.map_faces\n    modules.globals.video_encoder = args.video_encoder\n    modules.globals.video_quality = args.video_quality\n    modules.globals.live_mirror = args.live_mirror\n    modules.globals.live_resizable = args.live_resizable\n    modules.globals.max_memory = args.max_memory\n    modules.globals.execution_providers = decode_execution_providers(args.execution_provider)\n    modules.globals.execution_threads = args.execution_threads\n    modules.globals.lang = args.lang\n\n    #for ENHANCER tumblers:\n    for enhancer_key in (\'face_enhancer\', \'face_enhancer_gpen256\', \'face_enhancer_gpen512\'):\n        modules.globals.fp_ui[enhancer_key] = enhancer_key in args.frame_processor\n\n    # translate deprecated args\n    if args.source_path_deprecated:\n        print(\'\\033[33mArgument -f and --face are deprecated. Use -s and --source instead.\\033[0m\')\n        modules.globals.source_path = args.source_path_deprecated\n        modules.globals.output_path = normalize_output_path(args.source_path_deprecated, modules.globals.target_path, args.output_path)\n    if args.cpu_cores_deprecated:\n        print(\'\\033[33mArgument --cpu-cores is deprecated. Use --execution-threads instead.\\033[0m\')\n        modules.globals.execution_threads = args.cpu_cores_deprecated\n    if args.gpu_vendor_deprecated == \'apple\':\n        print(\'\\033[33mArgument --gpu-vendor apple is deprecated. Use --execution-provider coreml instead.\\033[0m\')\n        modules.globals.execution_providers = decode_execution_providers([\'coreml\'])\n    if args.gpu_vendor_deprecated == \'nvidia\':\n        print(\'\\033[33mArgument --gpu-vendor nvidia is deprecated. Use --execution-provider cuda instead.\\033[0m\')\n        modules.globals.execution_providers = decode_execution_providers([\'cuda\'])\n    if args.gpu_vendor_deprecated == \'amd\':\n        print(\'\\033[33mArgument --gpu-vendor amd is deprecated. Use --execution-provider cuda instead.\\033[0m\')\n        modules.globals.execution_providers = decode_execution_providers([\'rocm\'])\n    if args.gpu_threads_deprecated:\n        print(\'\\033[33mArgument --gpu-threads is deprecated. Use --execution-threads instead.\\033[0m\')\n        modules.globals.execution_threads = args.gpu_threads_deprecated\n\n\ndef encode_execution_providers(execution_providers: List[str]) -> List[str]:\n    return [execution_provider.replace(\'ExecutionProvider\', \'\').lower() for execution_provider in execution_providers]\n\n\ndef decode_execution_providers(execution_providers: List[str]) -> List[str]:\n    return [provider for provider, encoded_execution_provider in zip(onnxruntime.get_available_providers(), encode_execution_providers(onnxruntime.get_available_providers()))\n            if any(execution_provider in encoded_execution_provider for execution_provider in execution_providers)]\n\n\ndef suggest_max_memory() -> int:\n    if platform.system().lower() == \'darwin\':\n        return 4\n    return 16\n\n\ndef suggest_default_execution_provider() -> str:\n    """Pick the best available provider: cuda > rocm > coreml > dml > cpu."""\n    available = encode_execution_providers(onnxruntime.get_available_providers())\n    for pref in (\'cuda\', \'rocm\', \'coreml\', \'dml\'):\n        if pref in available:\n            return pref\n    return \'cpu\'\n\n\ndef suggest_execution_providers() -> List[str]:\n    return encode_execution_providers(onnxruntime.get_available_providers())\n\n\ndef suggest_execution_threads() -> int:\n    """Suggest optimal thread count based on hardware and execution provider."""\n    import os\n    \n    # Get CPU count\n    cpu_count = os.cpu_count() or 4\n    \n    if \'DmlExecutionProvider\' in modules.globals.execution_providers:\n        return 1\n    if \'ROCMExecutionProvider\' in modules.globals.execution_providers:\n        return 1\n    if \'CUDAExecutionProvider\' in modules.globals.execution_providers:\n        return 2\n    \n    # For CPU execution, use most cores but leave some for system\n    return max(4, min(cpu_count - 2, 16))\n\n\ndef limit_resources() -> None:\n    # prevent tensorflow memory leak\n    if HAS_TENSORFLOW:\n        gpus = tensorflow.config.experimental.list_physical_devices(\'GPU\')\n        for gpu in gpus:\n            tensorflow.config.experimental.set_memory_growth(gpu, True)\n    # limit memory usage\n    if modules.globals.max_memory:\n        memory = modules.globals.max_memory * 1024 ** 3\n        if platform.system().lower() == \'windows\':\n            import ctypes\n            kernel32 = ctypes.windll.kernel32\n            kernel32.SetProcessWorkingSetSize(-1, ctypes.c_size_t(memory), ctypes.c_size_t(memory))\n        else:\n            import resource\n            resource.setrlimit(resource.RLIMIT_DATA, (memory, memory))\n\n\ndef release_resources() -> None:\n    if \'CUDAExecutionProvider\' in modules.globals.execution_providers and HAS_TORCH:\n        torch.cuda.empty_cache()\n\n\ndef pre_check() -> bool:\n    if sys.version_info < (3, 9):\n        update_status(\'Python version is not supported - please upgrade to 3.9 or higher.\')\n        return False\n    if not shutil.which(\'ffmpeg\'):\n        update_status(\'ffmpeg is not installed.\')\n        return False\n    return True\n\n\ndef update_status(message: str, scope: str = \'DLC.CORE\') -> None:\n    print(f\'[{scope}] {message}\')\n    if not modules.globals.headless:\n        ui.update_status(message)\n\ndef start() -> None:\n    """Start processing with performance monitoring."""\n    import time\n    \n    start_time = time.time()\n    \n    for frame_processor in get_frame_processors_modules(modules.globals.frame_processors):\n        if not frame_processor.pre_start():\n            return\n    update_status(\'Processing...\')\n    \n    # process image to image\n    if has_image_extension(modules.globals.target_path):\n        if modules.globals.nsfw_filter and ui.check_and_ignore_nsfw(modules.globals.target_path, destroy):\n            return\n        try:\n            shutil.copy2(modules.globals.target_path, modules.globals.output_path)\n        except Exception as e:\n            print("Error copying file:", str(e))\n        for frame_processor in get_frame_processors_modules(modules.globals.frame_processors):\n            update_status(\'Progressing...\', frame_processor.NAME)\n            frame_processor.process_image(modules.globals.source_path, modules.globals.output_path, modules.globals.output_path)\n            release_resources()\n        if is_image(modules.globals.target_path):\n            elapsed = time.time() - start_time\n            update_status(f\'Processing to image succeed! (Time: {elapsed:.2f}s)\')\n        else:\n            update_status(\'Processing to image failed!\')\n        return\n    \n    # process image to videos\n    if modules.globals.nsfw_filter and ui.check_and_ignore_nsfw(modules.globals.target_path, destroy):\n        return\n\n    # Detect FPS early (needed by both pipelines)\n    if modules.globals.keep_fps:\n        update_status(\'Detecting fps...\')\n        fps = detect_fps(modules.globals.target_path)\n    else:\n        fps = 30.0\n\n    video_created = False\n\n    # --- In-memory pipeline (non-map_faces only) ---\n    # Reads frames from FFmpeg pipe, processes in memory, encodes directly.\n    # Eliminates all per-frame PNG disk I/O for a major speed-up.\n    if not modules.globals.map_faces:\n        update_status(f\'Processing video in-memory at {fps} fps...\')\n        create_temp(modules.globals.target_path)\n\n        processing_start = time.time()\n        video_created = process_video_in_memory(\n            modules.globals.source_path,\n            modules.globals.target_path,\n            fps,\n        )\n        processing_time = time.time() - processing_start\n        release_resources()\n\n        if video_created:\n            update_status(f\'In-memory processing + encoding completed in {processing_time:.2f}s\')\n\n    # --- Disk-based fallback (required for map_faces, or if pipe failed) ---\n    if not video_created:\n        if not modules.globals.map_faces:\n            update_status(\'Falling back to disk-based processing...\')\n\n        extraction_start = time.time()\n        if not modules.globals.map_faces:\n            create_temp(modules.globals.target_path)\n            update_status(\'Extracting frames...\')\n            extract_frames(modules.globals.target_path)\n        extraction_time = time.time() - extraction_start\n\n        temp_frame_paths = get_temp_frame_paths(modules.globals.target_path)\n        total_frames = len(temp_frame_paths)\n        update_status(f\'Processing {total_frames} frames with {modules.globals.execution_threads} threads...\')\n\n        processing_start = time.time()\n        for frame_processor in get_frame_processors_modules(modules.globals.frame_processors):\n            update_status(\'Progressing...\', frame_processor.NAME)\n            frame_processor.process_video(modules.globals.source_path, temp_frame_paths)\n            release_resources()\n        processing_time = time.time() - processing_start\n        fps_processing = total_frames / processing_time if processing_time > 0 else 0\n        update_status(f\'Frame processing completed in {processing_time:.2f}s ({fps_processing:.2f} fps)\')\n\n        encoding_start = time.time()\n        update_status(f\'Creating video with {fps} fps...\')\n        video_created = create_video(modules.globals.target_path, fps)\n        encoding_time = time.time() - encoding_start\n        if video_created:\n            update_status(f\'Video encoding completed in {encoding_time:.2f}s\')\n\n    if not video_created:\n        update_status(\'Video encoding failed. No temporary output video was created.\')\n        clean_temp(modules.globals.target_path)\n        return\n    \n    # handle audio\n    if modules.globals.keep_audio:\n        if modules.globals.keep_fps:\n            update_status(\'Restoring audio...\')\n        else:\n            update_status(\'Restoring audio might cause issues as fps are not kept...\')\n        restore_audio(modules.globals.target_path, modules.globals.output_path)\n    else:\n        move_temp(modules.globals.target_path, modules.globals.output_path)\n    \n    # clean and validate\n    clean_temp(modules.globals.target_path)\n    \n    total_time = time.time() - start_time\n    if is_video(modules.globals.target_path) and modules.globals.output_path and os.path.isfile(modules.globals.output_path):\n        update_status(f\'Video processing succeeded! Total time: {total_time:.2f}s\')\n    else:\n        update_status(\'Processing to video failed!\')\n\n\ndef destroy(to_quit=True) -> None:\n    if modules.globals.target_path:\n        clean_temp(modules.globals.target_path)\n    if to_quit:\n        quit()\n\n\ndef run() -> None:\n    parse_args()\n    if not pre_check():\n        return\n    for frame_processor in get_frame_processors_modules(modules.globals.frame_processors):\n        if not frame_processor.pre_check():\n            return\n    # Pre-load face analyser in main thread before GUI starts\n    #from modules.face_analyser import get_face_analyser\n    #get_face_analyser()\n    limit_resources()\n    if modules.globals.headless:\n        start()\n    else:\n        window = ui.init(start, destroy, modules.globals.lang)\n        window.mainloop()',
    'modules/custom_types.py': 'from typing import Any\n\nfrom insightface.app.common import Face\nimport numpy\n \nFace = Face\nFrame = numpy.ndarray[Any, Any] ',
    'modules/face_analyser.py': 'import os\nimport shutil\nfrom typing import Any\nimport insightface\nimport threading\n\nimport modules.globals\nfrom modules import imread_unicode, imwrite_unicode\nfrom tqdm import tqdm\nfrom modules.typing import Frame\nfrom modules.cluster_analysis import find_cluster_centroids, find_closest_centroid\nfrom modules.utilities import get_temp_directory_path, create_temp, extract_frames, clean_temp, get_temp_frame_paths\nfrom pathlib import Path\n\nFACE_ANALYSER = None\nFACE_ANALYSER_LOCK = threading.Lock()\n\nDET_SIZE = (640, 640)\n\n\ndef get_face_analyser() -> Any:\n    """Get face analyser with thread-safe initialization."""\n    global FACE_ANALYSER\n\n    if FACE_ANALYSER is None:\n        with FACE_ANALYSER_LOCK:\n            # Double-check after acquiring lock\n            if FACE_ANALYSER is None:\n                from modules.processors.frame._onnx_enhancer import (\n                    build_provider_config,\n                )\n                providers = build_provider_config()\n                FACE_ANALYSER = insightface.app.FaceAnalysis(\n                    name=\'buffalo_l\',\n                    providers=providers,\n                    allowed_modules=[\'detection\', \'recognition\', \'landmark_2d_106\']\n                )\n                FACE_ANALYSER.prepare(ctx_id=0, det_size=DET_SIZE)\n                _optimize_det_model(FACE_ANALYSER, providers)\n    return FACE_ANALYSER\n\n\ndef _optimize_det_model(fa: Any, providers) -> None:\n    """Replace the detection model\'s ONNX session with a CoreML-optimized one.\n\n    Folds dynamic Shape→Gather chains into constants (the input size is\n    fixed at det_size), eliminating CPU↔ANE partition boundaries in the\n    RetinaFace FPN upsampling path.  21ms → 4ms on M3 Max.\n    """\n    from modules.onnx_optimize import optimize_for_coreml, IS_APPLE_SILICON\n    if not IS_APPLE_SILICON:\n        return\n\n    det_model = fa.det_model\n    model_path = getattr(det_model, \'model_file\', None)\n    if model_path is None or not os.path.exists(model_path):\n        return\n\n    input_shape = (1, 3, DET_SIZE[1], DET_SIZE[0])\n    optimized_path = optimize_for_coreml(model_path, input_shape=input_shape)\n    if optimized_path == model_path:\n        return\n\n    import onnxruntime\n    session_options = onnxruntime.SessionOptions()\n    session_options.graph_optimization_level = (\n        onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL\n    )\n\n    # Route detection to GPU shader cores (CPUAndGPU) instead of ANE.\n    # This lets detection run concurrently with the swap model on the\n    # ANE, overlapping the two inference calls.  Detection is fast\n    # enough on GPU (~4ms) and this frees ANE for the heavier swap.\n    det_providers = []\n    for p in providers:\n        name = p[0] if isinstance(p, tuple) else p\n        if name == "CoreMLExecutionProvider":\n            det_providers.append((\n                "CoreMLExecutionProvider",\n                {"ModelFormat": "MLProgram", "MLComputeUnits": "CPUAndGPU"},\n            ))\n        else:\n            det_providers.append(p)\n\n    det_model.session = onnxruntime.InferenceSession(\n        optimized_path, sess_options=session_options, providers=det_providers,\n    )\n\n\ndef _needs_landmark() -> bool:\n    """Check whether any active feature requires 106-point landmarks.\n\n    Landmarks are needed by face enhancers and mouth masking, but not\n    by the face swapper alone.\n    """\n    if getattr(modules.globals, "mouth_mask", False):\n        return True\n    processors = getattr(modules.globals, "frame_processors", [])\n    return any(p in processors for p in\n               ("face_enhancer", "face_enhancer_gpen256", "face_enhancer_gpen512"))\n\n\ndef _is_dml() -> bool:\n    return any("DmlExecutionProvider" in p for p in modules.globals.execution_providers)\n\n\ndef _analyse_faces(frame: Frame) -> list:\n    """Run face detection, then recognition (and optionally landmark).\n\n    Replaces InsightFace\'s ``FaceAnalysis.get()`` to skip the\n    landmark_2d_106 model when only face_swapper is active (saves ~1ms\n    per face and avoids an unnecessary ONNX session call).\n    """\n    fa = get_face_analyser()\n\n    bboxes, kpss = fa.det_model.detect(frame, max_num=0, metric="default")\n    if bboxes.shape[0] == 0:\n        return []\n\n    need_landmark = _needs_landmark()\n    rec_model = fa.models.get("recognition")\n    lmk_model = fa.models.get("landmark_2d_106") if need_landmark else None\n\n    from insightface.app.common import Face\n\n    faces = []\n    for i in range(bboxes.shape[0]):\n        face = Face(bbox=bboxes[i, 0:4],\n                    kps=kpss[i] if kpss is not None else None,\n                    det_score=bboxes[i, 4])\n        if rec_model is not None:\n            rec_model.get(frame, face)\n        if lmk_model is not None:\n            lmk_model.get(frame, face)\n        faces.append(face)\n\n    return faces\n\n\ndef get_one_face(frame: Frame, faces: Any = None) -> Any:\n    if faces is None:\n        if _is_dml():\n            with modules.globals.dml_lock:\n                faces = _analyse_faces(frame)\n        else:\n            faces = _analyse_faces(frame)\n    try:\n        return min(faces, key=lambda x: x.bbox[0])\n    except ValueError:\n        return None\n\n\ndef get_many_faces(frame: Frame) -> Any:\n    try:\n        if _is_dml():\n            with modules.globals.dml_lock:\n                return _analyse_faces(frame)\n        else:\n            return _analyse_faces(frame)\n    except IndexError:\n        return None\n\ndef detect_one_face_fast(frame: Frame) -> Any:\n    """Detection-only — skips landmark and recognition models.\n\n    Returns a Face with bbox, kps, det_score (enough for face swap).\n    ~10ms vs ~16ms for full get_one_face() at 1080p.\n    """\n    from insightface.app.common import Face\n    fa = get_face_analyser()\n    bboxes, kpss = fa.det_model.detect(frame, max_num=0, metric=\'default\')\n    if bboxes.shape[0] == 0:\n        return None\n    idx = int(bboxes[:, 0].argmin())\n    return Face(bbox=bboxes[idx, :4], kps=kpss[idx], det_score=bboxes[idx, 4])\n\n\ndef detect_many_faces_fast(frame: Frame) -> Any:\n    """Detection-only multi-face — skips landmark and recognition."""\n    from insightface.app.common import Face\n    fa = get_face_analyser()\n    bboxes, kpss = fa.det_model.detect(frame, max_num=0, metric=\'default\')\n    if bboxes.shape[0] == 0:\n        return None\n    return [Face(bbox=bboxes[i, :4], kps=kpss[i], det_score=bboxes[i, 4])\n            for i in range(bboxes.shape[0])]\n\n\ndef ensure_landmarks(frame: Frame, faces: Any) -> None:\n    """Run the 2d106 landmark model in-place on faces that lack it.\n\n    The fast webcam path (detect_one_face_fast / detect_many_faces_fast)\n    produces detection-only Face objects with no ``landmark_2d_106``.\n    Mouth masking needs those landmarks, so add them on demand only when\n    the feature is active — keeping the fast path fast otherwise.\n    """\n    if faces is None:\n        return\n    if not isinstance(faces, (list, tuple)):\n        faces = [faces]\n\n    fa = get_face_analyser()\n    lmk_model = fa.models.get("landmark_2d_106")\n    if lmk_model is None:\n        return\n\n    for face in faces:\n        if face is None:\n            continue\n        # insightface Face is a dict; missing keys raise AttributeError,\n        # so getattr(..., None) is the safe presence check.\n        if getattr(face, "landmark_2d_106", None) is None:\n            try:\n                lmk_model.get(frame, face)\n            except Exception as e:  # pragma: no cover - never break the swap\n                print(f"Error computing 2d106 landmarks: {e}")\n\n\ndef has_valid_map() -> bool:\n    for map in modules.globals.source_target_map:\n        if "source" in map and "target" in map:\n            return True\n    return False\n\ndef default_source_face() -> Any:\n    for map in modules.globals.source_target_map:\n        if "source" in map:\n            return map[\'source\'][\'face\']\n    return None\n\ndef simplify_maps() -> Any:\n    centroids = []\n    faces = []\n    for map in modules.globals.source_target_map:\n        if "source" in map and "target" in map:\n            centroids.append(map[\'target\'][\'face\'].normed_embedding)\n            faces.append(map[\'source\'][\'face\'])\n\n    modules.globals.simple_map = {\'source_faces\': faces, \'target_embeddings\': centroids}\n    return None\n\ndef add_blank_map() -> Any:\n    try:\n        max_id = -1\n        if len(modules.globals.source_target_map) > 0:\n            max_id = max(modules.globals.source_target_map, key=lambda x: x[\'id\'])[\'id\']\n\n        modules.globals.source_target_map.append({\n                \'id\' : max_id + 1\n                })\n    except ValueError:\n        return None\n    \ndef get_unique_faces_from_target_image() -> Any:\n    try:\n        modules.globals.source_target_map = []\n        target_frame = imread_unicode(modules.globals.target_path)\n        many_faces = get_many_faces(target_frame)\n        if many_faces is None:\n            return None\n        i = 0\n\n        for face in many_faces:\n            x_min, y_min, x_max, y_max = face[\'bbox\']\n            modules.globals.source_target_map.append({\n                \'id\' : i, \n                \'target\' : {\n                            \'cv2\' : target_frame[int(y_min):int(y_max), int(x_min):int(x_max)],\n                            \'face\' : face\n                            }\n                })\n            i = i + 1\n    except ValueError:\n        return None\n    \n    \ndef get_unique_faces_from_target_video() -> Any:\n    try:\n        modules.globals.source_target_map = []\n        frame_face_embeddings = []\n        face_embeddings = []\n    \n        print(\'Creating temp resources...\')\n        clean_temp(modules.globals.target_path)\n        create_temp(modules.globals.target_path)\n        print(\'Extracting frames...\')\n        extract_frames(modules.globals.target_path)\n\n        temp_frame_paths = get_temp_frame_paths(modules.globals.target_path)\n\n        i = 0\n        for temp_frame_path in tqdm(temp_frame_paths, desc="Extracting face embeddings from frames"):\n            temp_frame = imread_unicode(temp_frame_path)\n            many_faces = get_many_faces(temp_frame)\n            if many_faces is None:\n                continue\n\n            for face in many_faces:\n                face_embeddings.append(face.normed_embedding)\n            \n            frame_face_embeddings.append({\'frame\': i, \'faces\': many_faces, \'location\': temp_frame_path})\n            i += 1\n\n        centroids = find_cluster_centroids(face_embeddings)\n\n        for frame in frame_face_embeddings:\n            for face in frame[\'faces\']:\n                closest_centroid_index, _ = find_closest_centroid(centroids, face.normed_embedding)\n                face[\'target_centroid\'] = closest_centroid_index\n\n        for i in range(len(centroids)):\n            modules.globals.source_target_map.append({\n                \'id\' : i\n            })\n\n            temp = []\n            for frame in tqdm(frame_face_embeddings, desc=f"Mapping frame embeddings to centroids-{i}"):\n                temp.append({\'frame\': frame[\'frame\'], \'faces\': [face for face in frame[\'faces\'] if face[\'target_centroid\'] == i], \'location\': frame[\'location\']})\n\n            modules.globals.source_target_map[i][\'target_faces_in_frame\'] = temp\n\n        # dump_faces(centroids, frame_face_embeddings)\n        default_target_face()\n    except ValueError:\n        return None\n    \n\ndef default_target_face():\n    for map in modules.globals.source_target_map:\n        best_face = None\n        best_frame = None\n        for frame in map[\'target_faces_in_frame\']:\n            if len(frame[\'faces\']) > 0:\n                best_face = frame[\'faces\'][0]\n                best_frame = frame\n                break\n\n        for frame in map[\'target_faces_in_frame\']:\n            for face in frame[\'faces\']:\n                if face[\'det_score\'] > best_face[\'det_score\']:\n                    best_face = face\n                    best_frame = frame\n\n        x_min, y_min, x_max, y_max = best_face[\'bbox\']\n\n        target_frame = imread_unicode(best_frame[\'location\'])\n        map[\'target\'] = {\n                        \'cv2\' : target_frame[int(y_min):int(y_max), int(x_min):int(x_max)],\n                        \'face\' : best_face\n                        }\n\n\ndef dump_faces(centroids: Any, frame_face_embeddings: list):\n    temp_directory_path = get_temp_directory_path(modules.globals.target_path)\n\n    for i in range(len(centroids)):\n        if os.path.exists(temp_directory_path + f"/{i}") and os.path.isdir(temp_directory_path + f"/{i}"):\n            shutil.rmtree(temp_directory_path + f"/{i}")\n        Path(temp_directory_path + f"/{i}").mkdir(parents=True, exist_ok=True)\n\n        for frame in tqdm(frame_face_embeddings, desc=f"Copying faces to temp/./{i}"):\n            temp_frame = imread_unicode(frame[\'location\'])\n\n            j = 0\n            for face in frame[\'faces\']:\n                if face[\'target_centroid\'] == i:\n                    x_min, y_min, x_max, y_max = face[\'bbox\']\n\n                    if temp_frame[int(y_min):int(y_max), int(x_min):int(x_max)].size > 0:\n                        imwrite_unicode(temp_directory_path + f"/{i}/{frame[\'frame\']}_{j}.png", temp_frame[int(y_min):int(y_max), int(x_min):int(x_max)])\n                j += 1\n',
    'modules/gettext.py': 'import json\nfrom pathlib import Path\n\nclass LanguageManager:\n    def __init__(self, default_language="en"):\n        self.current_language = default_language\n        self.translations = {}\n        self.load_language(default_language)\n\n    def load_language(self, language_code) -> bool:\n        """load language file"""\n        if language_code == "en":\n            return True\n        try:\n            file_path = Path(__file__).parent.parent / f"locales/{language_code}.json"\n            with open(file_path, "r", encoding="utf-8") as file:\n                self.translations = json.load(file)\n            self.current_language = language_code\n            return True\n        except FileNotFoundError:\n            print(f"Language file not found: {language_code}")\n            return False\n\n    def _(self, key, default=None) -> str:\n        """get translate text"""\n        return self.translations.get(key, default if default else key)',
    'modules/globals.py': '# --- START OF FILE globals.py ---\n\nimport os\nfrom typing import List, Dict, Any\n\nROOT_DIR = os.path.dirname(os.path.abspath(__file__))\nWORKFLOW_DIR = os.path.join(ROOT_DIR, "workflow")\n\nfile_types = [\n    ("Image", ("*.png", "*.jpg", "*.jpeg", "*.gif", "*.bmp")),\n    ("Video", ("*.mp4", "*.mkv")),\n]\n\n# Face Mapping Data\nsource_target_map: List[Dict[str, Any]] = [] # Stores detailed map for image/video processing\nsimple_map: Dict[str, Any] = {}             # Stores simplified map (embeddings/faces) for live/simple mode\n\n# Paths\nsource_path: str | None = None\ntarget_path: str | None = None\noutput_path: str | None = None\n\n# Processing Options\nframe_processors: List[str] = []\nkeep_fps: bool = True\nkeep_audio: bool = True\nkeep_frames: bool = False\nmany_faces: bool = False         # Process all detected faces with default source\nmap_faces: bool = False          # Use source_target_map or simple_map for specific swaps\npoisson_blend: bool = False      # Enable Poisson Blending for smoother face swaps\ncolor_correction: bool = False   # Enable color correction (implementation specific)\nnsfw_filter: bool = False\n\n# Video Output Options\nvideo_encoder: str | None = None\nvideo_quality: int | None = None # Typically a CRF value or bitrate\n\n# Live Mode Options\nlive_mirror: bool = False\nlive_resizable: bool = True\ncamera_input_combobox: Any | None = None # Placeholder for UI element if needed\nwebcam_preview_running: bool = False\nshow_fps: bool = False\n\n# System Configuration\nmax_memory: int | None = None        # Memory limit in GB? (Needs clarification)\nexecution_providers: List[str] = []  # e.g., [\'CUDAExecutionProvider\', \'CPUExecutionProvider\']\nexecution_threads: int | None = None # Number of threads for CPU execution\nheadless: bool | None = None         # Run without UI?\nlog_level: str = "error"             # Logging level (e.g., \'debug\', \'info\', \'warning\', \'error\')\n\n# Face Processor UI Toggles (Example)\nfp_ui: Dict[str, bool] = {"face_enhancer": False, "face_enhancer_gpen256": False, "face_enhancer_gpen512": False}\n\n# Face Swapper Specific Options\nface_swapper_enabled: bool = True # General toggle for the swapper processor\nopacity: float = 1.0              # Blend factor for the swapped face (0.0-1.0)\nsharpness: float = 0.0            # Sharpness enhancement for swapped face (0.0-1.0+)\n\n# Mouth Mask Options\nmouth_mask: bool = False           # Enable mouth area masking/pasting\nshow_mouth_mask_box: bool = False  # Visualize the mouth mask area (for debugging)\nmask_feather_ratio: int = 12       # Denominator for feathering calculation (higher = smaller feather)\nmask_down_size: float = 0.1        # Expansion factor for lower lip mask (relative)\nmask_size: float = 1.0             # Expansion factor for upper lip mask (relative)\nmouth_mask_size: float = 0.0       # Mouth mask size (0-100; 0=off, 100=mouth to chin)\n\n# --- START: Added for Frame Interpolation ---\nenable_interpolation: bool = True # Toggle temporal smoothing\ninterpolation_weight: float = 0  # Blend weight for current frame (0.0-1.0). Lower=smoother.\n# --- END: Added for Frame Interpolation ---\n\n# --- END OF FILE globals.py ---\n\nimport threading\ndml_lock = threading.Lock()\n',
    'modules/gpu_processing.py': '# --- START OF FILE gpu_processing.py ---\n"""\nGPU-accelerated image processing using OpenCV CUDA (cv2.cuda.GpuMat).\n\nProvides drop-in replacements for common cv2 functions.  When OpenCV is built\nwith CUDA support the functions transparently upload → process → download via\nGpuMat; otherwise they fall back to the regular CPU path so the rest of the\ncodebase never has to care whether CUDA is available.\n\nUsage\n-----\n    from modules.gpu_processing import (\n        gpu_gaussian_blur, gpu_sharpen, gpu_add_weighted,\n        gpu_resize, gpu_cvt_color, gpu_flip,\n        is_gpu_accelerated,\n    )\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport cv2\nimport numpy as np\nfrom typing import Tuple\n\n# ---------------------------------------------------------------------------\n# CUDA availability detection (evaluated once at import time)\n# ---------------------------------------------------------------------------\nCUDA_AVAILABLE: bool = False\n\n# OpenCV CUDA per-operation acceleration is DISABLED by default.\n# Each gpu_* call uploads to GPU, processes, then downloads back to CPU.\n# At webcam resolution (~960x540) this upload/download overhead far exceeds\n# the time saved on the actual operation, making it slower than pure CPU.\n# The heavy lifting (face detection, swap, enhancement) runs on GPU via\n# ONNX Runtime\'s CUDAExecutionProvider, which is where GPU matters.\n#\n# To force-enable, set OPENCV_CUDA_PROCESSING=1 in your environment.\nif os.environ.get("OPENCV_CUDA_PROCESSING") == "1":\n    try:\n        _test_mat = cv2.cuda.GpuMat()\n        _has_gauss = hasattr(cv2.cuda, "createGaussianFilter")\n        _has_resize = hasattr(cv2.cuda, "resize")\n        _has_cvt = hasattr(cv2.cuda, "cvtColor")\n        if _has_gauss and _has_resize and _has_cvt:\n            CUDA_AVAILABLE = True\n            print("[gpu_processing] OpenCV CUDA processing enabled via OPENCV_CUDA_PROCESSING=1.")\n    except Exception:\n        pass\n\n\n# ---------------------------------------------------------------------------\n# Internal helpers\n# ---------------------------------------------------------------------------\n\ndef _ensure_uint8(img: np.ndarray) -> np.ndarray:\n    """Clip and convert to uint8 if necessary."""\n    if img.dtype != np.uint8:\n        return np.clip(img, 0, 255).astype(np.uint8)\n    return img\n\n\ndef _ksize_odd(ksize: Tuple[int, int]) -> Tuple[int, int]:\n    """Ensure kernel dimensions are positive and odd (required by GaussianBlur)."""\n    kw = max(1, ksize[0] // 2 * 2 + 1) if ksize[0] > 0 else 0\n    kh = max(1, ksize[1] // 2 * 2 + 1) if ksize[1] > 0 else 0\n    return (kw, kh)\n\n\ndef _cv_type_for(img: np.ndarray) -> int:\n    """Return the OpenCV type constant matching *img* (uint8 only)."""\n    channels = 1 if img.ndim == 2 else img.shape[2]\n    if channels == 1:\n        return cv2.CV_8UC1\n    elif channels == 3:\n        return cv2.CV_8UC3\n    elif channels == 4:\n        return cv2.CV_8UC4\n    return cv2.CV_8UC3  # fallback\n\n\n# ---------------------------------------------------------------------------\n# Public API – Gaussian Blur\n# ---------------------------------------------------------------------------\n\ndef gpu_gaussian_blur(\n    src: np.ndarray,\n    ksize: Tuple[int, int],\n    sigma_x: float,\n    sigma_y: float = 0,\n) -> np.ndarray:\n    """Drop-in replacement for ``cv2.GaussianBlur`` with CUDA acceleration.\n\n    Parameters match ``cv2.GaussianBlur(src, ksize, sigmaX, sigmaY)``.\n    When *ksize* is ``(0, 0)`` OpenCV computes the kernel size from *sigma_x*.\n    """\n    if CUDA_AVAILABLE:\n        try:\n            src_u8 = _ensure_uint8(src)\n            cv_type = _cv_type_for(src_u8)\n            ks = _ksize_odd(ksize) if ksize != (0, 0) else ksize\n\n            gauss = cv2.cuda.createGaussianFilter(cv_type, cv_type, ks, sigma_x, sigma_y)\n            gpu_src = cv2.cuda.GpuMat()\n            gpu_src.upload(src_u8)\n            gpu_dst = gauss.apply(gpu_src)\n            return gpu_dst.download()\n        except cv2.error:\n            pass\n\n    return cv2.GaussianBlur(src, ksize, sigma_x, sigmaY=sigma_y)\n\n\n# ---------------------------------------------------------------------------\n# Public API – addWeighted\n# ---------------------------------------------------------------------------\n\ndef gpu_add_weighted(\n    src1: np.ndarray,\n    alpha: float,\n    src2: np.ndarray,\n    beta: float,\n    gamma: float,\n) -> np.ndarray:\n    """Drop-in replacement for ``cv2.addWeighted`` with CUDA acceleration."""\n    if CUDA_AVAILABLE:\n        try:\n            s1 = _ensure_uint8(src1)\n            s2 = _ensure_uint8(src2)\n            g1 = cv2.cuda.GpuMat()\n            g2 = cv2.cuda.GpuMat()\n            g1.upload(s1)\n            g2.upload(s2)\n            gpu_dst = cv2.cuda.addWeighted(g1, alpha, g2, beta, gamma)\n            return gpu_dst.download()\n        except cv2.error:\n            pass\n\n    return cv2.addWeighted(src1, alpha, src2, beta, gamma)\n\n\n# ---------------------------------------------------------------------------\n# Public API – Unsharp-mask sharpening\n# ---------------------------------------------------------------------------\n\ndef gpu_sharpen(\n    src: np.ndarray,\n    strength: float,\n    sigma: float = 3,\n) -> np.ndarray:\n    """Unsharp-mask sharpening, optionally GPU-accelerated.\n\n    Equivalent to::\n\n        blurred = GaussianBlur(src, (0,0), sigma)\n        result  = addWeighted(src, 1+strength, blurred, -strength, 0)\n    """\n    if strength <= 0:\n        return src\n\n    if CUDA_AVAILABLE:\n        try:\n            src_u8 = _ensure_uint8(src)\n            cv_type = _cv_type_for(src_u8)\n\n            gauss = cv2.cuda.createGaussianFilter(cv_type, cv_type, (0, 0), sigma)\n            gpu_src = cv2.cuda.GpuMat()\n            gpu_src.upload(src_u8)\n            gpu_blurred = gauss.apply(gpu_src)\n            gpu_sharp = cv2.cuda.addWeighted(gpu_src, 1.0 + strength, gpu_blurred, -strength, 0)\n            result = gpu_sharp.download()\n            return np.clip(result, 0, 255).astype(np.uint8)\n        except cv2.error:\n            pass\n\n    blurred = cv2.GaussianBlur(src, (0, 0), sigma)\n    sharpened = cv2.addWeighted(src, 1.0 + strength, blurred, -strength, 0)\n    return np.clip(sharpened, 0, 255).astype(np.uint8)\n\n\n# ---------------------------------------------------------------------------\n# Public API – Resize\n# ---------------------------------------------------------------------------\n\n# Map common cv2 interpolation flags to their CUDA equivalents\n_INTERP_MAP = {\n    cv2.INTER_NEAREST: cv2.INTER_NEAREST,\n    cv2.INTER_LINEAR: cv2.INTER_LINEAR,\n    cv2.INTER_CUBIC: cv2.INTER_CUBIC,\n    cv2.INTER_AREA: cv2.INTER_AREA,\n    cv2.INTER_LANCZOS4: cv2.INTER_LANCZOS4,\n}\n\n\ndef gpu_resize(\n    src: np.ndarray,\n    dsize: Tuple[int, int],\n    fx: float = 0,\n    fy: float = 0,\n    interpolation: int = cv2.INTER_LINEAR,\n) -> np.ndarray:\n    """Drop-in replacement for ``cv2.resize`` with CUDA acceleration.\n\n    Parameters match ``cv2.resize(src, dsize, fx=fx, fy=fy, interpolation=...)``.\n    """\n    if CUDA_AVAILABLE:\n        try:\n            src_u8 = _ensure_uint8(src)\n            gpu_src = cv2.cuda.GpuMat()\n            gpu_src.upload(src_u8)\n\n            interp = _INTERP_MAP.get(interpolation, cv2.INTER_LINEAR)\n\n            if dsize and dsize[0] > 0 and dsize[1] > 0:\n                gpu_dst = cv2.cuda.resize(gpu_src, dsize, interpolation=interp)\n            else:\n                gpu_dst = cv2.cuda.resize(gpu_src, (0, 0), fx=fx, fy=fy, interpolation=interp)\n\n            return gpu_dst.download()\n        except cv2.error:\n            pass\n\n    return cv2.resize(src, dsize, fx=fx, fy=fy, interpolation=interpolation)\n\n\n# ---------------------------------------------------------------------------\n# Public API – Color conversion\n# ---------------------------------------------------------------------------\n\ndef gpu_cvt_color(\n    src: np.ndarray,\n    code: int,\n) -> np.ndarray:\n    """Drop-in replacement for ``cv2.cvtColor`` with CUDA acceleration.\n\n    Parameters match ``cv2.cvtColor(src, code)``.\n    """\n    if CUDA_AVAILABLE:\n        try:\n            src_u8 = _ensure_uint8(src)\n            gpu_src = cv2.cuda.GpuMat()\n            gpu_src.upload(src_u8)\n            gpu_dst = cv2.cuda.cvtColor(gpu_src, code)\n            return gpu_dst.download()\n        except cv2.error:\n            pass\n\n    return cv2.cvtColor(src, code)\n\n\n# ---------------------------------------------------------------------------\n# Public API – Flip\n# ---------------------------------------------------------------------------\n\ndef gpu_flip(\n    src: np.ndarray,\n    flip_code: int,\n) -> np.ndarray:\n    """Drop-in replacement for ``cv2.flip`` with CUDA acceleration.\n\n    Parameters match ``cv2.flip(src, flipCode)``.\n    *flip_code*: 0 = vertical, 1 = horizontal, -1 = both.\n    """\n    if CUDA_AVAILABLE:\n        try:\n            src_u8 = _ensure_uint8(src)\n            gpu_src = cv2.cuda.GpuMat()\n            gpu_src.upload(src_u8)\n            gpu_dst = cv2.cuda.flip(gpu_src, flip_code)\n            return gpu_dst.download()\n        except cv2.error:\n            pass\n\n    return cv2.flip(src, flip_code)\n\n\n# ---------------------------------------------------------------------------\n# Convenience: check at runtime whether GPU path is active\n# ---------------------------------------------------------------------------\n\ndef is_gpu_accelerated() -> bool:\n    """Return ``True`` when the CUDA path will be used."""\n    return CUDA_AVAILABLE\n\n# --- END OF FILE gpu_processing.py ---\n',
    'modules/metadata.py': "name = 'Deep-Live-Cam'\nversion = '2.1.5'\nedition = 'GitHub Edition'",
    'modules/onnx_optimize.py': '"""ONNX model optimizations for CoreML execution on Apple Silicon.\n\nEach pass eliminates a different CPU↔ANE round-trip that ORT\'s CoreML EP\nwould otherwise introduce:\n\n1. **Shape/Gather constant folding** — Dynamic ``Shape`` → ``Gather`` chains\n   (e.g. for FPN upsample target sizes in RetinaFace) force ops onto CPU even\n   when the input dimensions are known at load time.  We run ONNX shape\n   inference with the known input size and replace these chains with constants.\n   Float32-noise-level differences only (max ~6e-6).\n\n2. **Pad(reflect) decomposition** — CoreML doesn\'t support ``Pad(mode=reflect)``.\n   Models using reflect padding (e.g. inswapper_128) get split into many CoreML\n   subgraphs with CPU fallbacks between each.  We rewrite each ``Pad(reflect)``\n   as equivalent ``Slice`` + ``Concat`` ops that CoreML handles natively.\n   Bit-for-bit identical output. (Fixed upstream in microsoft/onnxruntime#28073.)\n\n3. **Split → Slice decomposition** — CoreML\'s EP doesn\'t support the ONNX\n   ``Split`` op, causing partition boundaries in models with channel-wise\n   splits (e.g. GFPGAN\'s SFT modulation). Each 2-way Split becomes two Slices.\n\n4. **Scalar Gather widening** — ORT\'s CoreML EP rejects ``Gather`` nodes with\n   rank-0 (scalar) indices. StyleGAN-derived models (GFPGAN) slice per-layer\n   style codes using exactly this pattern. We widen each scalar index to\n   ``[1]`` and squeeze the added axis on the Gather output.\n   (Filed upstream as microsoft/onnxruntime#28180.)\n\nAll passes are cached on disk with a ``_coreml`` suffix so the rewrite cost\nis paid only once per model.\n"""\n\nimport os\nimport platform\n\nimport numpy as np\n\nIS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"\n\n\ndef optimize_for_coreml(model_path: str, input_shape: tuple = None) -> str:\n    """Return path to a CoreML-optimized ONNX model.\n\n    Applies all applicable optimizations and caches the result next to\n    the original model (with ``_coreml`` suffix).\n\n    Args:\n        model_path: Path to the original ONNX model.\n        input_shape: Optional fixed input shape (e.g. ``(1, 3, 640, 640)``).\n            When provided, enables Shape/Gather constant folding.\n\n    Returns the optimized path, or the original path if no optimizations\n    apply or we\'re not on Apple Silicon.\n    """\n    if not IS_APPLE_SILICON:\n        return model_path\n\n    base, ext = os.path.splitext(model_path)\n    optimized_path = f"{base}_coreml{ext}"\n    if os.path.exists(optimized_path):\n        if os.path.getmtime(optimized_path) >= os.path.getmtime(model_path):\n            return optimized_path\n\n    import onnx\n    from onnx import numpy_helper\n\n    model = onnx.load(model_path)\n    changed = False\n\n    if _fold_shape_gather(model, input_shape):\n        changed = True\n\n    # TODO(ort>=1.26): drop this pass. Fixed upstream by microsoft/onnxruntime#28073.\n    if _decompose_reflect_pad(model):\n        changed = True\n\n    if _decompose_split(model):\n        changed = True\n\n    # TODO: drop this pass once microsoft/onnxruntime#28180 ships. The CoreML\n    # Gather op builder rejects rank-0 (scalar) indices; we widen them to [1]\n    # + Squeeze so StyleGAN-family models (GFPGAN) stay on ANE.\n    if _rewrite_scalar_gather(model):\n        changed = True\n\n    if not changed:\n        return model_path\n\n    # Preserve insightface\'s emap convention: the INSwapper class reads\n    # graph.initializer[-1] as the embedding map.  If the original model\n    # had a (512, 512) matrix as its last initializer, keep it last.\n    _preserve_emap_position(model, numpy_helper)\n\n    onnx.save(model, optimized_path)\n    return optimized_path\n\n\n# ---------------------------------------------------------------------------\n# Pass 1: Fold Shape → Gather chains into constants\n# ---------------------------------------------------------------------------\n\ndef _fold_shape_gather(model, input_shape) -> bool:\n    """Replace dynamic Shape→Gather chains with constants when input size is known.\n\n    Only removes a Shape node when ALL of its consumers are Gather nodes\n    that are also being folded.  This prevents breaking graphs where\n    a Shape output feeds into other ops as well.\n    """\n    if input_shape is None:\n        return False\n\n    from onnx import numpy_helper, shape_inference\n\n    graph = model.graph\n\n    # Set fixed input dimensions for shape inference\n    inp = graph.input[0]\n    dims = inp.type.tensor_type.shape.dim\n    for i, size in enumerate(input_shape):\n        if i < len(dims):\n            dims[i].dim_value = size\n\n    try:\n        model_inferred = shape_inference.infer_shapes(model)\n    except Exception:\n        return False\n\n    # Extract inferred shapes\n    value_shapes = {}\n    for vi in list(model_inferred.graph.value_info) + list(graph.input) + list(graph.output):\n        shape_dims = vi.type.tensor_type.shape.dim\n        shape = []\n        for d in shape_dims:\n            if d.dim_value > 0:\n                shape.append(d.dim_value)\n            else:\n                shape.append(None)\n        value_shapes[vi.name] = shape\n\n    inits = {init.name: numpy_helper.to_array(init) for init in graph.initializer}\n\n    # Build consumer map: output_name → list of consuming nodes\n    consumers = {}\n    for node in graph.node:\n        for i in node.input:\n            consumers.setdefault(i, []).append(node)\n\n    # Also check graph outputs — an output name consumed by the graph\n    # output list must not be removed\n    graph_output_names = {o.name for o in graph.output}\n\n    # Find Shape nodes with fully-known output\n    shape_constants = {}\n    for node in graph.node:\n        if node.op_type == "Shape":\n            inp_shape = value_shapes.get(node.input[0])\n            if inp_shape and all(isinstance(d, int) for d in inp_shape):\n                shape_constants[node.output[0]] = np.array(inp_shape, dtype=np.int64)\n\n    if not shape_constants:\n        return False\n\n    # Find Gather nodes consuming Shape constants\n    gather_constants = {}\n    for node in graph.node:\n        if node.op_type == "Gather" and node.input[0] in shape_constants:\n            idx_name = node.input[1]\n            if idx_name in inits:\n                idx = int(inits[idx_name])\n                val = int(shape_constants[node.input[0]][idx])\n                gather_constants[node.output[0]] = np.array(val, dtype=np.int64)\n\n    if not gather_constants:\n        return False\n\n    # Determine which Gather nodes to fold (always safe — we replace\n    # the output with a constant initializer)\n    gather_remove_ids = set()\n    for node in graph.node:\n        if node.op_type == "Gather" and node.output[0] in gather_constants:\n            gather_remove_ids.add(id(node))\n\n    # Determine which Shape nodes are safe to remove: only if ALL\n    # consumers of the Shape output are Gather nodes being folded,\n    # and the output isn\'t a graph output.\n    shape_remove_ids = set()\n    for node in graph.node:\n        if node.op_type == "Shape" and node.output[0] in shape_constants:\n            out_name = node.output[0]\n            if out_name in graph_output_names:\n                continue\n            node_consumers = consumers.get(out_name, [])\n            if all(id(c) in gather_remove_ids for c in node_consumers):\n                shape_remove_ids.add(id(node))\n\n    remove_ids = gather_remove_ids | shape_remove_ids\n\n    # Add Gather output constants as initializers\n    existing = {i.name for i in graph.initializer}\n    for name, val in gather_constants.items():\n        if name not in existing:\n            graph.initializer.append(numpy_helper.from_array(val, name=name))\n\n    new_nodes = [n for n in graph.node if id(n) not in remove_ids]\n    del graph.node[:]\n    graph.node.extend(new_nodes)\n    return True\n\n\n# ---------------------------------------------------------------------------\n# Pass 2: Decompose Pad(reflect) → Slice + Concat\n#\n# TEMPORARY: fixed upstream in microsoft/onnxruntime#28073 (merged 2026-04-20).\n# Once the ORT floor is >= 1.26.0, MLProgram handles Pad(mode=reflect) natively\n# via MIL tensor_operation.pad and this entire pass can be deleted.\n# ---------------------------------------------------------------------------\n\ndef _decompose_reflect_pad(model) -> bool:\n    """Rewrite Pad(reflect) as Slice+Concat sequences CoreML can handle."""\n    from onnx import numpy_helper, helper\n\n    graph = model.graph\n    inits = {init.name: numpy_helper.to_array(init) for init in graph.initializer}\n\n    reflect_pads = []\n    for node in graph.node:\n        if node.op_type == "Pad":\n            mode = "constant"\n            for attr in node.attribute:\n                if attr.name == "mode":\n                    mode = attr.s.decode()\n            if mode == "reflect" and len(node.input) > 1 and node.input[1] in inits:\n                reflect_pads.append(node)\n\n    if not reflect_pads:\n        return False\n\n    existing_names = {i.name for i in graph.initializer}\n\n    def ensure_const(name, value):\n        if name not in existing_names:\n            graph.initializer.append(\n                numpy_helper.from_array(np.array(value, dtype=np.int64), name=name)\n            )\n            existing_names.add(name)\n\n    ensure_const("_rp_ax2", [2])\n    ensure_const("_rp_ax3", [3])\n\n    max_pad = 0\n    for node in reflect_pads:\n        pads = inits[node.input[1]].tolist()\n        max_pad = max(max_pad, int(pads[2]), int(pads[3]))\n\n    for v in range(1, max_pad + 2):\n        ensure_const(f"_rp_p{v}", [v])\n        ensure_const(f"_rp_n{v}", [-v])\n\n    _counter = [0]\n\n    def uid():\n        _counter[0] += 1\n        return _counter[0]\n\n    pad_ids = {id(n) for n in reflect_pads}\n    pad_init_names = set()\n\n    new_nodes = []\n    for node in graph.node:\n        if id(node) not in pad_ids:\n            new_nodes.append(node)\n            continue\n\n        pads = inits[node.input[1]].tolist()\n        h_pad, w_pad = int(pads[2]), int(pads[3])\n\n        for inp in node.input[1:]:\n            if inp in inits:\n                pad_init_names.add(inp)\n\n        current = node.input[0]\n\n        if h_pad > 0:\n            top = []\n            for i in range(h_pad, 0, -1):\n                name = f"_rp_t{uid()}"\n                new_nodes.append(helper.make_node(\n                    "Slice",\n                    inputs=[current, f"_rp_p{i}", f"_rp_p{i+1}", "_rp_ax2"],\n                    outputs=[name],\n                ))\n                top.append(name)\n\n            bot = []\n            for i in range(1, h_pad + 1):\n                name = f"_rp_b{uid()}"\n                new_nodes.append(helper.make_node(\n                    "Slice",\n                    inputs=[current, f"_rp_n{i+1}", f"_rp_n{i}", "_rp_ax2"],\n                    outputs=[name],\n                ))\n                bot.append(name)\n\n            h_out = f"_rp_h{uid()}"\n            new_nodes.append(helper.make_node(\n                "Concat", inputs=top + [current] + bot, outputs=[h_out], axis=2\n            ))\n            current = h_out\n\n        if w_pad > 0:\n            left = []\n            for i in range(w_pad, 0, -1):\n                name = f"_rp_l{uid()}"\n                new_nodes.append(helper.make_node(\n                    "Slice",\n                    inputs=[current, f"_rp_p{i}", f"_rp_p{i+1}", "_rp_ax3"],\n                    outputs=[name],\n                ))\n                left.append(name)\n\n            right = []\n            for i in range(1, w_pad + 1):\n                name = f"_rp_r{uid()}"\n                new_nodes.append(helper.make_node(\n                    "Slice",\n                    inputs=[current, f"_rp_n{i+1}", f"_rp_n{i}", "_rp_ax3"],\n                    outputs=[name],\n                ))\n                right.append(name)\n\n            new_nodes.append(helper.make_node(\n                "Concat",\n                inputs=left + [current] + right,\n                outputs=[node.output[0]],\n                axis=3,\n            ))\n        elif h_pad > 0:\n            new_nodes.append(helper.make_node(\n                "Identity", inputs=[current], outputs=[node.output[0]]\n            ))\n\n    # Remove old Pad initializers\n    clean_inits = [i for i in graph.initializer if i.name not in pad_init_names]\n    del graph.initializer[:]\n    graph.initializer.extend(clean_inits)\n\n    del graph.node[:]\n    graph.node.extend(new_nodes)\n    return True\n\n\n# ---------------------------------------------------------------------------\n# Pass 3: Decompose Split → Slice pairs\n# ---------------------------------------------------------------------------\n\ndef _decompose_split(model) -> bool:\n    """Rewrite Split(axis=1) as Slice pairs that CoreML can handle.\n\n    CoreML\'s EP doesn\'t support the ONNX ``Split`` op, causing partition\n    boundaries in models that use channel-wise splits (e.g. GFPGAN\'s SFT\n    modulation layers).  Each Split with two outputs becomes two Slice ops.\n    """\n    from onnx import numpy_helper, helper\n\n    graph = model.graph\n\n    splits = []\n    for node in graph.node:\n        if node.op_type == "Split":\n            axis = 0\n            split_sizes = []\n            for attr in node.attribute:\n                if attr.name == "axis":\n                    axis = attr.i\n                if attr.name == "split":\n                    split_sizes = list(attr.ints)\n            if axis == 1 and len(split_sizes) == 2 and len(node.output) == 2:\n                splits.append((node, split_sizes))\n\n    if not splits:\n        return False\n\n    existing = {i.name for i in graph.initializer}\n\n    def ensure_const(name, value):\n        if name not in existing:\n            graph.initializer.append(\n                numpy_helper.from_array(np.array(value, dtype=np.int64), name=name)\n            )\n            existing.add(name)\n\n    ensure_const("_sp_ax1", [1])\n\n    # Collect all needed boundary constants\n    for _, (a, b) in splits:\n        ensure_const("_sp_s0", [0])\n        ensure_const(f"_sp_s{a}", [a])\n        ensure_const(f"_sp_s{a + b}", [a + b])\n\n    split_ids = {id(node) for node, _ in splits}\n    replacements = {}\n    for node, (a, b) in splits:\n        slice0 = helper.make_node(\n            "Slice",\n            inputs=[node.input[0], "_sp_s0", f"_sp_s{a}", "_sp_ax1"],\n            outputs=[node.output[0]],\n        )\n        slice1 = helper.make_node(\n            "Slice",\n            inputs=[node.input[0], f"_sp_s{a}", f"_sp_s{a + b}", "_sp_ax1"],\n            outputs=[node.output[1]],\n        )\n        replacements[id(node)] = [slice0, slice1]\n\n    new_nodes = []\n    for node in graph.node:\n        if id(node) in split_ids:\n            new_nodes.extend(replacements[id(node)])\n        else:\n            new_nodes.append(node)\n\n    del graph.node[:]\n    graph.node.extend(new_nodes)\n    return True\n\n\n# ---------------------------------------------------------------------------\n# Pass 4: Widen scalar Gather indices to [1] + Squeeze\n#\n# TEMPORARY: filed upstream as microsoft/onnxruntime#28180. ORT\'s CoreML EP\n# GatherOpBuilder::IsOpSupportedImpl rejects rank-0 (scalar) indices with\n# `Gather does not support scalar \'indices\'`. The builder\'s own comment\n# describes the workaround (promote to [1], squeeze the added axis) but\n# doesn\'t apply it. We do the same thing at the ONNX level so StyleGAN-\n# family models (GFPGAN is the hot example — 16 per-layer style-code\n# slices) don\'t split the CoreML subgraph. Once the upstream fix ships\n# and the ORT floor is raised, delete this pass.\n# ---------------------------------------------------------------------------\n\ndef _rewrite_scalar_gather(model) -> bool:\n    """Rewrite Gather(data, scalar_idx) as Gather(data, [scalar_idx]) + Squeeze.\n\n    Only touches Gather nodes whose index is a rank-0 int64 constant or\n    initializer; everything else passes through unchanged. The rewrite\n    is semantically identical — indices get an added leading axis, the\n    Squeeze removes it after the gather.\n    """\n    from onnx import numpy_helper, helper, TensorProto\n\n    graph = model.graph\n\n    # Opset 13 moved Squeeze\'s axes from attribute to input.\n    opset = next(\n        (o.version for o in model.opset_import if o.domain in ("", "ai.onnx")),\n        11,\n    )\n\n    const_values = {}\n    for n in graph.node:\n        if n.op_type == "Constant":\n            for a in n.attribute:\n                if a.name == "value":\n                    const_values[n.output[0]] = a.t\n    init_values = {i.name: i for i in graph.initializer}\n\n    def scalar_int64(name):\n        """Return int value if `name` resolves to a rank-0 int64 constant, else None."""\n        tensor = const_values.get(name) or init_values.get(name)\n        if tensor is None or tensor.data_type != TensorProto.INT64:\n            return None\n        arr = numpy_helper.to_array(tensor)\n        return int(arr) if arr.ndim == 0 else None\n\n    rewrote = 0\n    new_nodes = []\n    for n in graph.node:\n        if n.op_type == "Gather":\n            val = scalar_int64(n.input[1])\n            if val is not None:\n                axis = next((a.i for a in n.attribute if a.name == "axis"), 0)\n                idx_1d_name = f"{n.input[1]}_1d_{rewrote}"\n                idx_const = helper.make_node(\n                    "Constant",\n                    inputs=[],\n                    outputs=[idx_1d_name],\n                    value=helper.make_tensor(idx_1d_name, TensorProto.INT64, [1], [val]),\n                )\n                gather_out = f"{n.output[0]}_pre_squeeze_{rewrote}"\n                new_gather = helper.make_node(\n                    "Gather",\n                    inputs=[n.input[0], idx_1d_name],\n                    outputs=[gather_out],\n                    name=n.name,\n                    axis=axis,\n                )\n                if opset < 13:\n                    squeeze = helper.make_node(\n                        "Squeeze",\n                        inputs=[gather_out],\n                        outputs=[n.output[0]],\n                        name=(n.name or "gather") + "_squeeze",\n                        axes=[axis],\n                    )\n                    new_nodes.extend([idx_const, new_gather, squeeze])\n                else:\n                    axes_name = f"{idx_1d_name}_sq_axes"\n                    axes_const = helper.make_node(\n                        "Constant",\n                        inputs=[],\n                        outputs=[axes_name],\n                        value=helper.make_tensor(axes_name, TensorProto.INT64, [1], [axis]),\n                    )\n                    squeeze = helper.make_node(\n                        "Squeeze",\n                        inputs=[gather_out, axes_name],\n                        outputs=[n.output[0]],\n                        name=(n.name or "gather") + "_squeeze",\n                    )\n                    new_nodes.extend([idx_const, axes_const, new_gather, squeeze])\n                rewrote += 1\n                continue\n        new_nodes.append(n)\n\n    if rewrote == 0:\n        return False\n\n    del graph.node[:]\n    graph.node.extend(new_nodes)\n    return True\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef _preserve_emap_position(model, numpy_helper):\n    """Keep the insightface emap (512×512 matrix) as the last initializer."""\n    graph = model.graph\n    emap_init = None\n    for init in graph.initializer:\n        if not init.name.startswith("_rp_"):\n            arr = numpy_helper.to_array(init)\n            if len(arr.shape) == 2 and arr.shape[0] == 512 and arr.shape[1] == 512:\n                emap_init = init\n                break\n\n    if emap_init is not None:\n        inits = [i for i in graph.initializer if i.name != emap_init.name]\n        del graph.initializer[:]\n        graph.initializer.extend(inits)\n        graph.initializer.append(emap_init)\n',
    'modules/paths.py': '"""Shared path constants for the Deep-Live-Cam project."""\n\nimport os\n\nROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\nMODELS_DIR = os.path.join(ROOT_DIR, "models")\n',
    'modules/platform_info.py': '"""Centralized platform + accelerator detection.\n\nImported once at startup to expose typed flags the rest of the codebase\ncan branch on without re-querying `platform`, `torch.cuda`, or\n`onnxruntime.get_available_providers()` repeatedly.\n\nThe banner printed by :func:`print_banner` is the single user-facing\nreport of which code path the app will take.\n"""\nfrom __future__ import annotations\n\nimport platform as _platform\nimport sys\nfrom typing import List, Tuple\n\nIS_WINDOWS: bool = _platform.system() == "Windows"\nIS_MACOS: bool = _platform.system() == "Darwin"\nIS_LINUX: bool = _platform.system() == "Linux"\nIS_APPLE_SILICON: bool = IS_MACOS and _platform.machine() == "arm64"\n\n\ndef _detect_torch_cuda() -> bool:\n    try:\n        import torch  # noqa: WPS433 — local import, avoid hard dep at module load\n        return bool(torch.cuda.is_available())\n    except Exception:\n        return False\n\n\ndef _detect_onnx_providers() -> List[str]:\n    try:\n        import onnxruntime\n        return list(onnxruntime.get_available_providers())\n    except Exception:\n        return []\n\n\nHAS_TORCH_CUDA: bool = _detect_torch_cuda()\nONNX_PROVIDERS: List[str] = _detect_onnx_providers()\nHAS_CUDA_PROVIDER: bool = "CUDAExecutionProvider" in ONNX_PROVIDERS\nHAS_COREML_PROVIDER: bool = "CoreMLExecutionProvider" in ONNX_PROVIDERS\nHAS_DML_PROVIDER: bool = "DmlExecutionProvider" in ONNX_PROVIDERS\n\n\ndef camera_backends() -> List[Tuple[int, int]]:\n    """Return an ordered list of ``(device_index, cv2_backend)`` attempts.\n\n    Windows prefers MSMF (60fps capable) with DirectShow as fallback.\n    macOS/Linux use the default backend (AVFoundation / V4L2).\n    """\n    import cv2\n    if IS_WINDOWS:\n        return [\n            (0, cv2.CAP_MSMF),\n            (0, cv2.CAP_DSHOW),\n            (0, cv2.CAP_ANY),\n        ]\n    return [(0, cv2.CAP_ANY)]\n\n\ndef accelerator_label() -> str:\n    if HAS_CUDA_PROVIDER:\n        return "CUDA (NVIDIA)"\n    if IS_APPLE_SILICON and HAS_COREML_PROVIDER:\n        return "CoreML (Apple Neural Engine)"\n    if HAS_COREML_PROVIDER:\n        return "CoreML"\n    if HAS_DML_PROVIDER:\n        return "DirectML"\n    return "CPU"\n\n\ndef print_banner() -> None:\n    """Print a one-line summary of the platform + accelerator selection."""\n    os_label = f"{_platform.system()} {_platform.machine()}"\n    print(\n        f"[platform] {os_label} | python {sys.version.split()[0]} | "\n        f"accelerator: {accelerator_label()} | providers: {ONNX_PROVIDERS}",\n        flush=True,\n    )\n',
    'modules/predicter.py': 'import numpy\nimport opennsfw2\nfrom PIL import Image\nimport cv2  # Add OpenCV import\nimport modules.globals  # Import globals to access the color correction toggle\nfrom modules.gpu_processing import gpu_cvt_color\n\nfrom modules.typing import Frame\n\nMAX_PROBABILITY = 0.85\n\n# Preload the model once for efficiency\nmodel = None\n\ndef predict_frame(target_frame: Frame) -> bool:\n    # Convert the frame to RGB before processing if color correction is enabled\n    if modules.globals.color_correction:\n        target_frame = gpu_cvt_color(target_frame, cv2.COLOR_BGR2RGB)\n        \n    image = Image.fromarray(target_frame)\n    image = opennsfw2.preprocess_image(image, opennsfw2.Preprocessing.YAHOO)\n    global model\n    if model is None: \n        model = opennsfw2.make_open_nsfw_model()\n        \n    views = numpy.expand_dims(image, axis=0)\n    _, probability = model.predict(views)[0]\n    return probability > MAX_PROBABILITY\n\n\ndef predict_image(target_path: str) -> bool:\n    return opennsfw2.predict_image(target_path) > MAX_PROBABILITY\n\n\ndef predict_video(target_path: str) -> bool:\n    _, probabilities = opennsfw2.predict_video_frames(video_path=target_path, frame_interval=100)\n    return any(probability > MAX_PROBABILITY for probability in probabilities)\n',
    'modules/processors/__init__.py': '',
    'modules/processors/frame/__init__.py': '',
    'modules/processors/frame/_onnx_enhancer.py': '"""Shared ONNX-based face enhancement utilities for GPEN-BFR models.\n\nProvides session creation, pre/post processing, and the core\nenhance-face-via-ONNX pipeline.\n"""\n\nimport os\nimport platform\nimport threading\nfrom typing import Any\n\nimport cv2\nimport numpy as np\nimport onnxruntime\n\nimport modules.globals\n\nIS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"\n\n# Limit concurrent ONNX calls to avoid VRAM exhaustion on multi-face frames\nTHREAD_SEMAPHORE = threading.Semaphore(min(max(1, (os.cpu_count() or 1)), 8))\n\n\ndef build_provider_config(providers=None):\n    """Wrap raw provider name strings with optimised CUDA / CoreML options.\n\n    Providers that are already ``(name, options_dict)`` tuples are passed\n    through unchanged.  Non-CUDA providers are left as bare strings.\n    """\n    if providers is None:\n        providers = modules.globals.execution_providers\n\n    config = []\n    for p in providers:\n        if isinstance(p, tuple):\n            # Already configured – pass through\n            config.append(p)\n        elif p == "CUDAExecutionProvider":\n            # Use bare provider — ONNX Runtime\'s defaults are fastest on\n            # modern GPUs (Blackwell/sm_120).  Custom options like\n            # EXHAUSTIVE cudnn_conv_algo_search hurt performance on these\n            # architectures.\n            config.append(p)\n        elif p == "CoreMLExecutionProvider" and IS_APPLE_SILICON:\n            config.append((\n                "CoreMLExecutionProvider",\n                {\n                    "ModelFormat": "MLProgram",\n                    "MLComputeUnits": "ALL",\n                    "AllowLowPrecisionAccumulationOnGPU": 1,\n                },\n            ))\n        else:\n            config.append(p)\n    return config\n\n\ndef run_inference(session: onnxruntime.InferenceSession,\n                  input_name: str,\n                  input_tensor: "np.ndarray") -> "np.ndarray":\n    """Run ONNX inference, using IO binding when a CUDA session is active.\n\n    IO binding avoids redundant host↔device copies by transferring the\n    input tensor directly to GPU memory and letting ONNX Runtime allocate\n    the output on the device.  Falls back to the standard ``session.run``\n    path for non-CUDA providers or if binding fails.\n    """\n    if "CUDAExecutionProvider" in session.get_providers():\n        try:\n            io_binding = session.io_binding()\n\n            # Input: numpy → GPU\n            ort_input = onnxruntime.OrtValue.ortvalue_from_numpy(\n                input_tensor, "cuda", 0,\n            )\n            io_binding.bind_ortvalue_input(input_name, ort_input)\n\n            # Output: allocate on GPU (avoids a CPU-side allocation)\n            output_name = session.get_outputs()[0].name\n            io_binding.bind_output(output_name, "cuda", 0)\n\n            session.run_with_iobinding(io_binding)\n\n            return io_binding.get_outputs()[0].numpy()\n        except Exception:\n            # Fall back to standard path (e.g. ORT version mismatch,\n            # unsupported op, or VRAM pressure)\n            pass\n\n    return session.run(None, {input_name: input_tensor})[0]\n\n\ndef create_onnx_session(model_path: str) -> onnxruntime.InferenceSession:\n    """Create an ONNX Runtime session with optimised provider config.\n\n    On Apple Silicon, applies CoreML graph optimizations (Pad decomposition,\n    Shape/Gather folding, Split decomposition) to reduce CPU↔ANE partition\n    boundaries.\n    """\n    if IS_APPLE_SILICON:\n        from modules.onnx_optimize import optimize_for_coreml\n        # Infer input shape from the model for Shape/Gather folding\n        try:\n            import onnx\n            m = onnx.load(model_path)\n            inp = m.graph.input[0]\n            dims = inp.type.tensor_type.shape.dim\n            shape = tuple(d.dim_value for d in dims if d.dim_value > 0)\n            input_shape = shape if len(shape) == 4 else None\n        except Exception:\n            input_shape = None\n        model_path = optimize_for_coreml(model_path, input_shape=input_shape)\n\n    providers = build_provider_config()\n    session_options = onnxruntime.SessionOptions()\n    session_options.graph_optimization_level = (\n        onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL\n    )\n    session = onnxruntime.InferenceSession(\n        model_path, sess_options=session_options, providers=providers,\n    )\n    return session\n\n\ndef warmup_session(session: onnxruntime.InferenceSession) -> None:\n    """Run a dummy inference pass to trigger JIT / compile caching."""\n    try:\n        input_feed = {\n            inp.name: np.zeros(\n                [d if isinstance(d, int) and d > 0 else 1 for d in inp.shape],\n                dtype=np.float32,\n            )\n            for inp in session.get_inputs()\n        }\n        session.run(None, input_feed)\n    except Exception as e:\n        print(f"ONNX enhancer warmup skipped (non-fatal): {e}")\n\n\ndef preprocess_face(face_img: np.ndarray, input_size: int) -> np.ndarray:\n    """Resize, normalize, and convert a BGR face crop to ONNX input blob.\n\n    GPEN-BFR expects [1, 3, H, W] float32 in RGB, normalized to [-1, 1].\n    """\n    resized = cv2.resize(face_img, (input_size, input_size), interpolation=cv2.INTER_LINEAR)\n    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)\n    blob = rgb.astype(np.float32) / 255.0 * 2.0 - 1.0\n    blob = np.transpose(blob, (2, 0, 1))[np.newaxis, ...]\n    return blob\n\n\ndef postprocess_face(output: np.ndarray) -> np.ndarray:\n    """Convert ONNX output [1, 3, H, W] float32 back to BGR uint8 image."""\n    img = output[0].transpose(1, 2, 0)\n    img = ((img + 1.0) / 2.0 * 255.0)\n    img = np.clip(img, 0, 255).astype(np.uint8)\n    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)\n    return img\n\n\ndef _get_face_affine(face: Any, input_size: int):\n    """Compute affine transform to align a face to GPEN input space.\n\n    Returns (M, inv_M) — forward and inverse affine matrices.\n    """\n    template = np.array([\n        [0.31556875, 0.4615741],\n        [0.68262291, 0.4615741],\n        [0.50009375, 0.6405054],\n        [0.34947187, 0.8246919],\n        [0.65343645, 0.8246919],\n    ], dtype=np.float32) * input_size\n\n    landmarks = None\n    if hasattr(face, "kps") and face.kps is not None:\n        landmarks = face.kps.astype(np.float32)\n    elif hasattr(face, "landmark_2d_106") and face.landmark_2d_106 is not None:\n        lm106 = face.landmark_2d_106\n        landmarks = np.array([\n            lm106[38],  # left eye\n            lm106[88],  # right eye\n            lm106[86],  # nose tip\n            lm106[52],  # left mouth\n            lm106[61],  # right mouth\n        ], dtype=np.float32)\n\n    if landmarks is None or len(landmarks) < 5:\n        return None, None\n\n    M = cv2.estimateAffinePartial2D(landmarks, template, method=cv2.LMEDS)[0]\n    if M is None:\n        return None, None\n    inv_M = cv2.invertAffineTransform(M)\n    return M, inv_M\n\n\ndef enhance_face_onnx(\n    frame: np.ndarray,\n    face: Any,\n    session: onnxruntime.InferenceSession,\n    input_size: int,\n) -> np.ndarray:\n    """Enhance a single face in the frame using an ONNX face restoration model."""\n    M, inv_M = _get_face_affine(face, input_size)\n    if M is None:\n        return frame\n\n    face_crop = cv2.warpAffine(\n        frame, M, (input_size, input_size),\n        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE,\n    )\n\n    blob = preprocess_face(face_crop, input_size)\n    with THREAD_SEMAPHORE:\n        input_name = session.get_inputs()[0].name\n        output = run_inference(session, input_name, blob)\n    enhanced = postprocess_face(output)\n\n    # Create mask for blending (feathered edges)\n    mask = np.ones((input_size, input_size), dtype=np.float32)\n    border = max(1, input_size // 16)\n    mask[:border, :] = np.linspace(0, 1, border)[:, np.newaxis]\n    mask[-border:, :] = np.linspace(1, 0, border)[:, np.newaxis]\n    mask[:, :border] = np.minimum(mask[:, :border], np.linspace(0, 1, border)[np.newaxis, :])\n    mask[:, -border:] = np.minimum(mask[:, -border:], np.linspace(1, 0, border)[np.newaxis, :])\n\n    h, w = frame.shape[:2]\n    warped_enhanced = cv2.warpAffine(\n        enhanced, inv_M, (w, h),\n        flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0),\n    )\n    warped_mask = cv2.warpAffine(\n        mask, inv_M, (w, h),\n        flags=cv2.INTER_LINEAR, borderValue=0,\n    )\n\n    mask_3ch = warped_mask[:, :, np.newaxis]\n    result = (warped_enhanced.astype(np.float32) * mask_3ch +\n              frame.astype(np.float32) * (1.0 - mask_3ch))\n    return np.clip(result, 0, 255).astype(np.uint8)\n',
    'modules/processors/frame/core.py': 'import os\nimport subprocess\nimport sys\nimport importlib\nfrom concurrent.futures import ThreadPoolExecutor\nfrom types import ModuleType\nfrom typing import Any, List, Callable\n\nimport numpy as np\nfrom tqdm import tqdm\n\nimport modules\nimport modules.globals\nfrom modules.face_analyser import get_one_face\n\nFRAME_PROCESSORS_MODULES: List[ModuleType] = []\nFRAME_PROCESSORS_INTERFACE = [\n    \'pre_check\',\n    \'pre_start\',\n    \'process_frame\',\n    \'process_image\',\n    \'process_video\'\n]\n\nALLOWED_PROCESSORS = {\n    \'face_swapper\',\n    \'face_enhancer\',\n    \'face_enhancer_gpen256\',\n    \'face_enhancer_gpen512\'\n}\n\ndef load_frame_processor_module(frame_processor: str) -> Any:\n    if frame_processor not in ALLOWED_PROCESSORS:\n        print(f"Frame processor {frame_processor} is not allowed")\n        sys.exit()\n    try:\n        frame_processor_module = importlib.import_module(f\'modules.processors.frame.{frame_processor}\')\n        for method_name in FRAME_PROCESSORS_INTERFACE:\n            if not hasattr(frame_processor_module, method_name):\n                print(f"Frame processor {frame_processor} is missing required method {method_name}")\n                sys.exit()\n    except ImportError:\n        print(f"Frame processor {frame_processor} not found")\n        sys.exit()\n    return frame_processor_module\n\n\ndef get_frame_processors_modules(frame_processors: List[str]) -> List[ModuleType]:\n    global FRAME_PROCESSORS_MODULES\n\n    if not FRAME_PROCESSORS_MODULES:\n        for frame_processor in frame_processors:\n            frame_processor_module = load_frame_processor_module(frame_processor)\n            FRAME_PROCESSORS_MODULES.append(frame_processor_module)\n    set_frame_processors_modules_from_ui(frame_processors)\n    return FRAME_PROCESSORS_MODULES\n\ndef set_frame_processors_modules_from_ui(frame_processors: List[str]) -> None:\n    global FRAME_PROCESSORS_MODULES\n    current_processor_names = [proc.__name__.split(\'.\')[-1] for proc in FRAME_PROCESSORS_MODULES]\n\n    for frame_processor, state in modules.globals.fp_ui.items():\n        if state and frame_processor not in current_processor_names:\n            try:\n                frame_processor_module = load_frame_processor_module(frame_processor)\n                FRAME_PROCESSORS_MODULES.append(frame_processor_module)\n                if frame_processor not in modules.globals.frame_processors:\n                     modules.globals.frame_processors.append(frame_processor)\n            except SystemExit:\n                 print(f"Warning: Failed to load frame processor {frame_processor} requested by UI state.")\n            except Exception as e:\n                 print(f"Warning: Error loading frame processor {frame_processor} requested by UI state: {e}")\n\n        elif not state and frame_processor in current_processor_names:\n            try:\n                module_to_remove = next((mod for mod in FRAME_PROCESSORS_MODULES if mod.__name__.endswith(f\'.{frame_processor}\')), None)\n                if module_to_remove:\n                    FRAME_PROCESSORS_MODULES.remove(module_to_remove)\n                if frame_processor in modules.globals.frame_processors:\n                    modules.globals.frame_processors.remove(frame_processor)\n            except Exception as e:\n                 print(f"Warning: Error removing frame processor {frame_processor}: {e}")\n\ndef multi_process_frame(source_path: str, temp_frame_paths: List[str], process_frames: Callable[[str, List[str], Any], None], progress: Any = None) -> None:\n    """Process frames in parallel with optimized batching and memory management."""\n    max_workers = modules.globals.execution_threads\n    \n    # Determine optimal batch size based on available memory and thread count\n    # Process frames in batches to avoid memory overflow\n    batch_size = max(1, min(32, len(temp_frame_paths) // max(1, max_workers)))\n    \n    with ThreadPoolExecutor(max_workers=max_workers) as executor:\n        # Process in batches to manage memory better\n        for i in range(0, len(temp_frame_paths), batch_size):\n            batch = temp_frame_paths[i:i + batch_size]\n            futures = []\n            \n            for path in batch:\n                future = executor.submit(process_frames, source_path, [path], progress)\n                futures.append(future)\n            \n            # Wait for batch to complete before starting next batch\n            for future in futures:\n                try:\n                    future.result()\n                except Exception as e:\n                    print(f"Error processing frame: {e}")\n\n\ndef process_video(source_path: str, frame_paths: list[str], process_frames: Callable[[str, List[str], Any], None]) -> None:\n    progress_bar_format = \'{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]\'\n    total = len(frame_paths)\n    with tqdm(total=total, desc=\'Processing\', unit=\'frame\', dynamic_ncols=True, bar_format=progress_bar_format) as progress:\n        progress.set_postfix({\'execution_providers\': modules.globals.execution_providers, \'execution_threads\': modules.globals.execution_threads, \'max_memory\': modules.globals.max_memory})\n        multi_process_frame(source_path, frame_paths, process_frames, progress)\n\n\ndef process_video_in_memory(source_path: str, target_path: str, fps: float) -> bool:\n    """Process video frames in-memory using FFmpeg pipes, eliminating disk I/O.\n\n    Reads raw frames from the source video via an FFmpeg decoder pipe, runs each\n    frame through all active frame processors sequentially, and writes the\n    result directly to an FFmpeg encoder pipe.  This avoids extracting frames to\n    PNG on disk, which is the biggest I/O bottleneck in the disk-based pipeline.\n\n    Returns True on success, False on failure (caller should fall back to the\n    disk-based pipeline).\n    """\n    from modules import imread_unicode\n    from modules.face_analyser import get_one_face\n    from modules.utilities import (\n        get_video_dimensions,\n        estimate_frame_count,\n        get_temp_output_path,\n    )\n\n    temp_output_path = get_temp_output_path(target_path)\n\n    # --- Pre-load source face (needed by face_swapper in simple mode) ---\n    source_face = None\n    if source_path and os.path.exists(source_path):\n        source_img = imread_unicode(source_path)\n        if source_img is not None:\n            source_face = get_one_face(source_img)\n            del source_img\n        if source_face is None:\n            print("[DLC.CORE] Warning: No face detected in source image. "\n                  "Face swapping will be skipped.")\n\n    # --- Collect frame processors & reset per-video state ---\n    frame_processors = get_frame_processors_modules(modules.globals.frame_processors)\n    for fp in frame_processors:\n        if hasattr(fp, \'PREVIOUS_FRAME_RESULT\'):\n            fp.PREVIOUS_FRAME_RESULT = None\n\n    # --- Video metadata ---\n    try:\n        width, height = get_video_dimensions(target_path)\n    except Exception as e:\n        print(f"[DLC.CORE] Failed to get video dimensions: {e}")\n        return False\n\n    total_frames = estimate_frame_count(target_path, fps)\n    frame_size = width * height * 3\n\n    # --- Build encoder arguments ---\n    encoder = modules.globals.video_encoder\n    encoder_options: List[str] = []\n    is_hw_encoder = False\n\n    if \'CUDAExecutionProvider\' in modules.globals.execution_providers:\n        if encoder == \'libx264\':\n            encoder = \'h264_nvenc\'\n            is_hw_encoder = True\n            encoder_options = [\n                \'-preset\', \'p4\', \'-tune\', \'hq\', \'-rc\', \'vbr\',\n                \'-cq\', str(modules.globals.video_quality), \'-b:v\', \'0\',\n            ]\n        elif encoder == \'libx265\':\n            encoder = \'hevc_nvenc\'\n            is_hw_encoder = True\n            encoder_options = [\n                \'-preset\', \'p4\', \'-tune\', \'hq\', \'-rc\', \'vbr\',\n                \'-cq\', str(modules.globals.video_quality), \'-b:v\', \'0\',\n            ]\n    elif \'DmlExecutionProvider\' in modules.globals.execution_providers:\n        if encoder == \'libx264\':\n            encoder = \'h264_amf\'\n            is_hw_encoder = True\n            encoder_options = [\n                \'-quality\', \'quality\', \'-rc\', \'vbr_latency\',\n                \'-qp_i\', str(modules.globals.video_quality),\n                \'-qp_p\', str(modules.globals.video_quality),\n            ]\n        elif encoder == \'libx265\':\n            encoder = \'hevc_amf\'\n            is_hw_encoder = True\n            encoder_options = [\n                \'-quality\', \'quality\', \'-rc\', \'vbr_latency\',\n                \'-qp_i\', str(modules.globals.video_quality),\n                \'-qp_p\', str(modules.globals.video_quality),\n            ]\n\n    if not is_hw_encoder:\n        if encoder == \'libx264\':\n            encoder_options = [\n                \'-preset\', \'medium\',\n                \'-crf\', str(modules.globals.video_quality),\n                \'-tune\', \'film\',\n            ]\n        elif encoder == \'libx265\':\n            encoder_options = [\n                \'-preset\', \'medium\',\n                \'-crf\', str(modules.globals.video_quality),\n                \'-x265-params\', \'log-level=error\',\n            ]\n        elif encoder == \'libvpx-vp9\':\n            encoder_options = [\n                \'-crf\', str(modules.globals.video_quality),\n                \'-b:v\', \'0\', \'-cpu-used\', \'2\',\n            ]\n\n    # --- Attempt pipeline (hw encoder first, then sw fallback) ---\n    encoders_to_try = [(encoder, encoder_options)]\n    if is_hw_encoder:\n        # Software fallback\n        sw_encoder = \'libx264\'\n        sw_options = [\n            \'-preset\', \'medium\',\n            \'-crf\', str(modules.globals.video_quality),\n            \'-tune\', \'film\',\n        ]\n        encoders_to_try.append((sw_encoder, sw_options))\n\n    for attempt, (enc, enc_opts) in enumerate(encoders_to_try):\n        # Reset interpolation state on retry\n        if attempt > 0:\n            for fp in frame_processors:\n                if hasattr(fp, \'PREVIOUS_FRAME_RESULT\'):\n                    fp.PREVIOUS_FRAME_RESULT = None\n\n        success = _run_pipe_pipeline(\n            target_path, temp_output_path, fps,\n            source_face, frame_processors,\n            width, height, frame_size, total_frames,\n            enc, enc_opts,\n        )\n        if success:\n            return True\n\n        if attempt == 0 and is_hw_encoder:\n            print(f"[DLC.CORE] Hardware encoder \'{enc}\' failed, "\n                  f"retrying with software encoder...")\n\n    return False\n\n\ndef _run_pipe_pipeline(\n    target_path: str,\n    temp_output_path: str,\n    fps: float,\n    source_face: Any,\n    frame_processors: List[Any],\n    width: int,\n    height: int,\n    frame_size: int,\n    total_frames: int,\n    encoder: str,\n    encoder_options: List[str],\n) -> bool:\n    """Run the FFmpeg-pipe read → process → encode pipeline once."""\n\n    # --- Reader: decode source video to raw BGR24 on stdout ---\n    reader_cmd = [\n        \'ffmpeg\', \'-hide_banner\',\n        \'-hwaccel\', \'auto\',\n        \'-i\', target_path,\n        \'-f\', \'rawvideo\',\n        \'-pix_fmt\', \'bgr24\',\n        \'-v\', \'error\',\n        \'-\',\n    ]\n\n    # --- Writer: encode raw BGR24 from stdin ---\n    writer_cmd = [\n        \'ffmpeg\', \'-hide_banner\',\n        \'-f\', \'rawvideo\',\n        \'-pix_fmt\', \'bgr24\',\n        \'-s\', f\'{width}x{height}\',\n        \'-r\', str(fps),\n        \'-i\', \'-\',\n        \'-c:v\', encoder,\n    ]\n    writer_cmd.extend(encoder_options)\n    writer_cmd.extend([\n        \'-pix_fmt\', \'yuv420p\',\n        \'-movflags\', \'+faststart\',\n        \'-vf\', \'colorspace=bt709:iall=bt601-6-625:fast=1\',\n        \'-v\', \'error\',\n        \'-y\', temp_output_path,\n    ])\n\n    reader = None\n    writer = None\n    try:\n        reader = subprocess.Popen(\n            reader_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,\n        )\n        writer = subprocess.Popen(\n            writer_cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE,\n        )\n    except Exception as e:\n        print(f"[DLC.CORE] Failed to start FFmpeg pipes: {e}")\n        for proc in (reader, writer):\n            if proc:\n                try:\n                    proc.kill()\n                except Exception:\n                    pass\n        return False\n\n    processed_count = 0\n    bar_fmt = (\'{l_bar}{bar}| {n_fmt}/{total_fmt} \'\n               \'[{elapsed}<{remaining}, {rate_fmt}{postfix}]\')\n\n    try:\n        with tqdm(total=total_frames, desc=\'Processing\', unit=\'frame\',\n                  dynamic_ncols=True, bar_format=bar_fmt) as progress:\n            progress.set_postfix({\n                \'execution_providers\': modules.globals.execution_providers,\n                \'threads\': modules.globals.execution_threads,\n                \'mode\': \'in-memory\',\n            })\n\n            # Pipelined detection: while processing frame N (swap on\n            # ANE), start detecting the face in the next frame\n            # (detection on GPU).  They use different hardware units\n            # so the work overlaps.\n            detect_executor = ThreadPoolExecutor(max_workers=1)\n            pending_detect = None\n            use_pipeline = not modules.globals.many_faces\n\n            while True:\n                raw = reader.stdout.read(frame_size)\n                if len(raw) != frame_size:\n                    break\n\n                frame = np.frombuffer(raw, dtype=np.uint8).reshape(\n                    (height, width, 3)\n                ).copy()\n\n                # Get the detection result for THIS frame\n                if use_pipeline:\n                    if pending_detect is not None:\n                        target_face = pending_detect.result()\n                    else:\n                        target_face = get_one_face(frame)\n                    # Start detecting on THIS frame eagerly — the result\n                    # will be used for the next iteration.  At video\n                    # frame rates the face barely moves between frames.\n                    # Hand the detector its own copy: the frame processors\n                    # below mutate `frame` in place (paste-back), which\n                    # would otherwise race with detection.\n                    pending_detect = detect_executor.submit(\n                        get_one_face, frame.copy())\n                else:\n                    target_face = None\n\n                # Run frame through every active processor\n                for fp in frame_processors:\n                    try:\n                        frame = fp.process_frame(source_face, frame, target_face=target_face)\n                    except TypeError:\n                        frame = fp.process_frame(source_face, frame)\n\n                writer.stdin.write(frame.tobytes())\n                processed_count += 1\n                progress.update(1)\n\n            detect_executor.shutdown(wait=True)\n\n        # Graceful shutdown\n        writer.stdin.close()\n        writer.wait()\n        reader.wait()\n\n        if writer.returncode != 0:\n            stderr_out = writer.stderr.read().decode(errors=\'ignore\').strip()\n            if stderr_out:\n                print(f"[DLC.CORE] FFmpeg encoder error: {stderr_out}")\n            return False\n\n        return processed_count > 0 and os.path.isfile(temp_output_path)\n\n    except BrokenPipeError:\n        print("[DLC.CORE] FFmpeg pipe broken (encoder may not be available).")\n        return False\n    except Exception as e:\n        print(f"[DLC.CORE] In-memory processing error: {e}")\n        return False\n    finally:\n        for proc in (reader, writer):\n            if proc:\n                try:\n                    proc.kill()\n                except Exception:\n                    pass\n',
    'modules/processors/frame/face_enhancer.py': '# Uses ONNX Runtime for GFPGAN face enhancement (no torch/gfpgan dependency)\n\nfrom typing import Any, List\nimport cv2\nimport threading\nimport numpy as np\nimport os\n\nimport onnxruntime\n\nimport modules.globals\nimport modules.processors.frame.core\nfrom modules import imread_unicode, imwrite_unicode\nfrom modules.core import update_status\nfrom modules.face_analyser import get_many_faces\nfrom modules.typing import Frame, Face\nfrom modules.utilities import (\n    is_image,\n    is_video,\n)\n\nFACE_ENHANCER = None\nTHREAD_SEMAPHORE = threading.Semaphore()\nTHREAD_LOCK = threading.Lock()\nNAME = "DLC.FACE-ENHANCER"\n\nabs_dir = os.path.dirname(os.path.abspath(__file__))\nmodels_dir = os.path.join(\n    os.path.dirname(os.path.dirname(os.path.dirname(abs_dir))), "models"\n)\n\n# Standard FFHQ 5-point face template for 512x512 resolution\n# Points: left_eye, right_eye, nose, left_mouth, right_mouth\nFFHQ_TEMPLATE_512 = np.array(\n    [\n        [192.98138, 239.94708],\n        [318.90277, 240.19366],\n        [256.63416, 314.01935],\n        [201.26117, 371.41043],\n        [313.08905, 371.15118],\n    ],\n    dtype=np.float32,\n)\n\n\ndef pre_check() -> bool:\n    model_path = os.path.join(models_dir, "gfpgan-1024.onnx")\n    if not os.path.exists(model_path):\n        update_status(\n            f"GFPGAN ONNX model not found at {model_path}. "\n            "Please place gfpgan-1024.onnx in the models folder.",\n            NAME,\n        )\n        return False\n    return True\n\n\ndef pre_start() -> bool:\n    if not is_image(modules.globals.target_path) and not is_video(\n        modules.globals.target_path\n    ):\n        update_status("Select an image or video for target path.", NAME)\n        return False\n    return True\n\n\ndef get_face_enhancer() -> onnxruntime.InferenceSession:\n    """\n    Initializes and returns the GFPGAN ONNX Runtime inference session,\n    using the execution providers configured in modules.globals.\n    """\n    global FACE_ENHANCER\n\n    with THREAD_LOCK:\n        if FACE_ENHANCER is None:\n            model_path = os.path.join(models_dir, "gfpgan-1024.onnx")\n\n            if not os.path.exists(model_path):\n                raise FileNotFoundError(\n                    f"{NAME}: Model not found at {model_path}"\n                )\n\n            try:\n                from modules.processors.frame._onnx_enhancer import (\n                    create_onnx_session,\n                )\n\n                FACE_ENHANCER = create_onnx_session(model_path)\n\n                input_info = FACE_ENHANCER.get_inputs()[0]\n                output_info = FACE_ENHANCER.get_outputs()[0]\n                active_providers = FACE_ENHANCER.get_providers()\n                print(\n                    f"{NAME}: GFPGAN ONNX model loaded successfully."\n                )\n                print(\n                    f"{NAME}: Input: {input_info.name}, "\n                    f"shape: {input_info.shape}, type: {input_info.type}"\n                )\n                print(\n                    f"{NAME}: Output: {output_info.name}, "\n                    f"shape: {output_info.shape}, type: {output_info.type}"\n                )\n                print(f"{NAME}: Active providers: {active_providers}")\n\n            except Exception as e:\n                print(f"{NAME}: Error loading GFPGAN ONNX model: {e}")\n                FACE_ENHANCER = None\n                raise RuntimeError(\n                    f"{NAME}: Failed to load GFPGAN ONNX model: {e}"\n                )\n\n    if FACE_ENHANCER is None:\n        raise RuntimeError(\n            f"{NAME}: Failed to initialize GFPGAN ONNX session. Check logs."\n        )\n\n    return FACE_ENHANCER\n\n\ndef _align_face(\n    frame: Frame, landmarks_5: np.ndarray, output_size: int\n) -> tuple:\n    """\n    Align and crop a face from the frame using 5-point landmarks and the\n    standard FFHQ template.\n\n    Returns:\n        (aligned_face, affine_matrix) or (None, None) on failure.\n    """\n    # Scale the 512-base template to the desired output size\n    scale = output_size / 512.0\n    template = FFHQ_TEMPLATE_512 * scale\n\n    # Estimate a similarity transform (4 DOF: rotation, scale, tx, ty)\n    affine_matrix, _ = cv2.estimateAffinePartial2D(\n        landmarks_5, template, method=cv2.LMEDS\n    )\n    if affine_matrix is None:\n        return None, None\n\n    # Warp the face to the aligned position\n    aligned_face = cv2.warpAffine(\n        frame,\n        affine_matrix,\n        (output_size, output_size),\n        borderMode=cv2.BORDER_CONSTANT,\n        borderValue=(135, 133, 132),\n    )\n\n    return aligned_face, affine_matrix\n\n\n_HAS_TORCH_CUDA = False\ntry:\n    import torch\n    if torch.cuda.is_available():\n        _HAS_TORCH_CUDA = True\nexcept ImportError:\n    pass\n\n# Cache the feathered mask — it\'s the same for every call at a given size\n_enhancer_cache: dict = {\'mask\': None, \'mask_size\': 0}\n\n\ndef _paste_back(\n    frame: Frame,\n    enhanced_face: np.ndarray,\n    affine_matrix: np.ndarray,\n    output_size: int,\n) -> Frame:\n    """\n    Paste an enhanced (aligned) face back onto the original frame using the\n    inverse affine transform with feathered-edge blending.\n\n    Optimized: operates on a tight crop around the face bbox instead of the\n    full frame, and uses GPU for blending when available.\n    """\n    h, w = frame.shape[:2]\n    inv_matrix = cv2.invertAffineTransform(affine_matrix)\n\n    # Build or reuse cached feathered mask (uint8 — blended via cv2 SIMD ops)\n    if _enhancer_cache[\'mask_size\'] != output_size:\n        face_mask_f = np.ones((output_size, output_size), dtype=np.float32)\n        border = max(1, int(output_size * 0.05))\n        ramp_up = np.linspace(0.0, 1.0, border, dtype=np.float32)\n        ramp_down = np.linspace(1.0, 0.0, border, dtype=np.float32)\n        face_mask_f[:border, :] *= ramp_up[:, None]\n        face_mask_f[-border:, :] *= ramp_down[:, None]\n        face_mask_f[:, :border] *= ramp_up[None, :]\n        face_mask_f[:, -border:] *= ramp_down[None, :]\n        _enhancer_cache[\'mask\'] = (face_mask_f * 255.0).astype(np.uint8)\n        _enhancer_cache[\'mask_size\'] = output_size\n\n    # Compute tight bbox from affine corners (avoids full-frame warpAffine scan)\n    corners = np.array([[0, 0], [output_size, 0],\n                        [output_size, output_size], [0, output_size]],\n                       dtype=np.float32)\n    transformed = (inv_matrix[:, :2] @ corners.T).T + inv_matrix[:, 2]\n    x1 = max(0, int(np.floor(transformed[:, 0].min())))\n    x2 = min(w, int(np.ceil(transformed[:, 0].max())))\n    y1 = max(0, int(np.floor(transformed[:, 1].min())))\n    y2 = min(h, int(np.ceil(transformed[:, 1].max())))\n    if x1 >= x2 or y1 >= y2:\n        return frame\n\n    # Pad a few pixels for feathering\n    pad = max(1, int(output_size * 0.05)) + 2\n    y1p, y2p = max(0, y1 - pad), min(h, y2 + pad)\n    x1p, x2p = max(0, x1 - pad), min(w, x2 + pad)\n    crop_w, crop_h = x2p - x1p, y2p - y1p\n\n    # Warp enhanced face and mask into crop space only\n    inv_crop = inv_matrix.copy()\n    inv_crop[0, 2] -= x1p\n    inv_crop[1, 2] -= y1p\n\n    inv_restored_crop = cv2.warpAffine(\n        enhanced_face, inv_crop, (crop_w, crop_h),\n        borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0),\n    )\n    inv_mask_crop = cv2.warpAffine(\n        _enhancer_cache[\'mask\'], inv_crop, (crop_w, crop_h),\n        borderMode=cv2.BORDER_CONSTANT, borderValue=0,\n    )\n\n    target_crop = frame[y1p:y2p, x1p:x2p]\n\n    if _HAS_TORCH_CUDA:\n        # Upload uint8 alpha — smaller transfer, scale on device.\n        mask_t = torch.from_numpy(inv_mask_crop).cuda().float().mul_(1.0 / 255.0).unsqueeze(2)\n        enhanced_t = torch.from_numpy(inv_restored_crop).float().cuda()\n        target_t = torch.from_numpy(target_crop).float().cuda()\n        blended = (mask_t * enhanced_t + (1.0 - mask_t) * target_t\n                   ).to(torch.uint8).cpu().numpy()\n        frame[y1p:y2p, x1p:x2p] = blended\n    else:\n        # Fused uint8 blend via cv2 SIMD — ~7× faster than the float32 round-trip.\n        alpha_3c = cv2.merge([inv_mask_crop, inv_mask_crop, inv_mask_crop])\n        inv_alpha = 255 - alpha_3c\n        a_enh = cv2.multiply(inv_restored_crop, alpha_3c, scale=1.0 / 255.0)\n        a_tgt = cv2.multiply(target_crop, inv_alpha, scale=1.0 / 255.0)\n        frame[y1p:y2p, x1p:x2p] = cv2.add(a_enh, a_tgt)\n\n    return frame\n\n\ndef _preprocess_face(aligned_face: np.ndarray) -> np.ndarray:\n    """\n    Convert an aligned BGR uint8 face image to the ONNX model input tensor.\n    Format: NCHW float32, normalised to [-1, 1].\n    """\n    # BGR -> RGB, normalize, and transpose in one pass\n    # Fused: (x / 255.0 - 0.5) / 0.5 = x / 127.5 - 1.0\n    rgb = aligned_face[:, :, ::-1]  # BGR->RGB zero-copy view\n    chw = np.transpose(rgb, (2, 0, 1)).astype(np.float32)\n    chw *= (1.0 / 127.5)\n    chw -= 1.0\n    return chw[np.newaxis, ...]  # shape: (1, 3, H, W)\n\n\ndef _postprocess_face(output: np.ndarray) -> np.ndarray:\n    """\n    Convert the ONNX model output tensor back to a BGR uint8 image.\n    Expects input in NCHW format with values in [-1, 1].\n    """\n    # Fused: ((x + 1.0) / 2.0) * 255 = (x + 1.0) * 127.5\n    face = output[0]  # remove batch dim -> (3, H, W)\n    face = (face + 1.0) * 127.5\n    np.clip(face, 0, 255, out=face)\n    face = face.astype(np.uint8).transpose(1, 2, 0)  # CHW -> HWC\n    return face[:, :, ::-1].copy()  # RGB -> BGR\n\n\n# Cache for temporal enhancement skipping in live mode.\n# GFPGAN output barely changes between consecutive frames (same face,\n# same position), so we run inference every _ENH_INTERVAL frames and\n# reuse the cached enhanced face + affine matrix in between.\n_enh_live_cache: dict = {\n    \'enhanced_bgr\': None,\n    \'affine_matrix\': None,\n    \'align_size\': 0,\n    \'frame_count\': 0,\n}\n_ENH_INTERVAL = 2  # run inference every N frames, paste cached result otherwise\n\n\ndef enhance_face(temp_frame: Frame, detected_faces=None) -> Frame:\n    """Enhances all faces in a frame using the GFPGAN ONNX model.\n\n    Args:\n        detected_faces: Pre-detected face list. When provided, skips\n            the internal detection call (saves ~15-20ms per frame).\n            Also enables temporal caching — inference runs every\n            _ENH_INTERVAL frames, reusing the cached result otherwise.\n    """\n    session = get_face_enhancer()\n\n    # Determine model input resolution from the session metadata\n    input_info = session.get_inputs()[0]\n    input_name = input_info.name\n    input_shape = input_info.shape  # e.g. [1, 3, 512, 512]\n    try:\n        align_size = int(input_shape[2])\n        if align_size <= 0:\n            align_size = 512\n    except (ValueError, TypeError, IndexError):\n        align_size = 512\n\n    # Use pre-detected faces if available, otherwise detect\n    faces = detected_faces if detected_faces is not None else get_many_faces(temp_frame)\n    if not faces:\n        return temp_frame\n\n    # Temporal caching: only available when faces are pre-detected (live mode)\n    # AND we\'re in single-face mode — the cache holds exactly one enhancement,\n    # so reusing it in many_faces mode would paste the same face onto every\n    # detected target.\n    many_faces_mode = getattr(modules.globals, "many_faces", False)\n    use_cache = detected_faces is not None and not many_faces_mode\n    if use_cache:\n        _enh_live_cache[\'frame_count\'] += 1\n        run_inference_this_frame = (_enh_live_cache[\'frame_count\'] % _ENH_INTERVAL == 0\n                                   or _enh_live_cache[\'enhanced_bgr\'] is None)\n    else:\n        run_inference_this_frame = True\n\n    for face in faces:\n        if not hasattr(face, "kps") or face.kps is None:\n            continue\n\n        landmarks_5 = face.kps.astype(np.float32)\n        if landmarks_5.shape[0] < 5:\n            continue\n\n        if run_inference_this_frame:\n            aligned_face, affine_matrix = _align_face(\n                temp_frame, landmarks_5, output_size=align_size\n            )\n            if aligned_face is None or affine_matrix is None:\n                continue\n\n            try:\n                with THREAD_SEMAPHORE:\n                    from modules.processors.frame._onnx_enhancer import (\n                        run_inference,\n                    )\n                    input_tensor = _preprocess_face(aligned_face)\n                    output_tensor = run_inference(session, input_name, input_tensor)\n                    enhanced_bgr = _postprocess_face(output_tensor)\n\n                eh, ew = enhanced_bgr.shape[:2]\n                if eh != align_size or ew != align_size:\n                    enhanced_bgr = cv2.resize(\n                        enhanced_bgr,\n                        (align_size, align_size),\n                        interpolation=cv2.INTER_LANCZOS4,\n                    )\n\n                # Cache for reuse on next frame\n                if use_cache:\n                    _enh_live_cache[\'enhanced_bgr\'] = enhanced_bgr\n                    _enh_live_cache[\'affine_matrix\'] = affine_matrix\n                    _enh_live_cache[\'align_size\'] = align_size\n\n                _paste_back(\n                    temp_frame, enhanced_bgr, affine_matrix, output_size=align_size\n                )\n            except Exception as e:\n                print(f"{NAME}: Error enhancing a face: {e}")\n                continue\n        else:\n            # Reuse cached enhanced face — just paste back onto current frame\n            cached = _enh_live_cache\n            if cached[\'enhanced_bgr\'] is not None:\n                _paste_back(\n                    temp_frame, cached[\'enhanced_bgr\'],\n                    cached[\'affine_matrix\'],\n                    output_size=cached[\'align_size\'],\n                )\n        if not many_faces_mode:\n            break  # single-face live mode — only process first face\n\n    return temp_frame\n\n\ndef process_frame(source_face: Face | None, temp_frame: Frame,\n                   detected_faces=None) -> Frame:\n    """Processes a frame: enhances face if detected."""\n    return enhance_face(temp_frame, detected_faces=detected_faces)\n\n\ndef process_frame_v2(temp_frame: Frame, detected_faces=None) -> Frame:\n    """Processes a frame without source face (used by live webcam preview)."""\n    return enhance_face(temp_frame, detected_faces=detected_faces)\n\n\ndef process_frames(\n    source_path: str | None, temp_frame_paths: List[str], progress: Any = None\n) -> None:\n    """Processes multiple frames from file paths."""\n    for temp_frame_path in temp_frame_paths:\n        if not os.path.exists(temp_frame_path):\n            print(\n                f"{NAME}: Warning: Frame path not found {temp_frame_path}, skipping."\n            )\n            if progress:\n                progress.update(1)\n            continue\n\n        temp_frame = imread_unicode(temp_frame_path)\n        if temp_frame is None:\n            print(\n                f"{NAME}: Warning: Failed to read frame {temp_frame_path}, skipping."\n            )\n            if progress:\n                progress.update(1)\n            continue\n\n        result_frame = process_frame(None, temp_frame)\n        imwrite_unicode(temp_frame_path, result_frame)\n        if progress:\n            progress.update(1)\n\n\ndef process_image(\n    source_path: str | None, target_path: str, output_path: str\n) -> None:\n    """Processes a single image file."""\n    target_frame = imread_unicode(target_path)\n    if target_frame is None:\n        print(f"{NAME}: Error: Failed to read target image {target_path}")\n        return\n    result_frame = process_frame(None, target_frame)\n    imwrite_unicode(output_path, result_frame)\n    print(f"{NAME}: Enhanced image saved to {output_path}")\n\n\ndef process_video(\n    source_path: str | None, temp_frame_paths: List[str]\n) -> None:\n    """Processes video frames using the frame processor core."""\n    modules.processors.frame.core.process_video(\n        source_path, temp_frame_paths, process_frames\n    )\n',
    'modules/processors/frame/face_enhancer_gpen256.py': '"""GPEN-BFR-256 face enhancer — ONNX-based face restoration at 256x256."""\n\nfrom typing import Any, List\nimport os\nimport threading\n\nimport modules.globals\nimport modules.processors.frame.core\nfrom modules import imread_unicode, imwrite_unicode\nfrom modules.core import update_status\nfrom modules.face_analyser import get_one_face\nfrom modules.typing import Frame, Face\nfrom modules.utilities import (\n    is_image,\n    is_video,\n)\nfrom modules.processors.frame._onnx_enhancer import (\n    create_onnx_session,\n    warmup_session,\n    enhance_face_onnx,\n)\n\nNAME = "DLC.FACE-ENHANCER-GPEN256"\nINPUT_SIZE = 256\nMODEL_URL = "https://github.com/harisreedhar/Face-Upscalers-ONNX/releases/download/GPEN-BFR/GPEN-BFR-256.onnx"\nMODEL_FILE = "GPEN-BFR-256.onnx"\n\nENHANCER = None\nTHREAD_LOCK = threading.Lock()\n\nabs_dir = os.path.dirname(os.path.abspath(__file__))\nmodels_dir = os.path.join(\n    os.path.dirname(os.path.dirname(os.path.dirname(abs_dir))), "models"\n)\n\n\ndef pre_check() -> bool:\n    model_path = os.path.join(models_dir, MODEL_FILE)\n    if not os.path.exists(model_path):\n        update_status(f"Downloading {MODEL_FILE}...", NAME)\n        from modules.utilities import conditional_download\n        conditional_download(models_dir, [MODEL_URL])\n    return True\n\n\ndef pre_start() -> bool:\n    if not is_image(modules.globals.target_path) and not is_video(modules.globals.target_path):\n        update_status("Select an image or video for target path.", NAME)\n        return False\n    return True\n\n\ndef get_enhancer() -> Any:\n    global ENHANCER\n    with THREAD_LOCK:\n        if ENHANCER is None:\n            model_path = os.path.join(models_dir, MODEL_FILE)\n            if not os.path.exists(model_path):\n                from modules.utilities import conditional_download\n                conditional_download(models_dir, [MODEL_URL])\n            if not os.path.exists(model_path):\n                raise FileNotFoundError(f"Model file not found: {model_path}")\n            print(f"{NAME}: Loading ONNX model from {model_path}")\n            ENHANCER = create_onnx_session(model_path)\n            warmup_session(ENHANCER)\n            print(f"{NAME}: Model loaded successfully.")\n    return ENHANCER\n\n\ndef enhance_face(temp_frame: Frame, face: Face) -> Frame:\n    try:\n        session = get_enhancer()\n    except Exception as e:\n        print(f"{NAME}: {e}")\n        return temp_frame\n    try:\n        return enhance_face_onnx(temp_frame, face, session, INPUT_SIZE)\n    except Exception as e:\n        print(f"{NAME}: Error during face enhancement: {e}")\n        return temp_frame\n\n\ndef process_frame(source_face: Face | None, temp_frame: Frame, detected_faces=None) -> Frame:\n    if detected_faces:\n        target_face = detected_faces[0]\n    else:\n        target_face = get_one_face(temp_frame)\n    if target_face is None:\n        return temp_frame\n    return enhance_face(temp_frame, target_face)\n\n\ndef process_frame_v2(temp_frame: Frame) -> Frame:\n    target_face = get_one_face(temp_frame)\n    if target_face:\n        temp_frame = enhance_face(temp_frame, target_face)\n    return temp_frame\n\n\ndef process_frames(\n    source_path: str | None, temp_frame_paths: List[str], progress: Any = None\n) -> None:\n    for temp_frame_path in temp_frame_paths:\n        temp_frame = imread_unicode(temp_frame_path)\n        if temp_frame is None:\n            if progress:\n                progress.update(1)\n            continue\n        result = process_frame(None, temp_frame)\n        imwrite_unicode(temp_frame_path, result)\n        if progress:\n            progress.update(1)\n\n\ndef process_image(source_path: str | None, target_path: str, output_path: str) -> None:\n    target_frame = imread_unicode(target_path)\n    if target_frame is None:\n        print(f"{NAME}: Error: Failed to read target image {target_path}")\n        return\n    result_frame = process_frame(None, target_frame)\n    imwrite_unicode(output_path, result_frame)\n    print(f"{NAME}: Enhanced image saved to {output_path}")\n\n\ndef process_video(source_path: str | None, temp_frame_paths: List[str]) -> None:\n    modules.processors.frame.core.process_video(source_path, temp_frame_paths, process_frames)\n',
    'modules/processors/frame/face_enhancer_gpen512.py': '"""GPEN-BFR-512 face enhancer — ONNX-based face restoration at 512x512."""\n\nfrom typing import Any, List\nimport os\nimport threading\n\nimport modules.globals\nimport modules.processors.frame.core\nfrom modules import imread_unicode, imwrite_unicode\nfrom modules.core import update_status\nfrom modules.face_analyser import get_one_face\nfrom modules.typing import Frame, Face\nfrom modules.utilities import (\n    is_image,\n    is_video,\n)\nfrom modules.processors.frame._onnx_enhancer import (\n    create_onnx_session,\n    warmup_session,\n    enhance_face_onnx,\n)\n\nNAME = "DLC.FACE-ENHANCER-GPEN512"\nINPUT_SIZE = 512\nMODEL_URL = "https://github.com/harisreedhar/Face-Upscalers-ONNX/releases/download/GPEN-BFR/GPEN-BFR-512.onnx"\nMODEL_FILE = "GPEN-BFR-512.onnx"\n\nENHANCER = None\nTHREAD_LOCK = threading.Lock()\n\nabs_dir = os.path.dirname(os.path.abspath(__file__))\nmodels_dir = os.path.join(\n    os.path.dirname(os.path.dirname(os.path.dirname(abs_dir))), "models"\n)\n\n\ndef pre_check() -> bool:\n    model_path = os.path.join(models_dir, MODEL_FILE)\n    if not os.path.exists(model_path):\n        update_status(f"Downloading {MODEL_FILE}...", NAME)\n        from modules.utilities import conditional_download\n        conditional_download(models_dir, [MODEL_URL])\n    return True\n\n\ndef pre_start() -> bool:\n    if not is_image(modules.globals.target_path) and not is_video(modules.globals.target_path):\n        update_status("Select an image or video for target path.", NAME)\n        return False\n    return True\n\n\ndef get_enhancer() -> Any:\n    global ENHANCER\n    with THREAD_LOCK:\n        if ENHANCER is None:\n            model_path = os.path.join(models_dir, MODEL_FILE)\n            if not os.path.exists(model_path):\n                from modules.utilities import conditional_download\n                conditional_download(models_dir, [MODEL_URL])\n            if not os.path.exists(model_path):\n                raise FileNotFoundError(f"Model file not found: {model_path}")\n            print(f"{NAME}: Loading ONNX model from {model_path}")\n            ENHANCER = create_onnx_session(model_path)\n            warmup_session(ENHANCER)\n            print(f"{NAME}: Model loaded successfully.")\n    return ENHANCER\n\n\ndef enhance_face(temp_frame: Frame, face: Face) -> Frame:\n    try:\n        session = get_enhancer()\n    except Exception as e:\n        print(f"{NAME}: {e}")\n        return temp_frame\n    try:\n        return enhance_face_onnx(temp_frame, face, session, INPUT_SIZE)\n    except Exception as e:\n        print(f"{NAME}: Error during face enhancement: {e}")\n        return temp_frame\n\n\ndef process_frame(source_face: Face | None, temp_frame: Frame, detected_faces=None) -> Frame:\n    if detected_faces:\n        target_face = detected_faces[0]\n    else:\n        target_face = get_one_face(temp_frame)\n    if target_face is None:\n        return temp_frame\n    return enhance_face(temp_frame, target_face)\n\n\ndef process_frame_v2(temp_frame: Frame) -> Frame:\n    target_face = get_one_face(temp_frame)\n    if target_face:\n        temp_frame = enhance_face(temp_frame, target_face)\n    return temp_frame\n\n\ndef process_frames(\n    source_path: str | None, temp_frame_paths: List[str], progress: Any = None\n) -> None:\n    for temp_frame_path in temp_frame_paths:\n        temp_frame = imread_unicode(temp_frame_path)\n        if temp_frame is None:\n            if progress:\n                progress.update(1)\n            continue\n        result = process_frame(None, temp_frame)\n        imwrite_unicode(temp_frame_path, result)\n        if progress:\n            progress.update(1)\n\n\ndef process_image(source_path: str | None, target_path: str, output_path: str) -> None:\n    target_frame = imread_unicode(target_path)\n    if target_frame is None:\n        print(f"{NAME}: Error: Failed to read target image {target_path}")\n        return\n    result_frame = process_frame(None, target_frame)\n    imwrite_unicode(output_path, result_frame)\n    print(f"{NAME}: Enhanced image saved to {output_path}")\n\n\ndef process_video(source_path: str | None, temp_frame_paths: List[str]) -> None:\n    modules.processors.frame.core.process_video(source_path, temp_frame_paths, process_frames)\n',
    'modules/processors/frame/face_masking.py': 'import cv2\nimport numpy as np\nfrom modules.typing import Face, Frame\nimport modules.globals\nfrom modules.gpu_processing import gpu_gaussian_blur, gpu_resize\n\ndef apply_color_transfer(source, target):\n    """\n    Apply color transfer from target to source image using LAB color space.\n    Uses float32 throughout for performance (sufficient precision for 8-bit images).\n    """\n    # Convert to float32 [0,1] range for proper LAB conversion\n    source_f32 = source.astype(np.float32) / 255.0\n    target_f32 = target.astype(np.float32) / 255.0\n\n    source_lab = cv2.cvtColor(source_f32, cv2.COLOR_BGR2LAB)\n    target_lab = cv2.cvtColor(target_f32, cv2.COLOR_BGR2LAB)\n\n    source_mean, source_std = cv2.meanStdDev(source_lab)\n    target_mean, target_std = cv2.meanStdDev(target_lab)\n\n    # Reshape mean and std to be broadcastable (already float64 from meanStdDev, cast to f32)\n    source_mean = source_mean.reshape(1, 1, 3).astype(np.float32)\n    source_std = np.maximum(source_std.reshape(1, 1, 3), 1e-6).astype(np.float32)\n    target_mean = target_mean.reshape(1, 1, 3).astype(np.float32)\n    target_std = target_std.reshape(1, 1, 3).astype(np.float32)\n\n    # Perform the color transfer in LAB space\n    result_lab = (source_lab - source_mean) * (target_std / source_std) + target_mean\n\n    # Convert back to BGR and uint8\n    result_bgr = cv2.cvtColor(result_lab, cv2.COLOR_LAB2BGR)\n    return np.clip(result_bgr * 255.0, 0, 255).astype(np.uint8)\n\ndef create_face_mask(face: Face, frame: Frame) -> np.ndarray:\n    mask = np.zeros(frame.shape[:2], dtype=np.uint8)\n    landmarks = face.landmark_2d_106\n    if landmarks is not None:\n        # Convert landmarks to int32\n        landmarks = landmarks.astype(np.int32)\n\n        # Extract facial features\n        right_side_face = landmarks[0:16]\n        left_side_face = landmarks[17:32]\n        right_eye = landmarks[33:42]\n        right_eye_brow = landmarks[43:51]\n        left_eye = landmarks[87:96]\n        left_eye_brow = landmarks[97:105]\n\n        # Calculate padding\n        padding = int(\n            np.linalg.norm(right_side_face[0] - left_side_face[-1]) * 0.05\n        )  # 5% of face width\n\n        # Create a slightly larger convex hull for padding\n        face_outline = landmarks[0:33]\n        hull = cv2.convexHull(face_outline)\n        # Vectorized hull padding — expand each point outward from center\n        center = np.mean(face_outline, axis=0, dtype=np.float32)\n        hull_pts = hull.reshape(-1, 2).astype(np.float32)\n        directions = hull_pts - center\n        norms = np.linalg.norm(directions, axis=1, keepdims=True)\n        norms = np.maximum(norms, 1e-6)  # avoid division by zero\n        directions /= norms\n        hull_padded = (hull_pts + directions * padding).astype(np.int32)\n\n        # Fill the padded convex hull\n        cv2.fillConvexPoly(mask, hull_padded, 255)\n\n        # Smooth the mask edges (GPU-accelerated when available)\n        mask = gpu_gaussian_blur(mask, (5, 5), 3)\n\n    return mask\n\ndef create_lower_mouth_mask(\n    face: Face, frame: Frame\n) -> (np.ndarray, np.ndarray, tuple, np.ndarray):\n    mask = np.zeros(frame.shape[:2], dtype=np.uint8)\n    mouth_cutout = None\n    lower_lip_polygon = None\n    mouth_box = (0,0,0,0)\n\n    landmarks = face.landmark_2d_106\n    if landmarks is not None:\n        # Use outer mouth landmarks (52-71) to capture the full mouth area\n        lower_lip_order = list(range(52, 72))\n        \n        if max(lower_lip_order) >= landmarks.shape[0]:\n            return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\n        lower_lip_landmarks = landmarks[lower_lip_order].astype(np.float32)\n\n        # Calculate the center of the landmarks\n        center = np.mean(lower_lip_landmarks, axis=0)\n\n        # Expand the landmarks outward using the mouth_mask_size\n        mouth_mask_size = getattr(modules.globals, "mouth_mask_size", 0.0) # 0-100 slider\n        expansion_factor = 1 + (mouth_mask_size / 100.0) * 2.5\n\n        # Expand with extra downward bias toward chin\n        offsets = lower_lip_landmarks - center\n        chin_bias = 1 + (mouth_mask_size / 100.0) * 1.5\n        scale_y = np.where(offsets[:, 1] > 0, expansion_factor * chin_bias, expansion_factor)\n        expanded_landmarks = lower_lip_landmarks.copy()\n        expanded_landmarks[:, 0] = center[0] + offsets[:, 0] * expansion_factor\n        expanded_landmarks[:, 1] = center[1] + offsets[:, 1] * scale_y\n\n        # Convert back to integer coordinates\n        expanded_landmarks = expanded_landmarks.astype(np.int32)\n\n        # Calculate bounding box for the expanded lower mouth\n        min_x, min_y = np.min(expanded_landmarks, axis=0)\n        max_x, max_y = np.max(expanded_landmarks, axis=0)\n\n        # Add some padding to the bounding box\n        padding = int((max_x - min_x) * 0.1)  # 10% padding\n        min_x = max(0, min_x - padding)\n        min_y = max(0, min_y - padding)\n        max_x = min(frame.shape[1], max_x + padding)\n        max_y = min(frame.shape[0], max_y + padding)\n\n        # Ensure the bounding box dimensions are valid\n        if max_x <= min_x or max_y <= min_y:\n            if (max_x - min_x) <= 1:\n                max_x = min_x + 1\n            if (max_y - min_y) <= 1:\n                max_y = min_y + 1\n\n        # Create the mask\n        mask_roi = np.zeros((max_y - min_y, max_x - min_x), dtype=np.uint8)\n        # Shift polygon coordinates relative to the ROI\'s top-left corner\n        polygon_relative_to_roi = expanded_landmarks - [min_x, min_y]\n        cv2.fillPoly(mask_roi, [polygon_relative_to_roi], 255)\n\n        # Apply Gaussian blur to soften the mask edges (GPU-accelerated when available)\n        mask_roi = gpu_gaussian_blur(mask_roi, (15, 15), 5)\n\n        # Place the mask ROI in the full-sized mask\n        mask[min_y:max_y, min_x:max_x] = mask_roi\n\n        # Extract the masked area from the frame\n        mouth_cutout = frame[min_y:max_y, min_x:max_x].copy()\n\n        # Return the expanded lower lip polygon in original frame coordinates\n        lower_lip_polygon = expanded_landmarks\n        mouth_box = (min_x, min_y, max_x, max_y)\n\n    return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\ndef create_eyes_mask(face: Face, frame: Frame) -> (np.ndarray, np.ndarray, tuple, np.ndarray):\n    mask = np.zeros(frame.shape[:2], dtype=np.uint8)\n    eyes_cutout = None\n    landmarks = face.landmark_2d_106\n    if landmarks is not None:\n        # Left eye landmarks (87-96) and right eye landmarks (33-42)\n        left_eye = landmarks[87:96]\n        right_eye = landmarks[33:42]\n        \n        # Calculate centers and dimensions for each eye\n        left_eye_center = np.mean(left_eye, axis=0).astype(np.int32)\n        right_eye_center = np.mean(right_eye, axis=0).astype(np.int32)\n        \n        # Calculate eye dimensions with size adjustment\n        def get_eye_dimensions(eye_points):\n            x_coords = eye_points[:, 0]\n            y_coords = eye_points[:, 1]\n            width = int((np.max(x_coords) - np.min(x_coords)) * (1 + modules.globals.mask_down_size * modules.globals.eyes_mask_size))\n            height = int((np.max(y_coords) - np.min(y_coords)) * (1 + modules.globals.mask_down_size * modules.globals.eyes_mask_size))\n            return width, height\n        \n        left_width, left_height = get_eye_dimensions(left_eye)\n        right_width, right_height = get_eye_dimensions(right_eye)\n        \n        # Add extra padding\n        padding = int(max(left_width, right_width) * 0.2)\n        \n        # Calculate bounding box for both eyes\n        min_x = min(left_eye_center[0] - left_width//2, right_eye_center[0] - right_width//2) - padding\n        max_x = max(left_eye_center[0] + left_width//2, right_eye_center[0] + right_width//2) + padding\n        min_y = min(left_eye_center[1] - left_height//2, right_eye_center[1] - right_height//2) - padding\n        max_y = max(left_eye_center[1] + left_height//2, right_eye_center[1] + right_height//2) + padding\n        \n        # Ensure coordinates are within frame bounds\n        min_x = max(0, min_x)\n        min_y = max(0, min_y)\n        max_x = min(frame.shape[1], max_x)\n        max_y = min(frame.shape[0], max_y)\n        \n        # Create mask for the eyes region\n        mask_roi = np.zeros((max_y - min_y, max_x - min_x), dtype=np.uint8)\n        \n        # Draw ellipses for both eyes\n        left_center = (left_eye_center[0] - min_x, left_eye_center[1] - min_y)\n        right_center = (right_eye_center[0] - min_x, right_eye_center[1] - min_y)\n        \n        # Calculate axes lengths (half of width and height)\n        left_axes = (left_width//2, left_height//2)\n        right_axes = (right_width//2, right_height//2)\n        \n        # Draw filled ellipses\n        cv2.ellipse(mask_roi, left_center, left_axes, 0, 0, 360, 255, -1)\n        cv2.ellipse(mask_roi, right_center, right_axes, 0, 0, 360, 255, -1)\n        \n        # Apply Gaussian blur to soften mask edges (GPU-accelerated when available)\n        mask_roi = gpu_gaussian_blur(mask_roi, (15, 15), 5)\n        \n        # Place the mask ROI in the full-sized mask\n        mask[min_y:max_y, min_x:max_x] = mask_roi\n        \n        # Extract the masked area from the frame\n        eyes_cutout = frame[min_y:max_y, min_x:max_x].copy()\n        \n        # Create polygon points for visualization\n        def create_ellipse_points(center, axes):\n            t = np.linspace(0, 2*np.pi, 32)\n            x = center[0] + axes[0] * np.cos(t)\n            y = center[1] + axes[1] * np.sin(t)\n            return np.column_stack((x, y)).astype(np.int32)\n        \n        # Generate points for both ellipses\n        left_points = create_ellipse_points((left_eye_center[0], left_eye_center[1]), (left_width//2, left_height//2))\n        right_points = create_ellipse_points((right_eye_center[0], right_eye_center[1]), (right_width//2, right_height//2))\n        \n        # Combine points for both eyes\n        eyes_polygon = np.vstack([left_points, right_points])\n        \n    return mask, eyes_cutout, (min_x, min_y, max_x, max_y), eyes_polygon\n\ndef create_curved_eyebrow(points):\n    if len(points) >= 5:\n        # Sort points by x-coordinate\n        sorted_idx = np.argsort(points[:, 0])\n        sorted_points = points[sorted_idx]\n        \n        # Calculate dimensions\n        x_min, y_min = np.min(sorted_points, axis=0)\n        x_max, y_max = np.max(sorted_points, axis=0)\n        width = x_max - x_min\n        height = y_max - y_min\n        \n        # Create more points for smoother curve\n        num_points = 50\n        x = np.linspace(x_min, x_max, num_points)\n        \n        # Fit quadratic curve through points for more natural arch\n        coeffs = np.polyfit(sorted_points[:, 0], sorted_points[:, 1], 2)\n        y = np.polyval(coeffs, x)\n        \n        # Increased offsets to create more separation\n        top_offset = height * 0.5  # Increased from 0.3 to shift up more\n        bottom_offset = height * 0.2  # Increased from 0.1 to shift down more\n        \n        # Create smooth curves\n        top_curve = y - top_offset\n        bottom_curve = y + bottom_offset\n        \n        # Create curved endpoints with more pronounced taper\n        end_points = 5\n        start_x = np.linspace(x[0] - width * 0.15, x[0], end_points)  # Increased taper\n        end_x = np.linspace(x[-1], x[-1] + width * 0.15, end_points)  # Increased taper\n        \n        # Create tapered ends\n        start_curve = np.column_stack((\n            start_x,\n            np.linspace(bottom_curve[0], top_curve[0], end_points)\n        ))\n        end_curve = np.column_stack((\n            end_x,\n            np.linspace(bottom_curve[-1], top_curve[-1], end_points)\n        ))\n        \n        # Combine all points to form a smooth contour\n        contour_points = np.vstack([\n            start_curve,\n            np.column_stack((x, top_curve)),\n            end_curve,\n            np.column_stack((x[::-1], bottom_curve[::-1]))\n        ])\n        \n        # Add slight padding for better coverage\n        center = np.mean(contour_points, axis=0)\n        vectors = contour_points - center\n        padded_points = center + vectors * 1.2  # Increased padding slightly\n        \n        return padded_points\n    return points\n\ndef create_eyebrows_mask(face: Face, frame: Frame) -> (np.ndarray, np.ndarray, tuple, np.ndarray):\n    mask = np.zeros(frame.shape[:2], dtype=np.uint8)\n    eyebrows_cutout = None\n    landmarks = face.landmark_2d_106\n    if landmarks is not None:\n        # Left eyebrow landmarks (97-105) and right eyebrow landmarks (43-51)\n        left_eyebrow = landmarks[97:105].astype(np.float32)\n        right_eyebrow = landmarks[43:51].astype(np.float32)\n        \n        # Calculate centers and dimensions for each eyebrow\n        left_center = np.mean(left_eyebrow, axis=0)\n        right_center = np.mean(right_eyebrow, axis=0)\n        \n        # Calculate bounding box with padding adjusted by size\n        all_points = np.vstack([left_eyebrow, right_eyebrow])\n        padding_factor = modules.globals.eyebrows_mask_size\n        min_x = np.min(all_points[:, 0]) - 25 * padding_factor\n        max_x = np.max(all_points[:, 0]) + 25 * padding_factor\n        min_y = np.min(all_points[:, 1]) - 20 * padding_factor\n        max_y = np.max(all_points[:, 1]) + 15 * padding_factor\n        \n        # Ensure coordinates are within frame bounds\n        min_x = max(0, int(min_x))\n        min_y = max(0, int(min_y))\n        max_x = min(frame.shape[1], int(max_x))\n        max_y = min(frame.shape[0], int(max_y))\n        \n        # Create mask for the eyebrows region\n        mask_roi = np.zeros((max_y - min_y, max_x - min_x), dtype=np.uint8)\n        \n        try:\n            # Convert points to local coordinates\n            left_local = left_eyebrow - [min_x, min_y]\n            right_local = right_eyebrow - [min_x, min_y]\n            \n            def create_curved_eyebrow(points):\n                if len(points) >= 5:\n                    # Sort points by x-coordinate\n                    sorted_idx = np.argsort(points[:, 0])\n                    sorted_points = points[sorted_idx]\n                    \n                    # Calculate dimensions\n                    x_min, y_min = np.min(sorted_points, axis=0)\n                    x_max, y_max = np.max(sorted_points, axis=0)\n                    width = x_max - x_min\n                    height = y_max - y_min\n                    \n                    # Create more points for smoother curve\n                    num_points = 50\n                    x = np.linspace(x_min, x_max, num_points)\n                    \n                    # Fit quadratic curve through points for more natural arch\n                    coeffs = np.polyfit(sorted_points[:, 0], sorted_points[:, 1], 2)\n                    y = np.polyval(coeffs, x)\n                    \n                    # Increased offsets to create more separation\n                    top_offset = height * 0.5  # Increased from 0.3 to shift up more\n                    bottom_offset = height * 0.2  # Increased from 0.1 to shift down more\n                    \n                    # Create smooth curves\n                    top_curve = y - top_offset\n                    bottom_curve = y + bottom_offset\n                    \n                    # Create curved endpoints with more pronounced taper\n                    end_points = 5\n                    start_x = np.linspace(x[0] - width * 0.15, x[0], end_points)  # Increased taper\n                    end_x = np.linspace(x[-1], x[-1] + width * 0.15, end_points)  # Increased taper\n                    \n                    # Create tapered ends\n                    start_curve = np.column_stack((\n                        start_x,\n                        np.linspace(bottom_curve[0], top_curve[0], end_points)\n                    ))\n                    end_curve = np.column_stack((\n                        end_x,\n                        np.linspace(bottom_curve[-1], top_curve[-1], end_points)\n                    ))\n                    \n                    # Combine all points to form a smooth contour\n                    contour_points = np.vstack([\n                        start_curve,\n                        np.column_stack((x, top_curve)),\n                        end_curve,\n                        np.column_stack((x[::-1], bottom_curve[::-1]))\n                    ])\n                    \n                    # Add slight padding for better coverage\n                    center = np.mean(contour_points, axis=0)\n                    vectors = contour_points - center\n                    padded_points = center + vectors * 1.2  # Increased padding slightly\n                    \n                    return padded_points\n                return points\n            \n            # Generate and draw eyebrow shapes\n            left_shape = create_curved_eyebrow(left_local)\n            right_shape = create_curved_eyebrow(right_local)\n            \n            # Apply multi-stage blurring for natural feathering (GPU-accelerated when available)\n            # First, strong Gaussian blur for initial softening\n            mask_roi = gpu_gaussian_blur(mask_roi, (21, 21), 7)\n            \n            # Second, medium blur for transition areas\n            mask_roi = gpu_gaussian_blur(mask_roi, (11, 11), 3)\n            \n            # Finally, light blur for fine details\n            mask_roi = gpu_gaussian_blur(mask_roi, (5, 5), 1)\n            \n            # Normalize mask values\n            mask_roi = cv2.normalize(mask_roi, None, 0, 255, cv2.NORM_MINMAX)\n            \n            # Place the mask ROI in the full-sized mask\n            mask[min_y:max_y, min_x:max_x] = mask_roi\n            \n            # Extract the masked area from the frame\n            eyebrows_cutout = frame[min_y:max_y, min_x:max_x].copy()\n            \n            # Combine points for visualization\n            eyebrows_polygon = np.vstack([\n                left_shape + [min_x, min_y],\n                right_shape + [min_x, min_y]\n            ]).astype(np.int32)\n            \n        except Exception as e:\n            # Fallback to simple polygons if curve fitting fails\n            left_local = left_eyebrow - [min_x, min_y]\n            right_local = right_eyebrow - [min_x, min_y]\n            cv2.fillPoly(mask_roi, [left_local.astype(np.int32)], 255)\n            cv2.fillPoly(mask_roi, [right_local.astype(np.int32)], 255)\n            mask_roi = gpu_gaussian_blur(mask_roi, (21, 21), 7)\n            mask[min_y:max_y, min_x:max_x] = mask_roi\n            eyebrows_cutout = frame[min_y:max_y, min_x:max_x].copy()\n            eyebrows_polygon = np.vstack([left_eyebrow, right_eyebrow]).astype(np.int32)\n        \n    return mask, eyebrows_cutout, (min_x, min_y, max_x, max_y), eyebrows_polygon\n\ndef apply_mask_area(\n    frame: np.ndarray,\n    cutout: np.ndarray,\n    box: tuple,\n    face_mask: np.ndarray,\n    polygon: np.ndarray,\n) -> np.ndarray:\n    min_x, min_y, max_x, max_y = box\n    box_width = max_x - min_x\n    box_height = max_y - min_y\n\n    if (\n        cutout is None\n        or box_width is None\n        or box_height is None\n        or face_mask is None\n        or polygon is None\n    ):\n        return frame\n\n    try:\n        resized_cutout = gpu_resize(cutout, (box_width, box_height))\n        roi = frame[min_y:max_y, min_x:max_x]\n\n        if roi.shape != resized_cutout.shape:\n            resized_cutout = gpu_resize(\n                resized_cutout, (roi.shape[1], roi.shape[0])\n            )\n\n        color_corrected_area = apply_color_transfer(resized_cutout, roi)\n\n        # Create mask for the area\n        polygon_mask = np.zeros(roi.shape[:2], dtype=np.uint8)\n        \n        # Split points for left and right parts if needed\n        if len(polygon) > 50:  # Arbitrary threshold to detect if we have multiple parts\n            mid_point = len(polygon) // 2\n            left_points = polygon[:mid_point] - [min_x, min_y]\n            right_points = polygon[mid_point:] - [min_x, min_y]\n            cv2.fillPoly(polygon_mask, [left_points], 255)\n            cv2.fillPoly(polygon_mask, [right_points], 255)\n        else:\n            adjusted_polygon = polygon - [min_x, min_y]\n            cv2.fillPoly(polygon_mask, [adjusted_polygon], 255)\n\n        # Apply strong initial feathering (GPU-accelerated when available)\n        polygon_mask = gpu_gaussian_blur(polygon_mask, (21, 21), 7)\n\n        # Apply additional feathering\n        feather_amount = min(\n            30,\n            box_width // modules.globals.mask_feather_ratio,\n            box_height // modules.globals.mask_feather_ratio,\n        )\n        feathered_mask = cv2.GaussianBlur(\n            polygon_mask.astype(np.float32), (0, 0), feather_amount\n        )\n        max_val = feathered_mask.max()\n        if max_val > 1e-6:\n            feathered_mask *= np.float32(1.0 / max_val)\n\n        # Apply additional smoothing to the mask edges\n        feathered_mask = cv2.GaussianBlur(feathered_mask, (5, 5), 1)\n\n        face_mask_roi = face_mask[min_y:max_y, min_x:max_x]\n        combined_mask = feathered_mask * (face_mask_roi.astype(np.float32) * np.float32(1.0 / 255.0))\n\n        combined_mask_3ch = combined_mask[:, :, np.newaxis]\n        inv_mask = np.float32(1.0) - combined_mask_3ch\n        blended = (\n            color_corrected_area * combined_mask_3ch + roi * inv_mask\n        ).astype(np.uint8)\n\n        # Apply face mask to blended result\n        face_mask_f32 = face_mask_roi[:, :, np.newaxis].astype(np.float32) * np.float32(1.0 / 255.0)\n        face_mask_3channel = np.broadcast_to(face_mask_f32, blended.shape)\n        final_blend = blended * face_mask_3channel + roi * (np.float32(1.0) - face_mask_3channel)\n\n        frame[min_y:max_y, min_x:max_x] = final_blend.astype(np.uint8)\n    except Exception as e:\n        pass\n\n    return frame\n\ndef draw_mask_visualization(\n    frame: Frame,\n    mask_data: tuple,\n    label: str,\n    draw_method: str = "polygon"\n) -> Frame:\n    mask, cutout, (min_x, min_y, max_x, max_y), polygon = mask_data\n\n    vis_frame = frame.copy()\n\n    # Ensure coordinates are within frame bounds\n    height, width = vis_frame.shape[:2]\n    min_x, min_y = max(0, min_x), max(0, min_y)\n    max_x, max_y = min(width, max_x), min(height, max_y)\n\n    if draw_method == "ellipse" and len(polygon) > 50:  # For eyes\n        # Split points for left and right parts\n        mid_point = len(polygon) // 2\n        left_points = polygon[:mid_point]\n        right_points = polygon[mid_point:]\n        \n        try:\n            # Fit ellipses to points - need at least 5 points\n            if len(left_points) >= 5 and len(right_points) >= 5:\n                # Convert points to the correct format for ellipse fitting\n                left_points = left_points.astype(np.float32)\n                right_points = right_points.astype(np.float32)\n                \n                # Fit ellipses\n                left_ellipse = cv2.fitEllipse(left_points)\n                right_ellipse = cv2.fitEllipse(right_points)\n                \n                # Draw the ellipses\n                cv2.ellipse(vis_frame, left_ellipse, (0, 255, 0), 2)\n                cv2.ellipse(vis_frame, right_ellipse, (0, 255, 0), 2)\n        except Exception as e:\n            # If ellipse fitting fails, draw simple rectangles as fallback\n            left_rect = cv2.boundingRect(left_points)\n            right_rect = cv2.boundingRect(right_points)\n            cv2.rectangle(vis_frame, \n                        (left_rect[0], left_rect[1]), \n                        (left_rect[0] + left_rect[2], left_rect[1] + left_rect[3]), \n                        (0, 255, 0), 2)\n            cv2.rectangle(vis_frame,\n                        (right_rect[0], right_rect[1]),\n                        (right_rect[0] + right_rect[2], right_rect[1] + right_rect[3]),\n                        (0, 255, 0), 2)\n    else:  # For mouth and eyebrows\n        # Draw the polygon\n        if len(polygon) > 50:  # If we have multiple parts\n            mid_point = len(polygon) // 2\n            left_points = polygon[:mid_point]\n            right_points = polygon[mid_point:]\n            cv2.polylines(vis_frame, [left_points], True, (0, 255, 0), 2, cv2.LINE_AA)\n            cv2.polylines(vis_frame, [right_points], True, (0, 255, 0), 2, cv2.LINE_AA)\n        else:\n            cv2.polylines(vis_frame, [polygon], True, (0, 255, 0), 2, cv2.LINE_AA)\n\n    # Add label\n    cv2.putText(\n        vis_frame,\n        label,\n        (min_x, min_y - 10),\n        cv2.FONT_HERSHEY_SIMPLEX,\n        0.5,\n        (255, 255, 255),\n        1,\n    )\n\n    return vis_frame',
    'modules/processors/frame/face_swapper.py': 'from typing import Any, List, Optional, Tuple\nimport cv2\nimport insightface\nimport logging\nimport threading\nimport numpy as np\nimport platform\nimport modules.globals\nimport modules.processors.frame.core\nfrom modules import imread_unicode, imwrite_unicode\nfrom modules.core import update_status\nfrom modules.face_analyser import get_one_face, get_many_faces, default_source_face\nfrom modules.typing import Face, Frame\nfrom modules.utilities import (\n    conditional_download,\n    is_image,\n    is_video,\n)\nfrom modules.cluster_analysis import find_closest_centroid\nfrom modules.gpu_processing import gpu_gaussian_blur, gpu_sharpen, gpu_add_weighted, gpu_resize\nimport os\nfrom collections import deque\nimport time\n\nFACE_SWAPPER = None\nTHREAD_LOCK = threading.Lock()\nNAME = "DLC.FACE-SWAPPER"\n\n# --- START: Added for Interpolation ---\nPREVIOUS_FRAME_RESULT = None # Stores the final processed frame from the previous step\n# --- END: Added for Interpolation ---\n\n# --- Poisson blend (ported from deep-live-cam-gumroad-edition) ---\n# Root-cause fix for the "wobble": the blend mask is NOT built from the\n# independently-detected 106-pt landmarks (they jitter sub-pixel every frame\n# and seamlessClone is hyper-sensitive to its mask boundary). Instead it is\n# derived from the swap\'s OWN affine transform (M) + the swapped pixels\n# (bgr_fake), so the mask is locked exactly to where the swapped face was\n# placed — no independent jitter source, no EMA, no lag. The mask is cached\n# when the face is nearly still so an identical array is reused (zero wobble).\n_ELLIPTICAL_MASK_CACHE: dict = {}\n_poisson_cached_mask: Optional[np.ndarray] = None\n_poisson_cached_key: Optional[tuple] = None\n\n\ndef _create_elliptical_mask(size: Tuple[int, int]) -> np.ndarray:\n    """Fixed, heavily-blurred elliptical mask in aligned-face space.\n\n    Geometry-based (not content-adaptive) and cached by size — identical\n    every frame for the same model input size, so it contributes no jitter.\n    """\n    global _ELLIPTICAL_MASK_CACHE\n    if size in _ELLIPTICAL_MASK_CACHE:\n        return _ELLIPTICAL_MASK_CACHE[size]\n    h, w = size\n    center = (w // 2, h // 2)\n    axes = (int(w * 0.44), int(h * 0.44))\n    mask = np.zeros((h, w), dtype=np.float32)\n    cv2.ellipse(mask, center, axes, 0, 0, 360, 1, -1)\n    if h * w < 65536:\n        mask = cv2.GaussianBlur(mask, (31, 31), 12)\n    else:\n        mask = gpu_gaussian_blur(mask, (31, 31), 12)\n    _ELLIPTICAL_MASK_CACHE[size] = mask\n    return mask\n\n\ndef _apply_poisson_blend(swapped_frame: Frame, original_frame: Frame,\n                         target_face: Face, affine_matrix: np.ndarray = None,\n                         bgr_fake: np.ndarray = None) -> Frame:\n    """Poisson-blend the swapped face onto the original frame.\n\n    Preferred path derives the blend mask from the swap\'s inverse affine so\n    it tracks the swapped face exactly per-frame (no landmark jitter, no\n    smoothing). Falls back to a cached bbox-ellipse if the affine is absent.\n    Writes only the blended ellipse back so other faces are preserved.\n    """\n    global _poisson_cached_mask, _poisson_cached_key\n    try:\n        # ---- Preferred: blend ONLY the genuinely-swapped region ----\n        # Use the exact paste-back mask (warped elliptical mask), eroded so\n        # the Poisson seam sits on solidly-swapped pixels only.\n        if affine_matrix is not None and bgr_fake is not None:\n            try:\n                h, w = swapped_frame.shape[:2]\n                fh, fw = bgr_fake.shape[:2]\n                inv = cv2.invertAffineTransform(affine_matrix)\n                corners = np.array([[0, 0, 1], [fw, 0, 1], [fw, fh, 1], [0, fh, 1]],\n                                   dtype=np.float32)\n                t = corners @ inv.T\n                px1 = max(0, int(np.floor(t[:, 0].min())))\n                py1 = max(0, int(np.floor(t[:, 1].min())))\n                px2 = min(w, int(np.ceil(t[:, 0].max())))\n                py2 = min(h, int(np.ceil(t[:, 1].max())))\n                rw, rh = px2 - px1, py2 - py1\n                if rw > 8 and rh > 8:\n                    roi_aff = inv.copy()\n                    roi_aff[0, 2] -= px1\n                    roi_aff[1, 2] -= py1\n                    fm = _create_elliptical_mask((fh, fw))\n                    mroi = cv2.warpAffine(fm, roi_aff, (rw, rh),\n                                          flags=cv2.INTER_LINEAR,\n                                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n                    bin_roi = np.where(mroi > 0.5, np.uint8(255), np.uint8(0))\n                    k = max(3, (min(rw, rh) // 20) | 1)\n                    bin_roi = cv2.erode(bin_roi,\n                                        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))\n                    bx, by, bw, bh = cv2.boundingRect(bin_roi)\n                    if bw > 0 and bh > 0:\n                        mx1, my1 = px1 + bx, py1 + by\n                        mx2, my2 = mx1 + bw - 1, my1 + bh - 1\n                        # seamlessClone needs the cloned region off the border\n                        if mx1 > 0 and my1 > 0 and mx2 < w - 1 and my2 < h - 1:\n                            mask = np.zeros((h, w), dtype=np.uint8)\n                            mask[py1:py2, px1:px2] = bin_roi\n                            center = (mx1 + bw // 2, my1 + bh // 2)\n                            blended = cv2.seamlessClone(swapped_frame, original_frame,\n                                                        mask, center, cv2.NORMAL_CLONE)\n                            np.copyto(swapped_frame[my1:my2 + 1, mx1:mx2 + 1],\n                                      blended[my1:my2 + 1, mx1:mx2 + 1],\n                                      where=mask[my1:my2 + 1, mx1:mx2 + 1, None].astype(bool))\n                            return swapped_frame\n            except Exception:\n                pass  # fall through to the robust bbox-ellipse path below\n        # ---- Fallback: bbox-ellipse (defensive, cached when still) ----\n        if not hasattr(target_face, \'bbox\') or target_face.bbox is None:\n            return swapped_frame\n        x1, y1, x2, y2 = target_face.bbox.astype(int)\n        h, w = swapped_frame.shape[:2]\n        x1, y1 = (max(0, x1), max(0, y1))\n        x2, y2 = (min(w, x2), min(h, y2))\n        if x2 <= x1 or y2 <= y1 or x2 - x1 <= 10 or (y2 - y1 <= 10):\n            return swapped_frame\n        padding = int(min(x2 - x1, y2 - y1) * 0.1)\n        x1_p = max(0, x1 - padding)\n        y1_p = max(0, y1 - padding)\n        x2_p = min(w, x2 + padding)\n        y2_p = min(h, y2 + padding)\n        center_x = int(round((x1 + x2) / 2.0))\n        center_y = int(round((y1 + y2) / 2.0))\n        radius_x = max(1, int(round((x2_p - x1_p) / 2.0)))\n        radius_y = max(1, int(round((y2_p - y1_p) / 2.0)))\n        if not (0 <= center_x < w and 0 <= center_y < h):\n            return swapped_frame\n        center = (center_x, center_y)\n        if center_x - radius_x < 0 or center_x + radius_x >= w or center_y - radius_y < 0 or (center_y + radius_y >= h):\n            return swapped_frame\n        # Reuse cached mask when center/radius unchanged frame-to-frame\n        # (face nearly still) — saves the np.zeros + cv2.ellipse, and the\n        # identical array means literally zero wobble while still.\n        mask_key = (center_x, center_y, radius_x, radius_y, h, w)\n        if _poisson_cached_key == mask_key and _poisson_cached_mask is not None:\n            mask = _poisson_cached_mask\n        else:\n            mask = np.zeros((h, w), dtype=np.uint8)\n            cv2.ellipse(mask, center, (radius_x, radius_y), 0, 0, 360, 255, -1)\n            if np.sum(mask) == 0:\n                return swapped_frame\n            _poisson_cached_mask = mask\n            _poisson_cached_key = mask_key\n        blended = cv2.seamlessClone(swapped_frame, original_frame, mask, center, cv2.NORMAL_CLONE)\n        # Composite ONLY this face\'s ellipse back (ROI-bounded) so previously\n        # blended faces in multi-face mode are preserved.\n        rx0 = max(0, center_x - radius_x)\n        rx1 = min(w, center_x + radius_x + 1)\n        ry0 = max(0, center_y - radius_y)\n        ry1 = min(h, center_y + radius_y + 1)\n        roi_mask = mask[ry0:ry1, rx0:rx1]\n        np.copyto(swapped_frame[ry0:ry1, rx0:rx1],\n                  blended[ry0:ry1, rx0:rx1],\n                  where=roi_mask[:, :, None].astype(bool))\n        return swapped_frame\n    except Exception:\n        return swapped_frame\n\n# --- START: Mac M1-M5 Optimizations ---\nIS_APPLE_SILICON = platform.system() == \'Darwin\' and platform.machine() == \'arm64\'\nFRAME_CACHE = deque(maxlen=3)  # Cache for frame reuse\nFACE_DETECTION_CACHE = {}  # Cache face detections\nLAST_DETECTION_TIME = 0\nDETECTION_INTERVAL = 0.033  # ~30 FPS detection rate for live mode\nFRAME_SKIP_COUNTER = 0\nADAPTIVE_QUALITY = True\n# --- END: Mac M1-M5 Optimizations ---\n\nabs_dir = os.path.dirname(os.path.abspath(__file__))\nmodels_dir = os.path.join(\n    os.path.dirname(os.path.dirname(os.path.dirname(abs_dir))), "models"\n)\n\ndef pre_check() -> bool:\n    # Use models_dir instead of abs_dir to save to the correct location\n    download_directory_path = models_dir\n    \n    # Make sure the models directory exists, catch permission errors if they occur\n    try:\n        os.makedirs(download_directory_path, exist_ok=True)\n    except OSError as e:\n        logging.error(f"Failed to create directory {download_directory_path} due to permission error: {e}")\n        return False\n    \n    # Use the direct download URL from Hugging Face (FP32 model for broad GPU compatibility)\n    conditional_download(\n        download_directory_path,\n        [\n            "https://huggingface.co/hacksider/deep-live-cam/resolve/main/inswapper_128.onnx"\n        ],\n    )\n    return True\n\n\ndef pre_start() -> bool:\n    # Check for either model variant\n    fp16_path = os.path.join(models_dir, "inswapper_128_fp16.onnx")\n    fp32_path = os.path.join(models_dir, "inswapper_128.onnx")\n    if not os.path.exists(fp16_path) and not os.path.exists(fp32_path):\n        update_status(f"Model not found in {models_dir}. Please download inswapper_128.onnx.", NAME)\n        return False\n\n    # Try to get the face swapper to ensure it loads correctly\n    if get_face_swapper() is None:\n        # Error message already printed within get_face_swapper\n        return False\n\n    return True\n\n\ndef get_face_swapper() -> Any:\n    global FACE_SWAPPER\n\n    with THREAD_LOCK:\n        if FACE_SWAPPER is None:\n            # Prefer FP16 on GPUs with Tensor Cores (Turing+) — half the\n            # memory bandwidth, faster inference.  Fall back to FP32 for\n            # older GPUs (e.g. GTX 16xx) where FP16 can produce NaN.\n            fp32_path = os.path.join(models_dir, "inswapper_128.onnx")\n            fp16_path = os.path.join(models_dir, "inswapper_128_fp16.onnx")\n            use_fp16 = _HAS_TORCH_CUDA and os.path.exists(fp16_path)\n            if use_fp16:\n                model_path = fp16_path\n            elif os.path.exists(fp32_path):\n                model_path = fp32_path\n            else:\n                update_status(f"No inswapper model found in {models_dir}.", NAME)\n                return None\n            # On Apple Silicon, rewrite Pad(reflect) → Slice+Concat so\n            # CoreML can run the entire model in a single partition on\n            # the Neural Engine instead of bouncing between CPU and ANE.\n            if IS_APPLE_SILICON:\n                from modules.onnx_optimize import optimize_for_coreml\n                model_path = optimize_for_coreml(model_path)\n\n            update_status(f"Loading face swapper model from: {model_path}", NAME)\n            try:\n                providers_config = []\n                for p in modules.globals.execution_providers:\n                    if p == "CoreMLExecutionProvider" and IS_APPLE_SILICON:\n                        # Enhanced CoreML configuration for M1-M5\n                        providers_config.append((\n                            "CoreMLExecutionProvider",\n                            {\n                                "ModelFormat": "MLProgram",\n                                "MLComputeUnits": "ALL",  # Use Neural Engine + GPU + CPU\n                                "SpecializationStrategy": "FastPrediction",\n                                "AllowLowPrecisionAccumulationOnGPU": 1,\n                                "EnableOnSubgraphs": 1,\n                            }\n                        ))\n                    elif p == "CUDAExecutionProvider":\n                        # Use bare provider — ONNX Runtime defaults are\n                        # fastest on modern GPUs (Blackwell/sm_120).\n                        providers_config.append(p)\n                    else:\n                        providers_config.append(p)\n                FACE_SWAPPER = insightface.model_zoo.get_model(\n                    model_path,\n                    providers=providers_config,\n                )\n                # Set up CUDA graph session for faster inference\n                if _HAS_TORCH_CUDA and any(\n                    p == "CUDAExecutionProvider" or\n                    (isinstance(p, tuple) and p[0] == "CUDAExecutionProvider")\n                    for p in providers_config\n                ):\n                    _init_cuda_graph_session(model_path, FACE_SWAPPER)\n                update_status("Face swapper model loaded successfully.", NAME)\n            except Exception as e:\n                update_status(f"Error loading face swapper model: {e}", NAME)\n                FACE_SWAPPER = None\n                return None\n    return FACE_SWAPPER\n\n\n_HAS_TORCH_CUDA = False\ntry:\n    import torch\n    if torch.cuda.is_available():\n        _HAS_TORCH_CUDA = True\nexcept ImportError:\n    pass\n\n# Cache for paste-back\n_paste_cache = {\n    \'soft_alpha\': None,  # feathered alpha mask in aligned-face space\n    \'alpha_size\': 0,\n}\n\n\ndef _get_soft_alpha(size: int) -> np.ndarray:\n    """Feathered alpha template in aligned-face space, cached.\n\n    The legacy paste-back eroded and Gaussian-blurred the warped mask in\n    output coordinates with kernels scaled to the output face size, which\n    made the per-frame cost quartic in face linear size. Doing the same\n    erode+blur once in aligned space and then warping the *soft* mask\n    per-frame gives a visually equivalent feather at O(crop_area) cost —\n    the feather radius scales naturally with the affine transform.\n    """\n    if _paste_cache[\'alpha_size\'] != size:\n        # Elliptical (not square) template — matches the gumroad edition\'s\n        # _create_elliptical_mask. A full/eroded square leaves the aligned\n        # crop\'s corners near-opaque, so the swapped square\'s straight edges\n        # show as a visible box on the face. An ellipse (axes 0.44*size) zeroes\n        # the corners and the heavy blur feathers smoothly into the original.\n        center = (size // 2, size // 2)\n        axes = (int(size * 0.44), int(size * 0.44))\n        mask = np.zeros((size, size), dtype=np.uint8)\n        cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)\n        mask = cv2.GaussianBlur(mask, (31, 31), 12)\n        _paste_cache[\'soft_alpha\'] = mask  # uint8 [0, 255] — blended via cv2 SIMD ops\n        _paste_cache[\'alpha_size\'] = size\n    return _paste_cache[\'soft_alpha\']\n\n# CUDA graph swap session cache\n_cuda_graph_session = {\n    \'session\': None,\n    \'io_binding\': None,\n    \'ort_input\': None,\n    \'ort_latent\': None,\n    \'recorded\': False,\n}\n# Serializes CUDA-graph replay. The io_binding + ort_input/ort_latent are\n# shared across threads and run_with_iobinding mutates GPU-side buffers;\n# concurrent calls would produce wrong output.\n_cuda_graph_lock = threading.Lock()\n\n\nclass _CudaGraphSessionAdapter:\n    """Drop-in wrapper around an ONNX Runtime session.\n\n    Routes ``.run()`` through CUDA graph replay when a recorded graph is\n    available, and transparently proxies every other attribute to the\n    underlying session so insightface\'s INSwapper sees an unchanged API.\n    """\n\n    def __init__(self, underlying):\n        # Use object.__setattr__ to bypass our own __setattr__.\n        object.__setattr__(self, "_underlying", underlying)\n\n    def run(self, output_names, input_dict, **kwargs):\n        if _cuda_graph_session[\'recorded\']:\n            try:\n                keys = list(input_dict.keys())\n                blob = input_dict[keys[0]]\n                latent = input_dict[keys[1]]\n                return [_cuda_graph_swap_inference(blob, latent)]\n            except Exception:\n                pass\n        return self._underlying.run(output_names, input_dict, **kwargs)\n\n    def __getattr__(self, name):\n        return getattr(self._underlying, name)\n\n    def __setattr__(self, name, value):\n        setattr(self._underlying, name, value)\n\n\ndef _init_cuda_graph_session(model_path: str, swapper):\n    """Create a CUDA-graph-enabled ONNX session for the swap model.\n\n    CUDA graphs record the GPU kernel launch sequence once, then replay it\n    with near-zero CPU overhead on subsequent runs.  Requires static input\n    shapes (inswapper is always 1x3x128x128 + 1x512).\n    """\n    import onnxruntime as ort\n    try:\n        providers = [(\'CUDAExecutionProvider\', {\'enable_cuda_graph\': \'1\'})]\n        sess = ort.InferenceSession(model_path, providers=providers)\n\n        # Pre-allocate GPU buffers with correct shapes\n        inp_shape = (1, 3, swapper.input_size[1], swapper.input_size[0])\n        latent_shape = (1, 512)\n        dummy_inp = np.zeros(inp_shape, dtype=np.float32)\n        dummy_lat = np.zeros(latent_shape, dtype=np.float32)\n\n        ort_input = ort.OrtValue.ortvalue_from_numpy(dummy_inp, \'cuda\', 0)\n        ort_latent = ort.OrtValue.ortvalue_from_numpy(dummy_lat, \'cuda\', 0)\n\n        io = sess.io_binding()\n        io.bind_ortvalue_input(swapper.input_names[0], ort_input)\n        io.bind_ortvalue_input(swapper.input_names[1], ort_latent)\n        io.bind_output(swapper.output_names[0], \'cuda\', 0)\n\n        # First run records the CUDA graph\n        sess.run_with_iobinding(io)\n\n        _cuda_graph_session[\'session\'] = sess\n        _cuda_graph_session[\'io_binding\'] = io\n        _cuda_graph_session[\'ort_input\'] = ort_input\n        _cuda_graph_session[\'ort_latent\'] = ort_latent\n        _cuda_graph_session[\'recorded\'] = True\n\n        # Wrap swapper.session in an adapter instead of rebinding\n        # session.run. insightface\'s INSwapper.get() reads .run via the\n        # session attribute, so either works; the adapter survives any\n        # later attribute reads on the session and keeps the original\n        # session object untouched.\n        if not isinstance(swapper.session, _CudaGraphSessionAdapter):\n            swapper.session = _CudaGraphSessionAdapter(swapper.session)\n\n        import sys\n        print(f"[{NAME}] CUDA graph session initialized (swap model)")\n        sys.stdout.flush()\n    except Exception as e:\n        print(f"[{NAME}] CUDA graph init failed, using standard session: {e}")\n        _cuda_graph_session[\'recorded\'] = False\n\n\ndef _cuda_graph_swap_inference(blob: np.ndarray, latent: np.ndarray) -> np.ndarray:\n    """Run swap model via CUDA graph replay — minimal CPU overhead."""\n    cg = _cuda_graph_session\n    with _cuda_graph_lock:\n        cg[\'ort_input\'].update_inplace(blob)\n        cg[\'ort_latent\'].update_inplace(latent)\n        cg[\'session\'].run_with_iobinding(cg[\'io_binding\'])\n        return cg[\'io_binding\'].get_outputs()[0].numpy()\n\n\ndef _fast_paste_back(target_img: Frame, bgr_fake: np.ndarray, aimg: np.ndarray, M: np.ndarray) -> Frame:\n    """Paste bgr_fake back onto target_img via the inverse affine of M.\n\n    Restricts work to the face bbox in output coordinates and warps a\n    precomputed feathered alpha template per-frame instead of running a\n    size-scaled erode+blur on the warped mask. Cost is O(crop_area) regardless\n    of how much of the frame the face occupies.\n    """\n    h, w = target_img.shape[:2]\n    face_h, face_w = aimg.shape[:2]\n    # inswapper\'s aligned-face space is square (128x128). _get_soft_alpha\n    # caches a single NxN template keyed by N, so fail loudly if that ever\n    # stops being true rather than silently mis-warping the alpha mask.\n    assert face_h == face_w, f"Expected square aligned face, got {face_h}x{face_w}"\n    IM = cv2.invertAffineTransform(M)\n\n    # Bbox in output coords from the affine corners of the aligned-face square.\n    corners = np.array(\n        [[0, 0], [face_w, 0], [face_w, face_h], [0, face_h]], dtype=np.float32\n    )\n    transformed = (IM[:, :2] @ corners.T).T + IM[:, 2]\n    x1 = int(np.floor(transformed[:, 0].min()))\n    x2 = int(np.ceil(transformed[:, 0].max()))\n    y1 = int(np.floor(transformed[:, 1].min()))\n    y2 = int(np.ceil(transformed[:, 1].max()))\n    if x1 >= x2 or y1 >= y2:\n        return target_img\n\n    # Small interpolation margin only — the feather is baked into the template.\n    pad = 2\n    y1p, y2p = max(0, y1 - pad), min(h, y2 + pad + 1)\n    x1p, x2p = max(0, x1 - pad), min(w, x2 + pad + 1)\n\n    IM_crop = IM.copy()\n    IM_crop[0, 2] -= x1p\n    IM_crop[1, 2] -= y1p\n    crop_w, crop_h = x2p - x1p, y2p - y1p\n\n    soft_alpha = _get_soft_alpha(face_h)\n    bgr_fake_crop = cv2.warpAffine(bgr_fake, IM_crop, (crop_w, crop_h), borderMode=cv2.BORDER_REPLICATE)\n    alpha_crop = cv2.warpAffine(soft_alpha, IM_crop, (crop_w, crop_h), borderValue=0)\n\n    target_crop = target_img[y1p:y2p, x1p:x2p]\n\n    if _HAS_TORCH_CUDA:\n        # Scale alpha to [0, 1] on device — cheaper to upload uint8 than float.\n        mask_t = torch.from_numpy(alpha_crop).cuda().float().mul_(1.0 / 255.0).unsqueeze(2)\n        fake_t = torch.from_numpy(bgr_fake_crop).float().cuda()\n        tgt_t = torch.from_numpy(target_crop).float().cuda()\n        blended = (mask_t * fake_t + (1.0 - mask_t) * tgt_t).to(torch.uint8).cpu().numpy()\n        target_img[y1p:y2p, x1p:x2p] = blended\n    else:\n        # Fused uint8 blend via cv2 SIMD — no float32 round-trip.\n        # Measured ~7-8× faster than the old numpy float32 path on a 1000×1000 crop.\n        alpha_3c = cv2.merge([alpha_crop, alpha_crop, alpha_crop])\n        inv_alpha = 255 - alpha_3c\n        a_fake = cv2.multiply(bgr_fake_crop, alpha_3c, scale=1.0 / 255.0)\n        a_tgt = cv2.multiply(target_crop, inv_alpha, scale=1.0 / 255.0)\n        target_img[y1p:y2p, x1p:x2p] = cv2.add(a_fake, a_tgt)\n\n    return target_img\n\n\ndef swap_face(source_face: Face, target_face: Face, temp_frame: Frame) -> Frame:\n    """Optimized face swapping with better memory management and performance."""\n    face_swapper = get_face_swapper()\n    if face_swapper is None:\n        update_status("Face swapper model not loaded or failed to load. Skipping swap.", NAME)\n        return temp_frame\n\n    # Safety check for faces\n    if source_face is None or target_face is None:\n        return temp_frame\n    if not hasattr(source_face, \'normed_embedding\') or source_face.normed_embedding is None:\n        return temp_frame\n\n    # _fast_paste_back writes in-place on the GPU path.  Only copy when\n    # mouth_mask or opacity < 1 need an unmodified original.\n    opacity = getattr(modules.globals, "opacity", 1.0)\n    opacity = max(0.0, min(1.0, opacity))\n    mouth_mask_enabled = getattr(modules.globals, "mouth_mask", False)\n    poisson_blend_enabled = getattr(modules.globals, "poisson_blend", False)\n    color_correction_enabled = getattr(modules.globals, "color_correction", False)\n    # Poisson blend\'s seamlessClone needs the genuine pre-swap frame as its\n    # destination. Without this, original_frame aliases temp_frame, which\n    # _fast_paste_back mutates in place — so seamlessClone would blend the\n    # swapped face onto the already-swapped frame (no visible effect).\n    needs_original = opacity < 1.0 or mouth_mask_enabled or poisson_blend_enabled or color_correction_enabled\n    if needs_original:\n        original_frame = temp_frame.copy()\n    else:\n        original_frame = temp_frame\n\n    if temp_frame.dtype != np.uint8:\n        temp_frame = np.clip(temp_frame, 0, 255).astype(np.uint8)\n\n    try:\n        if not temp_frame.flags[\'C_CONTIGUOUS\']:\n            temp_frame = np.ascontiguousarray(temp_frame)\n\n        # Use paste_back=False and our optimized paste-back\n        if any("DmlExecutionProvider" in p for p in modules.globals.execution_providers):\n            with modules.globals.dml_lock:\n                bgr_fake, M = face_swapper.get(\n                    temp_frame, target_face, source_face, paste_back=False\n                )\n        else:\n            bgr_fake, M = face_swapper.get(\n                temp_frame, target_face, source_face, paste_back=False\n            )\n\n        if bgr_fake is None:\n            return original_frame\n\n        if not isinstance(bgr_fake, np.ndarray):\n            return original_frame\n\n        # Pass a dummy aimg with correct shape — _fast_paste_back only uses aimg.shape\n        # to create the white mask. Avoids redundant norm_crop2 (~0.6ms).\n        _face_size = face_swapper.input_size[0]\n        _aimg_dummy = np.empty((_face_size, _face_size, 3), dtype=np.uint8)\n\n        if color_correction_enabled:\n            target_aligned = cv2.warpAffine(\n                original_frame,\n                M,\n                (_face_size, _face_size),\n                flags=cv2.INTER_LINEAR,\n                borderMode=cv2.BORDER_REFLECT,\n            )\n            bgr_fake = apply_color_transfer(bgr_fake, target_aligned)\n\n        swapped_frame = _fast_paste_back(temp_frame, bgr_fake, _aimg_dummy, M)\n\n    except Exception as e:\n        print(f"Error during face swap: {e}")\n        return original_frame\n\n    # --- Post-swap Processing (Masking, Opacity, etc.) ---\n    # Now, work with the guaranteed uint8 \'swapped_frame\'\n\n    if mouth_mask_enabled: # Check if mouth_mask is enabled\n        # Create a mask for the target face\n        face_mask = create_face_mask(target_face, original_frame) # Use original_frame for mask creation geometry\n\n        # Create the mouth mask using the ORIGINAL frame (before swap) for cutout\n        mouth_mask, mouth_cutout, mouth_box, lower_lip_polygon = (\n            create_lower_mouth_mask(target_face, original_frame) # Use original_frame for real mouth cutout\n        )\n\n        # Apply the mouth area only if mouth_cutout exists\n        if mouth_cutout is not None and mouth_box != (0,0,0,0):\n            # Apply mouth area (from original) onto the \'swapped_frame\'\n            swapped_frame = apply_mouth_area(\n                swapped_frame, mouth_cutout, mouth_box, face_mask, lower_lip_polygon\n            )\n\n            # Draw bounding box only while slider is being dragged\n            if getattr(modules.globals, "show_mouth_mask_box", False):\n                mouth_mask_data = (mouth_mask, mouth_cutout, mouth_box, lower_lip_polygon)\n                swapped_frame = draw_mouth_mask_visualization(\n                    swapped_frame, target_face, mouth_mask_data\n                )\n        \n    # --- Poisson Blending ---\n    # Mask derived from the swap\'s own affine (M) + swapped pixels (bgr_fake),\n    # so it tracks the swapped face exactly per-frame — no landmark jitter,\n    # no EMA, no lag. See _apply_poisson_blend.\n    if getattr(modules.globals, "poisson_blend", False):\n        swapped_frame = _apply_poisson_blend(\n            swapped_frame, original_frame, target_face, M, bgr_fake\n        )\n\n    # Apply opacity blend between the original frame and the swapped frame\n    if opacity >= 1.0:\n        return swapped_frame.astype(np.uint8)\n\n    # Blend the original_frame with the (potentially mouth-masked) swapped_frame\n    final_swapped_frame = gpu_add_weighted(original_frame.astype(np.uint8), 1 - opacity, swapped_frame.astype(np.uint8), opacity, 0)\n    return final_swapped_frame.astype(np.uint8)\n\n\n# --- START: Mac M1-M5 Optimized Face Detection ---\ndef get_faces_optimized(frame: Frame, use_cache: bool = True) -> Optional[List[Face]]:\n    """Optimized face detection for live mode on Apple Silicon"""\n    global LAST_DETECTION_TIME, FACE_DETECTION_CACHE\n    \n    if not use_cache or not IS_APPLE_SILICON:\n        # Standard detection\n        if modules.globals.many_faces:\n            return get_many_faces(frame)\n        else:\n            face = get_one_face(frame)\n            return [face] if face else None\n    \n    # Adaptive detection rate for live mode\n    current_time = time.time()\n    time_since_last = current_time - LAST_DETECTION_TIME\n    \n    # Skip detection if too soon (adaptive frame skipping)\n    if time_since_last < DETECTION_INTERVAL and FACE_DETECTION_CACHE:\n        return FACE_DETECTION_CACHE.get(\'faces\')\n    \n    # Perform detection\n    LAST_DETECTION_TIME = current_time\n    if modules.globals.many_faces:\n        faces = get_many_faces(frame)\n    else:\n        face = get_one_face(frame)\n        faces = [face] if face else None\n    \n    # Cache results\n    FACE_DETECTION_CACHE[\'faces\'] = faces\n    FACE_DETECTION_CACHE[\'timestamp\'] = current_time\n    \n    return faces\n# --- END: Mac M1-M5 Optimized Face Detection ---\n\n# --- START: Helper function for interpolation and sharpening ---\ndef apply_post_processing(current_frame: Frame, swapped_face_bboxes: List[np.ndarray]) -> Frame:\n    """Applies sharpening and interpolation with Apple Silicon optimizations."""\n    global PREVIOUS_FRAME_RESULT\n\n    sharpness_value = getattr(modules.globals, "sharpness", 0.0)\n    enable_interpolation = getattr(modules.globals, "enable_interpolation", False)\n\n    # Skip copy when no post-processing is active\n    if sharpness_value <= 0.0 and not enable_interpolation:\n        PREVIOUS_FRAME_RESULT = None\n        return current_frame\n\n    processed_frame = current_frame.copy()\n\n    # 1. Apply Sharpening (if enabled) with optimized kernel for Apple Silicon\n    sharpness_value = getattr(modules.globals, "sharpness", 0.0)\n    if sharpness_value > 0.0 and swapped_face_bboxes:\n        height, width = processed_frame.shape[:2]\n        for bbox in swapped_face_bboxes:\n            # Ensure bbox is iterable and has 4 elements\n            if not hasattr(bbox, \'__iter__\') or len(bbox) != 4:\n                # print(f"Warning: Invalid bbox format for sharpening: {bbox}") # Debug\n                continue\n            x1, y1, x2, y2 = bbox\n            # Ensure coordinates are integers and within bounds\n            try:\n                 x1, y1 = max(0, int(x1)), max(0, int(y1))\n                 x2, y2 = min(width, int(x2)), min(height, int(y2))\n            except ValueError:\n                # print(f"Warning: Could not convert bbox coordinates to int: {bbox}") # Debug\n                continue\n\n\n            if x2 <= x1 or y2 <= y1:\n                continue\n\n            face_region = processed_frame[y1:y2, x1:x2]\n            if face_region.size == 0:\n                continue\n\n            # Apply sharpening (GPU-accelerated when CUDA OpenCV is available)\n            try:\n                sigma = 2 if IS_APPLE_SILICON else 3\n                sharpened_region = gpu_sharpen(face_region, strength=sharpness_value, sigma=sigma)\n                processed_frame[y1:y2, x1:x2] = sharpened_region\n            except cv2.error:\n                pass\n\n\n    # 2. Apply Interpolation (if enabled)\n    enable_interpolation = getattr(modules.globals, "enable_interpolation", False)\n    interpolation_weight = getattr(modules.globals, "interpolation_weight", 0.2)\n\n    final_frame = processed_frame # Start with the current (potentially sharpened) frame\n\n    if enable_interpolation and 0 < interpolation_weight < 1:\n        if PREVIOUS_FRAME_RESULT is not None and PREVIOUS_FRAME_RESULT.shape == processed_frame.shape and PREVIOUS_FRAME_RESULT.dtype == processed_frame.dtype:\n            # Perform interpolation\n            try:\n                 final_frame = gpu_add_weighted(\n                    PREVIOUS_FRAME_RESULT, 1.0 - interpolation_weight,\n                    processed_frame, interpolation_weight,\n                    0\n                 )\n                 # Ensure final frame is uint8\n                 final_frame = np.clip(final_frame, 0, 255).astype(np.uint8)\n            except cv2.error as interp_e:\n                 # print(f"Warning: OpenCV error during interpolation: {interp_e}") # Debug\n                 final_frame = processed_frame # Use current frame if interpolation fails\n                 PREVIOUS_FRAME_RESULT = None # Reset state if error occurs\n\n            # Update the state for the next frame *with the interpolated result*\n            PREVIOUS_FRAME_RESULT = final_frame.copy()\n        else:\n            # If previous frame invalid or doesn\'t match, use current frame and update state\n            if PREVIOUS_FRAME_RESULT is not None and PREVIOUS_FRAME_RESULT.shape != processed_frame.shape:\n                # print("Info: Frame shape changed, resetting interpolation state.") # Debug\n                pass\n            PREVIOUS_FRAME_RESULT = processed_frame.copy()\n    else:\n         # Interpolation is off or weight is invalid — no need to cache\n         PREVIOUS_FRAME_RESULT = None\n\n\n    return final_frame\n# --- END: Helper function for interpolation and sharpening ---\n\n\ndef process_frame(source_face: Face, temp_frame: Frame, target_face: Face = None) -> Frame:\n    """Process a single frame, swapping source_face onto detected target(s).\n\n    Args:\n        target_face: Pre-detected target face. When provided, skips the\n            internal face detection call (saves ~30-40ms per frame).\n            Ignored when many_faces mode is active.\n    """\n    if getattr(modules.globals, "opacity", 1.0) == 0:\n        global PREVIOUS_FRAME_RESULT\n        PREVIOUS_FRAME_RESULT = None\n        return temp_frame\n\n    processed_frame = temp_frame\n    swapped_face_bboxes = []\n\n    if modules.globals.many_faces:\n        many_faces = get_many_faces(processed_frame)\n        if many_faces:\n            current_swap_target = processed_frame.copy()\n            for face in many_faces:\n                current_swap_target = swap_face(source_face, face, current_swap_target)\n                if face is not None and hasattr(face, "bbox") and face.bbox is not None:\n                    swapped_face_bboxes.append(face.bbox.astype(int))\n            processed_frame = current_swap_target\n    else:\n        if target_face is None:\n            target_face = get_one_face(processed_frame)\n        if target_face:\n            processed_frame = swap_face(source_face, target_face, processed_frame)\n            if hasattr(target_face, "bbox") and target_face.bbox is not None:\n                swapped_face_bboxes.append(target_face.bbox.astype(int))\n\n    final_frame = apply_post_processing(processed_frame, swapped_face_bboxes)\n    return final_frame\n\n\ndef process_frame_v2(temp_frame: Frame, temp_frame_path: str = "") -> Frame:\n    """Handles complex mapping scenarios (map_faces=True) and live streams."""\n    if getattr(modules.globals, "opacity", 1.0) == 0:\n        # If opacity is 0, no swap happens, so no post-processing needed.\n        # Also reset interpolation state if it was active.\n        global PREVIOUS_FRAME_RESULT\n        PREVIOUS_FRAME_RESULT = None\n        return temp_frame\n\n    processed_frame = temp_frame # Start with the input frame\n    swapped_face_bboxes = [] # Keep track of where swaps happened\n\n    # Determine source/target pairs based on mode\n    source_target_pairs = []\n\n    # Ensure maps exist before accessing them\n    source_target_map = getattr(modules.globals, "source_target_map", None)\n    simple_map = getattr(modules.globals, "simple_map", None)\n\n    # Check if target is a file path (image or video) or live stream\n    is_file_target = modules.globals.target_path and (is_image(modules.globals.target_path) or is_video(modules.globals.target_path))\n\n    if is_file_target:\n        # Processing specific image or video file with pre-analyzed maps\n        if source_target_map:\n            if modules.globals.many_faces:\n                source_face = default_source_face() # Use default source for all targets\n                if source_face:\n                    for map_data in source_target_map:\n                        if is_image(modules.globals.target_path):\n                            target_info = map_data.get("target", {})\n                            if target_info: # Check if target info exists\n                                target_face = target_info.get("face")\n                                if target_face:\n                                    source_target_pairs.append((source_face, target_face))\n                        elif is_video(modules.globals.target_path):\n                             # Find faces for the current frame_path in video map\n                             target_frames_data = map_data.get("target_faces_in_frame", [])\n                             if target_frames_data: # Check if frame data exists\n                                 target_frames = [f for f in target_frames_data if f and f.get("location") == temp_frame_path]\n                                 for frame_data in target_frames:\n                                     faces_in_frame = frame_data.get("faces", [])\n                                     if faces_in_frame: # Check if faces exist\n                                         for target_face in faces_in_frame:\n                                             source_target_pairs.append((source_face, target_face))\n            else: # Single face or specific mapping\n                 for map_data in source_target_map:\n                    source_info = map_data.get("source", {})\n                    if not source_info:\n                        continue # Skip if no source info\n                    source_face = source_info.get("face")\n                    if not source_face:\n                        continue # Skip if no source defined for this map entry\n\n                    if is_image(modules.globals.target_path):\n                        target_info = map_data.get("target", {})\n                        if target_info:\n                           target_face = target_info.get("face")\n                           if target_face:\n                              source_target_pairs.append((source_face, target_face))\n                    elif is_video(modules.globals.target_path):\n                        target_frames_data = map_data.get("target_faces_in_frame", [])\n                        if target_frames_data:\n                           target_frames = [f for f in target_frames_data if f and f.get("location") == temp_frame_path]\n                           for frame_data in target_frames:\n                               faces_in_frame = frame_data.get("faces", [])\n                               if faces_in_frame:\n                                  for target_face in faces_in_frame:\n                                      source_target_pairs.append((source_face, target_face))\n\n    else:\n        # Live stream or webcam processing (analyze faces on the fly)\n        detected_faces = get_many_faces(processed_frame)\n        if detected_faces:\n            if modules.globals.many_faces:\n                 source_face = default_source_face() # Use default source for all detected targets\n                 if source_face:\n                     for target_face in detected_faces:\n                        source_target_pairs.append((source_face, target_face))\n            elif simple_map:\n                # Use simple_map (source_faces <-> target_embeddings)\n                source_faces = simple_map.get("source_faces", [])\n                target_embeddings = simple_map.get("target_embeddings", [])\n\n                if source_faces and target_embeddings and len(source_faces) == len(target_embeddings):\n                     # Match detected faces to the closest target embedding\n                     if len(detected_faces) <= len(target_embeddings):\n                          # More targets defined than detected - match each detected face\n                          for detected_face in detected_faces:\n                              if detected_face.normed_embedding is None:\n                                  continue\n                              closest_idx, _ = find_closest_centroid(target_embeddings, detected_face.normed_embedding)\n                              if 0 <= closest_idx < len(source_faces):\n                                  source_target_pairs.append((source_faces[closest_idx], detected_face))\n                     else:\n                          # More faces detected than targets defined - match each target embedding to closest detected face\n                          detected_embeddings = [f.normed_embedding for f in detected_faces if f.normed_embedding is not None]\n                          detected_faces_with_embedding = [f for f in detected_faces if f.normed_embedding is not None]\n                          if not detected_embeddings:\n                              return processed_frame # No embeddings to match\n\n                          for i, target_embedding in enumerate(target_embeddings):\n                              if 0 <= i < len(source_faces): # Ensure source face exists for this embedding\n                                 closest_idx, _ = find_closest_centroid(detected_embeddings, target_embedding)\n                                 if 0 <= closest_idx < len(detected_faces_with_embedding):\n                                     source_target_pairs.append((source_faces[i], detected_faces_with_embedding[closest_idx]))\n            else: # Fallback: if no map, use default source for the single detected face (if any)\n                source_face = default_source_face()\n                target_face = get_one_face(processed_frame, detected_faces) # Use faces already detected\n                if source_face and target_face:\n                    source_target_pairs.append((source_face, target_face))\n\n\n    # Perform swaps based on the collected pairs\n    current_swap_target = processed_frame.copy() # Apply swaps sequentially\n    for source_face, target_face in source_target_pairs:\n        if source_face and target_face:\n            current_swap_target = swap_face(source_face, target_face, current_swap_target)\n            if target_face is not None and hasattr(target_face, "bbox") and target_face.bbox is not None:\n                swapped_face_bboxes.append(target_face.bbox.astype(int))\n    processed_frame = current_swap_target # Assign final result\n\n\n    # Apply sharpening and interpolation\n    final_frame = apply_post_processing(processed_frame, swapped_face_bboxes)\n\n    return final_frame\n\n\ndef process_frames(\n    source_path: str, temp_frame_paths: List[str], progress: Any = None\n) -> None:\n    """\n    Processes a list of frame paths (typically for video).\n    Optimized with better memory management and caching.\n    Iterates through frames, applies the appropriate swapping logic based on globals,\n    and saves the result back to the frame path. Handles multi-threading via caller.\n    """\n    # Determine which processing function to use based on map_faces global setting\n    use_v2 = getattr(modules.globals, "map_faces", False)\n    source_face = None # Initialize source_face\n\n    # --- Pre-load source face only if needed (Simple Mode: map_faces=False) ---\n    if not use_v2:\n        if not source_path or not os.path.exists(source_path):\n            update_status(f"Error: Source path invalid or not provided for simple mode: {source_path}", NAME)\n            # Log the error but allow proceeding; subsequent check will stop processing.\n        else:\n            try:\n                source_img = imread_unicode(source_path)\n                if source_img is None:\n                    # Specific error for file reading failure\n                    update_status(f"Error reading source image file {source_path}. Please check the path and file integrity.", NAME)\n                else:\n                    source_face = get_one_face(source_img)\n                    if source_face is None:\n                        # Specific message for no face detected after successful read\n                        update_status(f"Warning: Successfully read source image {source_path}, but no face was detected. Swaps will be skipped.", NAME)\n                    # Free memory immediately after extracting face\n                    del source_img\n            except Exception as e:\n                # Print the specific exception caught\n                import traceback\n                print(f"{NAME}: Caught exception during source image processing for {source_path}:")\n                traceback.print_exc() # Print the full traceback\n                update_status(f"Error during source image reading or analysis {source_path}: {e}", NAME)\n                # Log general exception during the process\n\n    total_frames = len(temp_frame_paths)\n    # update_status(f"Processing {total_frames} frames. Use V2 (map_faces): {use_v2}", NAME) # Optional Debug\n\n    # --- Stop processing entirely if in Simple Mode and source face is invalid ---\n    if not use_v2 and source_face is None:\n        update_status("Halting video processing: Invalid or no face detected in source image for simple mode.", NAME)\n        if progress:\n            # Ensure the progress bar completes if it was started\n            remaining_updates = total_frames - progress.n if hasattr(progress, \'n\') else total_frames\n            if remaining_updates > 0:\n                progress.update(remaining_updates)\n        return # Exit the function entirely\n\n    # --- Process each frame path provided in the list ---\n    # Note: In the current core.py multi_process_frame, temp_frame_paths will usually contain only ONE path per call.\n    for i, temp_frame_path in enumerate(temp_frame_paths):\n        # update_status(f"Processing frame {i+1}/{total_frames}: {os.path.basename(temp_frame_path)}", NAME) # Optional Debug\n\n        # Read the target frame\n        temp_frame = None\n        try:\n            temp_frame = imread_unicode(temp_frame_path)\n            if temp_frame is None:\n                print(f"{NAME}: Error: Could not read frame: {temp_frame_path}, skipping.")\n                if progress:\n                    progress.update(1)\n                continue # Skip this frame if read fails\n        except Exception as read_e:\n            print(f"{NAME}: Error reading frame {temp_frame_path}: {read_e}, skipping.")\n            if progress:\n                progress.update(1)\n            continue\n\n        # Select processing function and execute\n        result_frame = None\n        try:\n            if use_v2:\n                # V2 uses global maps and needs the frame path for lookup in video mode\n                # update_status(f"Using process_frame_v2 for: {os.path.basename(temp_frame_path)}", NAME) # Optional Debug\n                result_frame = process_frame_v2(temp_frame, temp_frame_path)\n            else:\n                # Simple mode uses the pre-loaded source_face (already checked for validity above)\n                # update_status(f"Using process_frame (simple) for: {os.path.basename(temp_frame_path)}", NAME) # Optional Debug\n                result_frame = process_frame(source_face, temp_frame) # source_face is guaranteed to be valid here\n\n            # Check if processing actually returned a frame\n            if result_frame is None:\n                 print(f"{NAME}: Warning: Processing returned None for frame {temp_frame_path}. Using original.")\n                 result_frame = temp_frame\n\n        except Exception as proc_e:\n            print(f"{NAME}: Error processing frame {temp_frame_path}: {proc_e}")\n            # import traceback # Optional for detailed debugging\n            # traceback.print_exc()\n            result_frame = temp_frame # Use original frame on processing error\n\n        # Write the result back to the same frame path with optimized compression\n        try:\n            # Use PNG compression level 3 (faster) instead of default 9\n            write_success = imwrite_unicode(temp_frame_path, result_frame, [cv2.IMWRITE_PNG_COMPRESSION, 3])\n            if not write_success:\n                print(f"{NAME}: Error: Failed to write processed frame to {temp_frame_path}")\n        except Exception as write_e:\n            print(f"{NAME}: Error writing frame {temp_frame_path}: {write_e}")\n        \n        # Free memory immediately after processing\n        del temp_frame\n        if result_frame is not None:\n            del result_frame\n\n        # Update progress bar\n        if progress:\n            progress.update(1)\n        # else: # Basic console progress (optional)\n        #     if (i + 1) % 10 == 0 or (i + 1) == total_frames: # Update every 10 frames or on last frame\n        #        update_status(f"Processed frame {i+1}/{total_frames}", NAME)\n\n\ndef process_image(source_path: str, target_path: str, output_path: str) -> None:\n    """Processes a single target image."""\n    # --- Reset interpolation state for single image processing ---\n    global PREVIOUS_FRAME_RESULT\n    PREVIOUS_FRAME_RESULT = None\n    # ---\n\n    use_v2 = getattr(modules.globals, "map_faces", False)\n\n    # Read target first\n    try:\n        target_frame = imread_unicode(target_path)\n        if target_frame is None:\n            update_status(f"Error: Could not read target image: {target_path}", NAME)\n            return\n    except Exception as read_e:\n        update_status(f"Error reading target image {target_path}: {read_e}", NAME)\n        return\n\n    result = None\n    try:\n        if use_v2:\n            if getattr(modules.globals, "many_faces", False):\n                 update_status("Processing image with \'map_faces\' and \'many_faces\'. Using pre-analysis map.", NAME)\n            # V2 processes based on global maps, doesn\'t need source_path here directly\n            # Assumes maps are pre-populated. Pass target_path for map lookup.\n            result = process_frame_v2(target_frame, target_path)\n\n        else: # Simple mode\n            try:\n                source_img = imread_unicode(source_path)\n                if source_img is None:\n                    update_status(f"Error: Could not read source image: {source_path}", NAME)\n                    return\n                source_face = get_one_face(source_img)\n                if not source_face:\n                    update_status(f"Error: No face found in source image: {source_path}", NAME)\n                    return\n            except Exception as src_e:\n                 update_status(f"Error reading or analyzing source image {source_path}: {src_e}", NAME)\n                 return\n\n            result = process_frame(source_face, target_frame)\n\n        # Write the result if processing was successful\n        if result is not None:\n            write_success = imwrite_unicode(output_path, result)\n            if write_success:\n                update_status(f"Output image saved to: {output_path}", NAME)\n            else:\n                update_status(f"Error: Failed to write output image to {output_path}", NAME)\n        else:\n            # This case might occur if process_frame/v2 returns None unexpectedly\n            update_status("Image processing failed (result was None).", NAME)\n\n    except Exception as proc_e:\n         update_status(f"Error during image processing: {proc_e}", NAME)\n         # import traceback\n         # traceback.print_exc()\n\n\ndef process_video(source_path: str, temp_frame_paths: List[str]) -> None:\n    """Sets up and calls the frame processing for video."""\n    # --- Reset interpolation state before starting video processing ---\n    global PREVIOUS_FRAME_RESULT\n    PREVIOUS_FRAME_RESULT = None\n    # ---\n\n    mode_desc = "\'map_faces\'" if getattr(modules.globals, "map_faces", False) else "\'simple\'"\n    if getattr(modules.globals, "map_faces", False) and getattr(modules.globals, "many_faces", False):\n        mode_desc += " and \'many_faces\'. Using pre-analysis map."\n    update_status(f"Processing video with {mode_desc} mode.", NAME)\n\n    # Pass the correct source_path (needed for simple mode in process_frames)\n    # The core processing logic handles calling the right frame function (process_frames)\n    modules.processors.frame.core.process_video(\n        source_path, temp_frame_paths, process_frames # Pass the newly modified process_frames\n    )\n\n# ==========================\n# MASKING FUNCTIONS (Mostly unchanged, added safety checks and minor improvements)\n# ==========================\n\ndef create_lower_mouth_mask(\n    face: Face, frame: Frame\n) -> (np.ndarray, np.ndarray, tuple, np.ndarray):\n    mask = np.zeros(frame.shape[:2], dtype=np.uint8)\n    mouth_cutout = None\n    lower_lip_polygon = None # Initialize\n    mouth_box = (0,0,0,0) # Initialize\n\n    # Validate face and landmarks\n    if face is None or not hasattr(face, \'landmark_2d_106\'):\n        # print("Warning: Invalid face object passed to create_lower_mouth_mask.")\n        return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\n    landmarks = face.landmark_2d_106\n\n    # Check landmark validity\n    if landmarks is None or not isinstance(landmarks, np.ndarray) or landmarks.shape[0] < 106:\n        # print("Warning: Invalid or insufficient landmarks for mouth mask.")\n        return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\n    try: # Wrap main logic in try-except\n        # Outer mouth/lip landmarks (52-63) — the lip outline only. In this\n        # repo\'s insightface 2d106 convention these 12 points, taken in index\n        # order, form a SIMPLE (non-self-intersecting) closed polygon that\n        # cv2.fillPoly fills as one solid region directly over the mouth.\n        # This is the last shipped, known-good landmark set; range(52,72)\n        # (the regression) added the inner-lip points and made the path\n        # self-intersect, and the ancient [65,66,62,...,0,8,7...] indices\n        # belong to a different/older landmark convention (they land on the\n        # inner lip + random jaw points, so the mask never covers the mouth).\n        lower_lip_order = list(range(52, 64))\n\n        # All indices must be valid for the loaded landmark set\n        if max(lower_lip_order) >= landmarks.shape[0]:\n            # print(f"Warning: Landmark index out of bounds for shape {landmarks.shape[0]}.")\n            return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\n        lower_lip_landmarks = landmarks[lower_lip_order].astype(np.float32)\n\n        # Filter out potential NaN or Inf values in landmarks\n        if not np.all(np.isfinite(lower_lip_landmarks)):\n            # print("Warning: Non-finite values detected in lower lip landmarks.")\n            return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\n        center = np.mean(lower_lip_landmarks, axis=0)\n        if not np.all(np.isfinite(center)): # Check center calculation\n            # print("Warning: Could not calculate valid center for mouth mask.")\n            return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\n        # Drive expansion from the Mouth Mask slider so it actually responds.\n        # The known-good version expanded by the now-unused mask_down_size\n        # constant, which is why the slider had no effect.\n        # s: 0.0 (slider ~0, tight lip outline) -> 1.0 (slider 100, mouth->chin).\n        mouth_mask_size = getattr(modules.globals, "mouth_mask_size", 0.0) # 0-100 slider\n        s = max(0.0, min(1.0, mouth_mask_size / 100.0))\n\n        # Uniformly scaling a simple polygon about its centroid keeps it simple\n        # (no self-intersection). x grows with expansion_factor; points below\n        # centre (toward the chin) also get an extra downward stretch so high\n        # slider values reach from the mouth down to the chin.\n        expansion_factor = 1.0 + s * 2.0          # 1.0x -> 3.0x\n        chin_bias = 1.0 + s * 2.0                  # extra downward stretch\n        offsets = lower_lip_landmarks - center\n        scale_y = np.where(offsets[:, 1] > 0,\n                           expansion_factor * chin_bias, expansion_factor)\n        expanded_landmarks = lower_lip_landmarks.copy()\n        expanded_landmarks[:, 0] = center[0] + offsets[:, 0] * expansion_factor\n        expanded_landmarks[:, 1] = center[1] + offsets[:, 1] * scale_y\n\n        # Ensure landmarks are finite after adjustments\n        if not np.all(np.isfinite(expanded_landmarks)):\n            # print("Warning: Non-finite values detected after expanding landmarks.")\n            return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\n        expanded_landmarks = expanded_landmarks.astype(np.int32)\n\n        min_x, min_y = np.min(expanded_landmarks, axis=0)\n        max_x, max_y = np.max(expanded_landmarks, axis=0)\n\n        # Add padding *after* initial min/max calculation\n        padding_ratio = 0.1 # Percentage padding\n        padding_x = int((max_x - min_x) * padding_ratio)\n        padding_y = int((max_y - min_y) * padding_ratio) # Use y-range for y-padding\n\n        # Apply padding and clamp to frame boundaries\n        frame_h, frame_w = frame.shape[:2]\n        min_x = max(0, min_x - padding_x)\n        min_y = max(0, min_y - padding_y)\n        max_x = min(frame_w, max_x + padding_x)\n        max_y = min(frame_h, max_y + padding_y)\n\n\n        if max_x > min_x and max_y > min_y:\n            # Create the mask ROI\n            mask_roi_h = max_y - min_y\n            mask_roi_w = max_x - min_x\n            mask_roi = np.zeros((mask_roi_h, mask_roi_w), dtype=np.uint8)\n\n            # Shift polygon coordinates relative to the ROI\'s top-left corner\n            polygon_relative_to_roi = expanded_landmarks - [min_x, min_y]\n\n            # Draw polygon on the ROI mask\n            cv2.fillPoly(mask_roi, [polygon_relative_to_roi], 255)\n\n            # Apply Gaussian blur (GPU-accelerated when available)\n            blur_k_size = getattr(modules.globals, "mask_blur_kernel", 15) # Default 15\n            blur_k_size = max(1, blur_k_size // 2 * 2 + 1) # Ensure odd\n            mask_roi = gpu_gaussian_blur(mask_roi, (blur_k_size, blur_k_size), 0)\n\n            # Place the mask ROI in the full-sized mask\n            mask[min_y:max_y, min_x:max_x] = mask_roi\n\n            # Extract the masked area from the *original* frame\n            mouth_cutout = frame[min_y:max_y, min_x:max_x].copy()\n\n            lower_lip_polygon = expanded_landmarks # Return polygon in original frame coords\n            mouth_box = (min_x, min_y, max_x, max_y) # Return the calculated box\n        else:\n            # print("Warning: Invalid mouth mask bounding box after padding/clamping.") # Optional debug\n            pass\n\n    except IndexError as idx_e:\n        # print(f"Warning: Landmark index out of bounds during mouth mask creation: {idx_e}") # Optional debug\n        pass\n    except Exception as e:\n        print(f"Error in create_lower_mouth_mask: {e}") # Print unexpected errors\n        # import traceback\n        # traceback.print_exc()\n        pass\n\n    # Return values, ensuring defaults if errors occurred\n    return mask, mouth_cutout, mouth_box, lower_lip_polygon\n\n\ndef draw_mouth_mask_visualization(\n    frame: Frame, face: Face, mouth_mask_data: tuple\n) -> Frame:\n\n    # Validate inputs\n    if frame is None or face is None or mouth_mask_data is None or len(mouth_mask_data) != 4:\n        return frame # Return original frame if inputs are invalid\n\n    mask, mouth_cutout, box, lower_lip_polygon = mouth_mask_data\n    (min_x, min_y, max_x, max_y) = box\n\n    # Check if polygon is valid for drawing\n    if lower_lip_polygon is None or not isinstance(lower_lip_polygon, np.ndarray) or len(lower_lip_polygon) < 3:\n        return frame # Cannot draw without a valid polygon\n\n    vis_frame = frame.copy()\n    height, width = vis_frame.shape[:2]\n\n    # Ensure box coordinates are valid integers within frame bounds\n    try:\n        min_x, min_y = max(0, int(min_x)), max(0, int(min_y))\n        max_x, max_y = min(width, int(max_x)), min(height, int(max_y))\n    except ValueError:\n        # print("Warning: Invalid coordinates for mask visualization box.")\n        return frame\n\n    if max_x <= min_x or max_y <= min_y:\n        return frame # Invalid box\n\n    # Draw the lower lip polygon (green outline)\n    try:\n         # Ensure polygon points are within frame boundaries before drawing\n         safe_polygon = lower_lip_polygon.copy()\n         safe_polygon[:, 0] = np.clip(safe_polygon[:, 0], 0, width - 1)\n         safe_polygon[:, 1] = np.clip(safe_polygon[:, 1], 0, height - 1)\n         cv2.polylines(vis_frame, [safe_polygon.astype(np.int32)], isClosed=True, color=(0, 255, 0), thickness=2)\n    except Exception as e:\n        print(f"Error drawing polygon for visualization: {e}") # Optional debug\n        pass\n\n    # Draw bounding box (red rectangle)\n    cv2.rectangle(vis_frame, (min_x, min_y), (max_x, max_y), (0, 0, 255), 2)\n\n    # Optional: Add labels\n    label_pos_y = min_y - 10 if min_y > 20 else max_y + 15 # Adjust position based on box location\n    label_pos_x = min_x\n    try:\n        cv2.putText(vis_frame, "Mouth Mask", (label_pos_x, label_pos_y),\n                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)\n    except Exception as e:\n        # print(f"Error drawing text for visualization: {e}") # Optional debug\n        pass\n\n\n    return vis_frame\n\n\ndef apply_mouth_area(\n    frame: np.ndarray,\n    mouth_cutout: np.ndarray,\n    mouth_box: tuple,\n    face_mask: np.ndarray, # Full face mask (for blending edges)\n    mouth_polygon: np.ndarray, # Specific polygon for the mouth area itself\n) -> np.ndarray:\n\n    # Basic validation\n    if (frame is None or mouth_cutout is None or mouth_box is None or\n        face_mask is None or mouth_polygon is None):\n        # print("Warning: Invalid input (None value) to apply_mouth_area") # Optional debug\n        return frame\n    if (mouth_cutout.size == 0 or face_mask.size == 0 or len(mouth_polygon) < 3):\n        # print("Warning: Invalid input (empty array/polygon) to apply_mouth_area") # Optional debug\n        return frame\n\n    try: # Wrap main logic in try-except\n        min_x, min_y, max_x, max_y = map(int, mouth_box) # Ensure integer coords\n        box_width = max_x - min_x\n        box_height = max_y - min_y\n\n        # Check box validity\n        if box_width <= 0 or box_height <= 0:\n            # print("Warning: Invalid mouth box dimensions in apply_mouth_area.")\n            return frame\n\n        # Define the Region of Interest (ROI) on the target frame (swapped frame)\n        frame_h, frame_w = frame.shape[:2]\n        # Clamp coordinates strictly within frame boundaries\n        min_y, max_y = max(0, min_y), min(frame_h, max_y)\n        min_x, max_x = max(0, min_x), min(frame_w, max_x)\n\n        # Recalculate box dimensions based on clamped coords\n        box_width = max_x - min_x\n        box_height = max_y - min_y\n        if box_width <= 0 or box_height <= 0:\n            # print("Warning: ROI became invalid after clamping in apply_mouth_area.")\n            return frame # ROI is invalid\n\n        roi = frame[min_y:max_y, min_x:max_x]\n\n        # Ensure ROI extraction was successful\n        if roi.size == 0:\n            # print("Warning: Extracted ROI is empty in apply_mouth_area.")\n            return frame\n\n        # Resize mouth cutout from original frame to fit the ROI size\n        resized_mouth_cutout = None\n        if roi.shape[:2] != mouth_cutout.shape[:2]:\n             # Check if mouth_cutout has valid dimensions before resizing\n             if mouth_cutout.shape[0] > 0 and mouth_cutout.shape[1] > 0:\n                  resized_mouth_cutout = gpu_resize(mouth_cutout, (box_width, box_height), interpolation=cv2.INTER_LINEAR)\n             else:\n                 # print("Warning: mouth_cutout has invalid dimensions, cannot resize.")\n                 return frame # Cannot proceed without valid cutout\n        else:\n             resized_mouth_cutout = mouth_cutout\n\n        # If resize failed or original was invalid\n        if resized_mouth_cutout is None or resized_mouth_cutout.size == 0:\n            # print("Warning: Mouth cutout is invalid after resize attempt.")\n            return frame\n\n        # --- Mask Creation ---\n        # Create a mask based on the mouth_polygon, relative to the ROI\n        polygon_mask_roi = np.zeros(roi.shape[:2], dtype=np.uint8)\n        adjusted_polygon = mouth_polygon - [min_x, min_y]\n        cv2.fillPoly(polygon_mask_roi, [adjusted_polygon.astype(np.int32)], 255)\n\n        # Feather the edges with Gaussian blur for smooth blending\n        feather_amount = max(1, min(30, min(box_width, box_height) // 8))\n        kernel_size = 2 * feather_amount + 1\n        feathered_mask = cv2.GaussianBlur(polygon_mask_roi.astype(np.float32), (kernel_size, kernel_size), 0)\n\n        # Normalize to [0.0, 1.0]\n        max_val = feathered_mask.max()\n        if max_val > 1e-6:\n            feathered_mask = feathered_mask / max_val\n        else:\n            feathered_mask.fill(0.0)\n\n        # --- Blending: paste original mouth onto swapped face ---\n        if len(frame.shape) == 3 and frame.shape[2] == 3:\n            mask_3ch = feathered_mask[:, :, np.newaxis].astype(np.float32)\n            inv_mask = 1.0 - mask_3ch\n\n            # Blend: (original_mouth * mask) + (swapped_face * (1 - mask))\n            blended_roi = (resized_mouth_cutout.astype(np.float32) * mask_3ch +\n                           roi.astype(np.float32) * inv_mask)\n\n            frame[min_y:max_y, min_x:max_x] = np.clip(blended_roi, 0, 255).astype(np.uint8)\n\n    except Exception as e:\n        print(f"Error applying mouth area: {e}") # Optional debug\n        # import traceback\n        # traceback.print_exc()\n        pass # Don\'t crash, just return the frame as is\n\n    return frame\n\n\ndef create_face_mask(face: Face, frame: Frame) -> np.ndarray:\n    """Creates a feathered mask covering the whole face area based on landmarks."""\n    if frame is None or not hasattr(frame, "shape") or len(frame.shape) < 2:\n        return np.zeros((0, 0), dtype=np.uint8)\n\n    mask = np.zeros(frame.shape[:2], dtype=np.uint8) # Start with uint8\n\n    # Validate inputs\n    if face is None or not hasattr(face, \'landmark_2d_106\'):\n        # print("Warning: Invalid face or frame for create_face_mask.")\n        return mask # Return empty mask\n\n    landmarks = face.landmark_2d_106\n    if landmarks is None or not isinstance(landmarks, np.ndarray) or landmarks.shape[0] < 106:\n        # print("Warning: Invalid or insufficient landmarks for face mask.")\n        return mask # Return empty mask\n\n    try: # Wrap main logic in try-except\n        # Filter out non-finite landmark values\n        if not np.all(np.isfinite(landmarks)):\n            # print("Warning: Non-finite values detected in landmarks for face mask.")\n            return mask\n\n        landmarks_int = landmarks.astype(np.int32)\n\n        # Use standard face outline landmarks (0-32)\n        # Use standard face outline (0-32)\n        face_outline = landmarks_int[0:33]\n\n        # Estimate forehead points to ensure mask covers the whole face (including forehead)\n        # This is critical for Poisson blending to work correctly on the forehead\n        eyebrows = landmarks_int[33:43]\n        if eyebrows.shape[0] > 0:\n            chin = landmarks_int[16]\n            eyebrow_center = np.mean(eyebrows, axis=0)\n            \n            # Vector from chin to eyebrows (upwards)\n            up_vector = eyebrow_center - chin\n            norm = np.linalg.norm(up_vector)\n            if norm > 0:\n                up_vector /= norm\n                \n                # Extend upwards by 1.0 of the chin-to-eyebrow distance (aggressive coverage)\n                # This ensures the mask covers the entire forehead for proper blending\n                forehead_offset = up_vector * (norm * 1.0)\n                \n                # Shift eyebrows up to create forehead points\n                forehead_points = eyebrows + forehead_offset\n                \n                # Expand the top points slightly outwards to cover forehead corners\n                # Calculate the center of the new top points\n                top_center = np.mean(forehead_points, axis=0)\n                \n                # Expand outwards by 20%\n                forehead_points = (forehead_points - top_center) * 1.2 + top_center\n                \n                # Combine outline and forehead points\n                face_outline = np.concatenate((face_outline, forehead_points.astype(np.int32)), axis=0)\n\n        # Calculate convex hull of these points\n        # Use try-except as convexHull can fail on degenerate input\n        try:\n             hull = cv2.convexHull(face_outline.astype(np.float32)) # Use float for accuracy\n             if hull is None or len(hull) < 3:\n                 # print("Warning: Convex hull calculation failed or returned too few points.")\n                 # Fallback: use bounding box of landmarks? Or just return empty mask?\n                 return mask\n\n             # Draw the filled convex hull on the mask\n             cv2.fillConvexPoly(mask, hull.astype(np.int32), 255)\n        except Exception as hull_e:\n             print(f"Error creating convex hull for face mask: {hull_e}")\n             return mask # Return empty mask on error\n\n\n        # Apply Gaussian blur to feather the mask edges (GPU-accelerated when available)\n        blur_k_size = getattr(modules.globals, "face_mask_blur", 31) # Default 31\n        blur_k_size = max(1, blur_k_size // 2 * 2 + 1) # Ensure odd and positive\n        mask = gpu_gaussian_blur(mask, (blur_k_size, blur_k_size), 0)\n\n        # --- Optional: Return float mask for apply_mouth_area ---\n        # mask = mask.astype(float) / 255.0\n        # ---\n\n    except IndexError:\n        # print("Warning: Landmark index out of bounds for face mask.") # Optional debug\n        pass\n    except Exception as e:\n        print(f"Error creating face mask: {e}") # Print unexpected errors\n        # import traceback\n        # traceback.print_exc()\n        pass\n\n    return mask # Return uint8 mask\n\n\ndef apply_color_transfer(source, target):\n    """\n    Apply color transfer using LAB color space. Handles potential division by zero and ensures output is uint8.\n    """\n    # Input validation\n    if source is None or target is None or source.size == 0 or target.size == 0:\n        # print("Warning: Invalid input to apply_color_transfer.")\n        return source # Return original source if invalid input\n\n    # Ensure images are 3-channel BGR uint8\n    if len(source.shape) != 3 or source.shape[2] != 3 or source.dtype != np.uint8:\n        # print("Warning: Source image for color transfer is not uint8 BGR.")\n        # Attempt conversion if possible, otherwise return original\n        try:\n            if len(source.shape) == 2: # Grayscale\n                source = cv2.cvtColor(source, cv2.COLOR_GRAY2BGR)\n            source = np.clip(source, 0, 255).astype(np.uint8)\n            if len(source.shape) != 3 or source.shape[2] != 3:\n                raise ValueError("Conversion failed")\n        except Exception:\n            return source\n    if len(target.shape) != 3 or target.shape[2] != 3 or target.dtype != np.uint8:\n        # print("Warning: Target image for color transfer is not uint8 BGR.")\n        try:\n            if len(target.shape) == 2: # Grayscale\n                target = cv2.cvtColor(target, cv2.COLOR_GRAY2BGR)\n            target = np.clip(target, 0, 255).astype(np.uint8)\n            if len(target.shape) != 3 or target.shape[2] != 3:\n                raise ValueError("Conversion failed")\n        except Exception:\n             return source # Return original source if target invalid\n\n    result_bgr = source # Default to original source in case of errors\n\n    try:\n        # Convert to float32 [0, 1] range for LAB conversion\n        source_float = source.astype(np.float32) / 255.0\n        target_float = target.astype(np.float32) / 255.0\n\n        source_lab = cv2.cvtColor(source_float, cv2.COLOR_BGR2LAB)\n        target_lab = cv2.cvtColor(target_float, cv2.COLOR_BGR2LAB)\n\n        # Compute statistics\n        source_mean, source_std = cv2.meanStdDev(source_lab)\n        target_mean, target_std = cv2.meanStdDev(target_lab)\n\n        # Reshape for broadcasting\n        source_mean = source_mean.reshape((1, 1, 3))\n        source_std = source_std.reshape((1, 1, 3))\n        target_mean = target_mean.reshape((1, 1, 3))\n        target_std = target_std.reshape((1, 1, 3))\n\n        # Avoid division by zero or very small std deviations (add epsilon)\n        epsilon = 1e-6\n        source_std = np.maximum(source_std, epsilon)\n        # target_std = np.maximum(target_std, epsilon) # Target std can be small\n\n        # Perform color transfer in LAB space\n        result_lab = (source_lab - source_mean) * (target_std / source_std) + target_mean\n\n        # --- No explicit clipping needed in LAB space typically ---\n        # Clipping is handled implicitly by the conversion back to BGR and then to uint8\n\n        # Convert back to BGR float [0, 1]\n        result_bgr_float = cv2.cvtColor(result_lab, cv2.COLOR_LAB2BGR)\n\n        # Clip final BGR values to [0, 1] range before scaling to [0, 255]\n        result_bgr_float = np.clip(result_bgr_float, 0.0, 1.0)\n\n        # Convert back to uint8 [0, 255]\n        result_bgr = (result_bgr_float * 255.0).astype("uint8")\n\n    except cv2.error as e:\n         # print(f"OpenCV error during color transfer: {e}. Returning original source.") # Optional debug\n         return source # Return original source if conversion fails\n    except Exception as e:\n         # print(f"Unexpected color transfer error: {e}. Returning original source.") # Optional debug\n         # import traceback\n         # traceback.print_exc()\n         return source\n\n    return result_bgr\n',
    'modules/run.py': "#!/usr/bin/env python3\n\n# Import the tkinter fix to patch the ScreenChanged error (module patches Tk on import)\nimport tkinter_fix  # noqa: F401\n\nimport core\n\nif __name__ == '__main__':\n    core.run()\n",
    'modules/tkinter_fix.py': 'import tkinter\n\n# Only needs to be imported once at the beginning of the application\ndef apply_patch():\n    # Create a monkey patch for the internal _tkinter module\n    original_init = tkinter.Tk.__init__\n    \n    def patched_init(self, *args, **kwargs):\n        # Call the original init\n        original_init(self, *args, **kwargs)\n        \n        # Define the missing ::tk::ScreenChanged procedure\n        self.tk.eval("""\n        if {[info commands ::tk::ScreenChanged] == ""} {\n            proc ::tk::ScreenChanged {args} {\n                # Do nothing\n                return\n            }\n        }\n        """)\n    \n    # Apply the monkey patch\n    tkinter.Tk.__init__ = patched_init\n\n# Apply the patch automatically when this module is imported\napply_patch() ',
    'modules/typing.py': 'from typing import Any\n\nfrom insightface.app.common import Face\nimport numpy\n\nFace = Face\nFrame = numpy.ndarray[Any, Any]\n',
    'modules/ui.py': '"""PySide6 UI for Deep-Live-Cam.\n\nPublic API kept stable for the rest of the codebase:\n    init(start, destroy, lang) -> _Window\n        Returned object has .mainloop() that core.py calls.\n    update_status(text)\n        Thread-safe; routed through Qt signal when called off-UI.\n    check_and_ignore_nsfw(target, destroy=None) -> bool\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport platform\nimport queue\nimport sys\nimport tempfile\nimport threading\nimport time\nimport webbrowser\nfrom typing import Callable, List, Optional, Tuple\n\nimport cv2\nimport numpy as np\nimport requests\nfrom PIL import Image, ImageOps\nfrom PySide6.QtCore import (\n    QObject,\n    QThread,\n    QTimer,\n    Qt,\n    Signal,\n)\nfrom PySide6.QtGui import QImage, QPixmap\nfrom PySide6.QtWidgets import (\n    QApplication,\n    QCheckBox,\n    QComboBox,\n    QDialog,\n    QFileDialog,\n    QGridLayout,\n    QGroupBox,\n    QHBoxLayout,\n    QLabel,\n    QMainWindow,\n    QPushButton,\n    QScrollArea,\n    QSizePolicy,\n    QSlider,\n    QVBoxLayout,\n    QWidget,\n)\n\nimport modules.globals\nimport modules.metadata\nfrom modules.capturer import get_video_frame, get_video_frame_total\nfrom modules.face_analyser import (\n    add_blank_map,\n    detect_many_faces_fast,\n    detect_one_face_fast,\n    ensure_landmarks,\n    get_one_face,\n    get_unique_faces_from_target_image,\n    get_unique_faces_from_target_video,\n    has_valid_map,\n    simplify_maps,\n)\nfrom modules.gettext import LanguageManager\nfrom modules.gpu_processing import gpu_cvt_color, gpu_flip, gpu_resize\nfrom modules.processors.frame.core import get_frame_processors_modules\nfrom modules.utilities import (\n    has_image_extension,\n    is_image,\n    is_video,\n)\nfrom modules import imread_unicode\nfrom modules.video_capture import VideoCapturer\n\nif platform.system() == "Windows":\n    from pygrabber.dshow_graph import FilterGraph\n\nimport json\n\n\n# ─── constants ────────────────────────────────────────────────────────────\n\nROOT_HEIGHT = 820\nROOT_WIDTH = 640\n\nPREVIEW_MAX_HEIGHT = 700\nPREVIEW_MAX_WIDTH = 1200\nPREVIEW_DEFAULT_WIDTH = 640\nPREVIEW_DEFAULT_HEIGHT = 360\n\nPOPUP_WIDTH = 750\nPOPUP_HEIGHT = 810\nPOPUP_SCROLL_WIDTH = 720\nPOPUP_SCROLL_HEIGHT = 700\n\nPOPUP_LIVE_WIDTH = 900\nPOPUP_LIVE_HEIGHT = 820\nPOPUP_LIVE_SCROLL_WIDTH = 870\nPOPUP_LIVE_SCROLL_HEIGHT = 700\n\nMAPPER_PREVIEW_SIZE = 100\nSOURCE_TARGET_PREVIEW_SIZE = 200\n\n\n# ─── modern dark stylesheet ───────────────────────────────────────────────\n\nQSS = """\nQMainWindow, QDialog { background-color: #1e1e1e; color: #e6e6e6; }\nQWidget { color: #e6e6e6; font-family: "Segoe UI", "SF Pro Display", "Helvetica Neue", Arial, sans-serif; font-size: 11pt; }\n\nQGroupBox {\n    background-color: #262626;\n    border: 1px solid #333333;\n    border-radius: 10px;\n    margin-top: 14px;\n    padding-top: 18px;\n    font-weight: 600;\n}\nQGroupBox::title {\n    subcontrol-origin: margin;\n    subcontrol-position: top left;\n    padding: 0 8px;\n    color: #9ec5ff;\n}\n\nQPushButton {\n    background-color: #2d6cdf;\n    color: white;\n    border: none;\n    border-radius: 8px;\n    padding: 8px 16px;\n    font-weight: 600;\n}\nQPushButton:hover  { background-color: #3a7af0; }\nQPushButton:pressed{ background-color: #1d57c2; }\nQPushButton:disabled { background-color: #444; color: #888; }\nQPushButton#secondary {\n    background-color: #3a3a3a;\n}\nQPushButton#secondary:hover { background-color: #4a4a4a; }\nQPushButton#danger { background-color: #c2412d; }\nQPushButton#danger:hover  { background-color: #d8523c; }\n\nQComboBox {\n    background-color: #2a2a2a;\n    border: 1px solid #404040;\n    border-radius: 6px;\n    padding: 6px 10px;\n    min-height: 24px;\n}\nQComboBox:hover { border-color: #2d6cdf; }\nQComboBox QAbstractItemView {\n    background-color: #2a2a2a;\n    selection-background-color: #2d6cdf;\n    border: 1px solid #404040;\n}\n\nQCheckBox {\n    spacing: 8px;\n    padding: 4px 0;\n}\nQCheckBox::indicator {\n    width: 36px; height: 18px;\n    border-radius: 9px;\n    background-color: #3a3a3a;\n}\nQCheckBox::indicator:checked {\n    background-color: #2d6cdf;\n}\n\nQSlider::groove:horizontal {\n    height: 6px;\n    background: #3a3a3a;\n    border-radius: 3px;\n}\nQSlider::handle:horizontal {\n    background: #ffffff;\n    width: 16px; height: 16px;\n    margin: -5px 0;\n    border-radius: 8px;\n    border: 1px solid #cccccc;\n}\nQSlider::sub-page:horizontal {\n    background: #2d6cdf;\n    border-radius: 3px;\n}\n\nQLabel#imageDrop {\n    background-color: #2a2a2a;\n    border: 2px dashed #444;\n    border-radius: 8px;\n}\nQLabel#statusLabel {\n    color: #b9b9b9;\n    font-size: 10pt;\n    font-style: italic;\n}\nQLabel#linkLabel {\n    color: #6ea8ff;\n    text-decoration: underline;\n}\n\nQScrollArea { border: none; background: transparent; }\n\nQFrame#card {\n    background-color: #262626;\n    border-radius: 10px;\n}\n"""\n\n\n# ─── module-level state ───────────────────────────────────────────────────\n\n_APP: Optional[QApplication] = None\n_MAIN: Optional["MainWindow"] = None\n_PREVIEW: Optional["PreviewWindow"] = None\n_WEBCAM_PREVIEW: Optional["WebcamPreviewWindow"] = None\n_MAPPER: Optional["MapperDialog"] = None\n_LIVE_MAPPER: Optional["LiveMapperDialog"] = None\n_LANG: Optional[LanguageManager] = None\n_BRIDGE: Optional["_UIBridge"] = None\n\n\ndef _(text: str) -> str:\n    """Translate via LanguageManager; falls back to identity."""\n    if _LANG is None:\n        return text\n    return _LANG._(text)\n\n\n# Preserve original cwd state for file dialogs.\n_RECENT_SOURCE_DIR: Optional[str] = None\n_RECENT_TARGET_DIR: Optional[str] = None\n_RECENT_OUTPUT_DIR: Optional[str] = None\n\n\n# ─── image utilities ─────────────────────────────────────────────────────\n\n\ndef fit_image_to_size(image, width: int, height: int):\n    """BGR ndarray → BGR ndarray scaled to fit within (width, height)."""\n    if width is None and height is None or width <= 0 or height <= 0:\n        return image\n    h, w = image.shape[:2]\n    ratio_w = width / w\n    ratio_h = height / h\n    ratio = min(ratio_w, ratio_h)\n    new_size = (max(1, int(w * ratio)), max(1, int(h * ratio)))\n    return gpu_resize(image, dsize=new_size)\n\n\ndef _bgr_to_qpixmap(bgr: np.ndarray) -> QPixmap:\n    """Zero-copy BGR ndarray → QPixmap."""\n    h, w = bgr.shape[:2]\n    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\n    qimg = QImage(rgb.data, w, h, w * 3, QImage.Format.Format_RGB888).copy()\n    return QPixmap.fromImage(qimg)\n\n\ndef _pil_to_qpixmap(image: Image.Image) -> QPixmap:\n    """PIL.Image → QPixmap."""\n    image = image.convert("RGBA")\n    data = image.tobytes("raw", "RGBA")\n    qimg = QImage(data, image.width, image.height, QImage.Format.Format_RGBA8888)\n    return QPixmap.fromImage(qimg.copy())\n\n\ndef render_image_preview(image_path: str, size: Tuple[int, int]) -> QPixmap:\n    image = Image.open(image_path)\n    if size:\n        image = ImageOps.fit(image, size, Image.LANCZOS)\n    return _pil_to_qpixmap(image)\n\n\ndef render_video_preview(\n    video_path: str, size: Tuple[int, int], frame_number: int = 0\n) -> Optional[QPixmap]:\n    capture = cv2.VideoCapture(video_path)\n    try:\n        if frame_number:\n            capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number)\n        has_frame, frame = capture.read()\n        if not has_frame:\n            return None\n        image = Image.fromarray(gpu_cvt_color(frame, cv2.COLOR_BGR2RGB))\n        if size:\n            image = ImageOps.fit(image, size, Image.LANCZOS)\n        return _pil_to_qpixmap(image)\n    finally:\n        capture.release()\n\n\n# ─── persistence ─────────────────────────────────────────────────────────\n\n\ndef save_switch_states():\n    state = {\n        "keep_fps": modules.globals.keep_fps,\n        "keep_audio": modules.globals.keep_audio,\n        "keep_frames": modules.globals.keep_frames,\n        "many_faces": modules.globals.many_faces,\n        "map_faces": modules.globals.map_faces,\n        "poisson_blend": modules.globals.poisson_blend,\n        "color_correction": modules.globals.color_correction,\n        "nsfw_filter": modules.globals.nsfw_filter,\n        "live_mirror": modules.globals.live_mirror,\n        "live_resizable": modules.globals.live_resizable,\n        "fp_ui": modules.globals.fp_ui,\n        "show_fps": modules.globals.show_fps,\n        "mouth_mask": modules.globals.mouth_mask,\n        "show_mouth_mask_box": modules.globals.show_mouth_mask_box,\n        "mouth_mask_size": modules.globals.mouth_mask_size,\n    }\n    try:\n        with open("switch_states.json", "w") as f:\n            json.dump(state, f)\n    except OSError:\n        pass\n\n\ndef load_switch_states():\n    try:\n        with open("switch_states.json", "r") as f:\n            state = json.load(f)\n        modules.globals.keep_fps = state.get("keep_fps", True)\n        modules.globals.keep_audio = state.get("keep_audio", True)\n        modules.globals.keep_frames = state.get("keep_frames", False)\n        modules.globals.many_faces = state.get("many_faces", False)\n        modules.globals.map_faces = state.get("map_faces", False)\n        modules.globals.poisson_blend = state.get("poisson_blend", False)\n        modules.globals.color_correction = state.get("color_correction", False)\n        modules.globals.nsfw_filter = state.get("nsfw_filter", False)\n        modules.globals.live_mirror = state.get("live_mirror", False)\n        modules.globals.live_resizable = state.get("live_resizable", False)\n        modules.globals.fp_ui = state.get("fp_ui", {"face_enhancer": False})\n        modules.globals.show_fps = state.get("show_fps", False)\n        # Mouth mask always starts disabled (slider at 0) on launch,\n        # regardless of the persisted value — enable it explicitly each session.\n        modules.globals.mouth_mask_size = 0.0\n        modules.globals.mouth_mask = False\n        modules.globals.show_mouth_mask_box = False\n    except FileNotFoundError:\n        pass\n    except (OSError, json.JSONDecodeError):\n        pass\n\n\n# ─── thread-safe status bridge ───────────────────────────────────────────\n\n\nclass _UIBridge(QObject):\n    """Single QObject that owns cross-thread signals."""\n\n    statusChanged = Signal(str)\n\n\ndef _emit_status(text: str) -> None:\n    if _BRIDGE is None:\n        print(text)\n        return\n    _BRIDGE.statusChanged.emit(text)\n\n\n# ─── public API ──────────────────────────────────────────────────────────\n\n\ndef update_status(text: str) -> None:\n    """Thread-safe status update — uses signal if called off-UI thread."""\n    _emit_status(_(text))\n    if _APP is not None and QThread.currentThread() is _APP.thread():\n        # On UI thread — flush events so the user sees the update during\n        # long synchronous start() runs.\n        _APP.processEvents()\n\n\ndef check_and_ignore_nsfw(target, destroy: Optional[Callable] = None) -> bool:\n    from numpy import ndarray\n    from modules.predicter import predict_frame, predict_image, predict_video\n\n    check_nsfw = None\n    if isinstance(target, str):\n        check_nsfw = predict_image if has_image_extension(target) else predict_video\n    elif isinstance(target, ndarray):\n        check_nsfw = predict_frame\n\n    if check_nsfw and check_nsfw(target):\n        if destroy:\n            destroy(to_quit=False)\n        update_status("Processing ignored!")\n        return True\n    return False\n\n\n# ─── camera enumeration (unchanged from tk version) ──────────────────────\n\n\ndef get_available_cameras() -> Tuple[List[int], List[str]]:\n    if platform.system() == "Windows":\n        try:\n            graph = FilterGraph()\n            devices = graph.get_input_devices()\n            if devices:\n                return list(range(len(devices))), devices\n            return [], ["No cameras found"]\n        except Exception as exc:\n            print(f"Error detecting cameras: {exc}")\n            return [], ["No cameras found"]\n\n    if platform.system() == "Darwin":\n        return [0, 1], ["Camera 0", "Camera 1"]\n\n    # Linux probe\n    indices: List[int] = []\n    names: List[str] = []\n    for i in range(10):\n        cap = cv2.VideoCapture(i)\n        if cap.isOpened():\n            indices.append(i)\n            names.append(f"Camera {i}")\n            cap.release()\n    return (indices, names) if names else ([], ["No cameras found"])\n\n\n# ─── main window ─────────────────────────────────────────────────────────\n\n\ndef _make_image_drop(text: str, size: Tuple[int, int]) -> QLabel:\n    label = QLabel(text)\n    label.setObjectName("imageDrop")\n    label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n    label.setFixedSize(size[0], size[1])\n    label.setText(text)\n    return label\n\n\nclass _Switch(QWidget):\n    """Compact toggle switch with label + optional tooltip."""\n\n    toggled = Signal(bool)\n\n    def __init__(self, text: str, initial: bool, tooltip: str = ""):\n        super().__init__()\n        layout = QHBoxLayout(self)\n        layout.setContentsMargins(0, 0, 0, 0)\n        self._checkbox = QCheckBox(text)\n        self._checkbox.setChecked(initial)\n        self._checkbox.toggled.connect(self.toggled.emit)\n        if tooltip:\n            self._checkbox.setToolTip(tooltip)\n        layout.addWidget(self._checkbox)\n        layout.addStretch(1)\n\n    def isChecked(self) -> bool:\n        return self._checkbox.isChecked()\n\n    def setChecked(self, value: bool) -> None:\n        self._checkbox.setChecked(value)\n\n\nclass MainWindow(QMainWindow):\n    def __init__(self, start_cb: Callable, destroy_cb: Callable):\n        super().__init__()\n        load_switch_states()\n        self._start_cb = start_cb\n        self._destroy_cb = destroy_cb\n\n        self.setWindowTitle(\n            f"{modules.metadata.name} {modules.metadata.version} {modules.metadata.edition}"\n        )\n        self.setMinimumSize(ROOT_WIDTH, ROOT_HEIGHT)\n        self.resize(ROOT_WIDTH, ROOT_HEIGHT)\n\n        root = QWidget()\n        self.setCentralWidget(root)\n        layout = QVBoxLayout(root)\n        layout.setContentsMargins(16, 16, 16, 16)\n        layout.setSpacing(12)\n\n        # Source/Target row\n        layout.addLayout(self._build_image_row())\n\n        # Options grid\n        layout.addWidget(self._build_options_card())\n\n        # Sliders card\n        layout.addWidget(self._build_sliders_card())\n\n        # Action buttons\n        layout.addLayout(self._build_action_row())\n\n        # Camera selection\n        layout.addWidget(self._build_camera_card())\n\n        # Status & footer\n        self._status_label = QLabel("")\n        self._status_label.setObjectName("statusLabel")\n        self._status_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n        layout.addWidget(self._status_label)\n\n        footer = QLabel("Deep Live Cam")\n        footer.setObjectName("linkLabel")\n        footer.setAlignment(Qt.AlignmentFlag.AlignCenter)\n        footer.setCursor(Qt.CursorShape.PointingHandCursor)\n        footer.mousePressEvent = lambda _e: webbrowser.open("https://deeplivecam.net")\n        layout.addWidget(footer)\n\n    # ── image row ────────────────────────────────────────────────────────\n\n    def _build_image_row(self) -> QHBoxLayout:\n        row = QHBoxLayout()\n        row.setSpacing(16)\n\n        # Source column\n        src_col = QVBoxLayout()\n        self.source_label = _make_image_drop(_("Source face"), (200, 200))\n        src_col.addWidget(self.source_label, alignment=Qt.AlignmentFlag.AlignCenter)\n        src_row = QHBoxLayout()\n        self.btn_select_source = QPushButton(_("Select a face"))\n        self.btn_select_source.setToolTip(\n            _("Choose the source face image to swap onto the target")\n        )\n        self.btn_select_source.clicked.connect(self._on_select_source)\n        self.btn_random_face = QPushButton("🔄")\n        self.btn_random_face.setObjectName("secondary")\n        self.btn_random_face.setFixedWidth(40)\n        self.btn_random_face.setToolTip(\n            _("Get a random face from thispersondoesnotexist.com")\n        )\n        self.btn_random_face.clicked.connect(self._on_random_face)\n        src_row.addWidget(self.btn_select_source)\n        src_row.addWidget(self.btn_random_face)\n        src_col.addLayout(src_row)\n\n        # Swap button column\n        swap_col = QVBoxLayout()\n        swap_col.addStretch(1)\n        self.btn_swap = QPushButton("↔")\n        self.btn_swap.setObjectName("secondary")\n        self.btn_swap.setFixedSize(44, 44)\n        self.btn_swap.setToolTip(_("Swap source and target images"))\n        self.btn_swap.clicked.connect(self._on_swap_paths)\n        swap_col.addWidget(self.btn_swap, alignment=Qt.AlignmentFlag.AlignCenter)\n        swap_col.addStretch(1)\n\n        # Target column\n        tgt_col = QVBoxLayout()\n        self.target_label = _make_image_drop(_("Target"), (200, 200))\n        tgt_col.addWidget(self.target_label, alignment=Qt.AlignmentFlag.AlignCenter)\n        self.btn_select_target = QPushButton(_("Select a target"))\n        self.btn_select_target.setToolTip(\n            _("Choose the target image or video to apply face swap to")\n        )\n        self.btn_select_target.clicked.connect(self._on_select_target)\n        tgt_col.addWidget(self.btn_select_target)\n\n        row.addLayout(src_col)\n        row.addLayout(swap_col)\n        row.addLayout(tgt_col)\n        return row\n\n    # ── options card ─────────────────────────────────────────────────────\n\n    def _build_options_card(self) -> QGroupBox:\n        card = QGroupBox(_("Options"))\n        grid = QGridLayout(card)\n        grid.setHorizontalSpacing(20)\n        grid.setVerticalSpacing(6)\n\n        def make(field, label, tip):\n            sw = _Switch(_(label), getattr(modules.globals, field), _(tip))\n            sw.toggled.connect(\n                lambda v, f=field: (\n                    setattr(modules.globals, f, v),\n                    save_switch_states(),\n                )\n            )\n            return sw\n\n        self.sw_keep_fps = make("keep_fps", "Keep fps",\n                                "Output video keeps the original frame rate")\n        self.sw_keep_audio = make("keep_audio", "Keep audio",\n                                  "Copy audio track from the source video to output")\n        self.sw_keep_frames = make("keep_frames", "Keep frames",\n                                   "Keep extracted frames on disk after processing")\n        self.sw_many_faces = make("many_faces", "Many faces",\n                                  "Swap every detected face, not just the primary one")\n        self.sw_poisson = make("poisson_blend", "Poisson Blend",\n                               "Blend face edges smoothly using Poisson blending")\n        self.sw_color_fix = make("color_correction", "Fix Blueish Cam",\n                                 "Fix blue/green color cast from some webcams")\n        self.sw_show_fps = make("show_fps", "Show FPS",\n                                "Display frames-per-second counter on the live preview")\n\n        # Map faces is special — closes mapper when toggled off.\n        self.sw_map_faces = _Switch(_("Map faces"), modules.globals.map_faces,\n                                    _("Manually assign which source face maps to which target face"))\n        self.sw_map_faces.toggled.connect(self._on_map_faces_toggled)\n\n        # Layout: 2 columns of switches\n        items = [\n            self.sw_keep_fps, self.sw_keep_audio,\n            self.sw_keep_frames, self.sw_many_faces,\n            self.sw_map_faces, self.sw_show_fps,\n            self.sw_poisson, self.sw_color_fix,\n        ]\n        for i, w in enumerate(items):\n            grid.addWidget(w, i // 2, i % 2)\n\n        # Face enhancer dropdown\n        enhancer_label = QLabel(_("Face Enhancer:"))\n        grid.addWidget(enhancer_label, len(items) // 2, 0)\n\n        self.cb_enhancer = QComboBox()\n        self.cb_enhancer.addItems(["None", "GFPGAN", "GPEN-512", "GPEN-256"])\n        initial = "None"\n        if modules.globals.fp_ui.get("face_enhancer", False):\n            initial = "GFPGAN"\n        elif modules.globals.fp_ui.get("face_enhancer_gpen512", False):\n            initial = "GPEN-512"\n        elif modules.globals.fp_ui.get("face_enhancer_gpen256", False):\n            initial = "GPEN-256"\n        self.cb_enhancer.setCurrentText(initial)\n        self.cb_enhancer.currentTextChanged.connect(self._on_enhancer_change)\n        self.cb_enhancer.setToolTip(_("Select a face enhancement model (None = no enhancement)"))\n        grid.addWidget(self.cb_enhancer, len(items) // 2, 1)\n\n        return card\n\n    # ── sliders card ─────────────────────────────────────────────────────\n\n    def _build_sliders_card(self) -> QGroupBox:\n        card = QGroupBox(_("Refinement"))\n        grid = QGridLayout(card)\n        grid.setHorizontalSpacing(12)\n        grid.setVerticalSpacing(10)\n\n        def slider(min_v, max_v, default, denom, on_change):\n            s = QSlider(Qt.Orientation.Horizontal)\n            s.setRange(int(min_v * denom), int(max_v * denom))\n            s.setValue(int(default * denom))\n            s.valueChanged.connect(lambda iv: on_change(iv / denom))\n            return s\n\n        # Transparency\n        grid.addWidget(QLabel(_("Transparency")), 0, 0)\n        self.s_transparency = slider(0.0, 1.0, 1.0, 100, self._on_transparency_change)\n        self.s_transparency.setToolTip(\n            _("Blend between original and swapped face (0% = original, 100% = fully swapped)")\n        )\n        grid.addWidget(self.s_transparency, 0, 1)\n\n        # Sharpness\n        grid.addWidget(QLabel(_("Sharpness")), 1, 0)\n        self.s_sharpness = slider(0.0, 5.0, 0.0, 10, self._on_sharpness_change)\n        self.s_sharpness.setToolTip(_("Sharpen the enhanced face output"))\n        grid.addWidget(self.s_sharpness, 1, 1)\n\n        # Mouth mask — always starts at 0 (disabled) on launch\n        grid.addWidget(QLabel(_("Mouth Mask")), 2, 0)\n        self.s_mouth = slider(0.0, 100.0, 0.0, 1,\n                              self._on_mouth_mask_change)\n        self.s_mouth.sliderPressed.connect(self._on_mouth_mask_pressed)\n        self.s_mouth.sliderReleased.connect(self._on_mouth_mask_released)\n        self.s_mouth.setToolTip(\n            _("0 = use swapped mouth, 100 = expose original mouth to chin area")\n        )\n        grid.addWidget(self.s_mouth, 2, 1)\n        return card\n\n    # ── action row ───────────────────────────────────────────────────────\n\n    def _build_action_row(self) -> QHBoxLayout:\n        row = QHBoxLayout()\n        self.btn_start = QPushButton(_("Start"))\n        self.btn_start.setToolTip(_("Begin processing the target image/video with selected face"))\n        self.btn_start.clicked.connect(self._on_start)\n\n        self.btn_destroy = QPushButton(_("Destroy"))\n        self.btn_destroy.setObjectName("danger")\n        self.btn_destroy.setToolTip(_("Stop processing and close the application"))\n        self.btn_destroy.clicked.connect(lambda: self._destroy_cb())\n\n        self.btn_preview = QPushButton(_("Preview"))\n        self.btn_preview.setObjectName("secondary")\n        self.btn_preview.setToolTip(_("Show/hide a preview of the processed output"))\n        self.btn_preview.clicked.connect(self._on_toggle_preview)\n\n        row.addWidget(self.btn_start)\n        row.addWidget(self.btn_destroy)\n        row.addWidget(self.btn_preview)\n        return row\n\n    # ── camera card ──────────────────────────────────────────────────────\n\n    def _build_camera_card(self) -> QGroupBox:\n        card = QGroupBox(_("Camera"))\n        layout = QHBoxLayout(card)\n\n        layout.addWidget(QLabel(_("Select Camera:")))\n        self._camera_indices, self._camera_names = get_available_cameras()\n\n        self.cb_camera = QComboBox()\n        if not self._camera_names or self._camera_names[0] == "No cameras found":\n            self.cb_camera.addItem("No cameras found")\n            self.cb_camera.setEnabled(False)\n            cam_ok = False\n        else:\n            self.cb_camera.addItems(self._camera_names)\n            cam_ok = True\n        self.cb_camera.setToolTip(_("Select which camera to use for live mode"))\n        layout.addWidget(self.cb_camera, 1)\n\n        self.btn_live = QPushButton(_("Live"))\n        self.btn_live.setEnabled(cam_ok)\n        self.btn_live.setToolTip(_("Start real-time face swap using webcam"))\n        self.btn_live.clicked.connect(self._on_live)\n        layout.addWidget(self.btn_live)\n\n        return card\n\n    # ── slot handlers ────────────────────────────────────────────────────\n\n    def set_status(self, text: str) -> None:\n        self._status_label.setText(text)\n\n    def _on_select_source(self) -> None:\n        global _RECENT_SOURCE_DIR\n        if _PREVIEW is not None:\n            _PREVIEW.hide()\n        path, _filter = QFileDialog.getOpenFileName(\n            self, _("select an source image"),\n            _RECENT_SOURCE_DIR or "",\n            "Images (*.png *.jpg *.jpeg *.gif *.bmp)",\n        )\n        if path and is_image(path):\n            modules.globals.source_path = path\n            _RECENT_SOURCE_DIR = os.path.dirname(path)\n            self.source_label.setPixmap(render_image_preview(path, (200, 200)))\n            self.source_label.setText("")\n        elif not path:\n            return\n        else:\n            modules.globals.source_path = None\n            self.source_label.clear()\n            self.source_label.setText(_("Source face"))\n\n    def _on_select_target(self) -> None:\n        global _RECENT_TARGET_DIR\n        if _PREVIEW is not None:\n            _PREVIEW.hide()\n        path, _filter = QFileDialog.getOpenFileName(\n            self, _("select an target image or video"),\n            _RECENT_TARGET_DIR or "",\n            "Media (*.png *.jpg *.jpeg *.gif *.bmp *.mp4 *.mkv)",\n        )\n        if not path:\n            return\n        if is_image(path):\n            modules.globals.target_path = path\n            _RECENT_TARGET_DIR = os.path.dirname(path)\n            self.target_label.setPixmap(render_image_preview(path, (200, 200)))\n            self.target_label.setText("")\n        elif is_video(path):\n            modules.globals.target_path = path\n            _RECENT_TARGET_DIR = os.path.dirname(path)\n            pm = render_video_preview(path, (200, 200))\n            if pm:\n                self.target_label.setPixmap(pm)\n                self.target_label.setText("")\n        else:\n            modules.globals.target_path = None\n            self.target_label.clear()\n            self.target_label.setText(_("Target"))\n\n    def _on_random_face(self) -> None:\n        if _PREVIEW is not None:\n            _PREVIEW.hide()\n        try:\n            response = requests.get(\n                "https://thispersondoesnotexist.com/",\n                headers={"User-Agent": "Mozilla/5.0"},\n                timeout=10,\n            )\n            response.raise_for_status()\n            temp_path = os.path.join(tempfile.gettempdir(), "deep_live_cam_random_face.jpg")\n            with open(temp_path, "wb") as f:\n                f.write(response.content)\n            modules.globals.source_path = temp_path\n            self.source_label.setPixmap(render_image_preview(temp_path, (200, 200)))\n            self.source_label.setText("")\n        except Exception as exc:\n            print(f"Failed to fetch random face: {exc}")\n\n    def _on_swap_paths(self) -> None:\n        global _RECENT_SOURCE_DIR, _RECENT_TARGET_DIR\n        sp = modules.globals.source_path\n        tp = modules.globals.target_path\n        if not (sp and tp and is_image(sp) and is_image(tp)):\n            return\n        modules.globals.source_path, modules.globals.target_path = tp, sp\n        _RECENT_SOURCE_DIR = os.path.dirname(tp)\n        _RECENT_TARGET_DIR = os.path.dirname(sp)\n        if _PREVIEW is not None:\n            _PREVIEW.hide()\n        self.source_label.setPixmap(render_image_preview(tp, (200, 200)))\n        self.target_label.setPixmap(render_image_preview(sp, (200, 200)))\n        self.source_label.setText("")\n        self.target_label.setText("")\n\n    def _on_map_faces_toggled(self, value: bool) -> None:\n        modules.globals.map_faces = value\n        save_switch_states()\n        if not value:\n            close_mapper_window()\n\n    def _on_enhancer_change(self, choice: str) -> None:\n        key_map = {\n            "None": None,\n            "GFPGAN": "face_enhancer",\n            "GPEN-512": "face_enhancer_gpen512",\n            "GPEN-256": "face_enhancer_gpen256",\n        }\n        for key in ("face_enhancer", "face_enhancer_gpen256", "face_enhancer_gpen512"):\n            _update_tumbler(key, False)\n        selected = key_map.get(choice)\n        if selected:\n            _update_tumbler(selected, True)\n        save_switch_states()\n\n    def _on_transparency_change(self, value: float) -> None:\n        modules.globals.opacity = value\n        pct = int(value * 100)\n        if pct == 0:\n            modules.globals.fp_ui["face_enhancer"] = False\n            update_status("Transparency set to 0% - Face swapping disabled.")\n        elif pct == 100:\n            modules.globals.face_swapper_enabled = True\n            update_status("Transparency set to 100%.")\n        else:\n            modules.globals.face_swapper_enabled = True\n            update_status(f"Transparency set to {pct}%")\n\n    def _on_sharpness_change(self, value: float) -> None:\n        modules.globals.sharpness = value\n        update_status(f"Sharpness set to {value:.1f}")\n\n    def _on_mouth_mask_change(self, value: float) -> None:\n        modules.globals.mouth_mask_size = value\n        modules.globals.mouth_mask = value > 0\n        if value <= 0:\n            modules.globals.show_mouth_mask_box = False\n\n    def _on_mouth_mask_pressed(self) -> None:\n        if modules.globals.mouth_mask_size > 0:\n            modules.globals.show_mouth_mask_box = True\n\n    def _on_mouth_mask_released(self) -> None:\n        modules.globals.show_mouth_mask_box = False\n\n    def _on_start(self) -> None:\n        if _MAPPER is not None and _MAPPER.isVisible():\n            update_status("Please complete pop-up or close it.")\n            return\n        if modules.globals.map_faces:\n            modules.globals.source_target_map = []\n            if is_image(modules.globals.target_path):\n                update_status("Getting unique faces")\n                get_unique_faces_from_target_image()\n            elif is_video(modules.globals.target_path):\n                update_status("Getting unique faces")\n                get_unique_faces_from_target_video()\n            if modules.globals.source_target_map:\n                _open_mapper_dialog(self._start_cb, modules.globals.source_target_map)\n            else:\n                update_status("No faces found in target")\n        else:\n            self._select_output_and_start()\n\n    def _select_output_and_start(self) -> None:\n        global _RECENT_OUTPUT_DIR\n        if is_image(modules.globals.target_path):\n            path, _f = QFileDialog.getSaveFileName(\n                self, _("save image output file"),\n                os.path.join(_RECENT_OUTPUT_DIR or "", "output.png"),\n                "Images (*.png *.jpg *.jpeg *.bmp)",\n            )\n        elif is_video(modules.globals.target_path):\n            path, _f = QFileDialog.getSaveFileName(\n                self, _("save video output file"),\n                os.path.join(_RECENT_OUTPUT_DIR or "", "output.mp4"),\n                "Videos (*.mp4 *.mkv)",\n            )\n        else:\n            return\n        if path:\n            modules.globals.output_path = path\n            _RECENT_OUTPUT_DIR = os.path.dirname(path)\n            self._start_cb()\n\n    def _on_toggle_preview(self) -> None:\n        if _PREVIEW is None:\n            return\n        if _PREVIEW.isVisible():\n            _PREVIEW.hide()\n        elif modules.globals.source_path and modules.globals.target_path:\n            _PREVIEW.init_for_target()\n            _PREVIEW.refresh_frame(0)\n            _PREVIEW.show()\n\n    def _on_live(self) -> None:\n        idx = self.cb_camera.currentIndex()\n        if idx < 0 or idx >= len(self._camera_indices):\n            update_status("No camera available")\n            return\n        camera_index = self._camera_indices[idx]\n        if _LIVE_MAPPER is not None and _LIVE_MAPPER.isVisible():\n            update_status("Source x Target Mapper is already open.")\n            _LIVE_MAPPER.raise_()\n            return\n        if not modules.globals.map_faces:\n            if modules.globals.source_path is None:\n                update_status("Please select a source image first")\n                return\n            from modules.face_analyser import get_face_analyser\n            from modules.processors.frame.face_swapper import get_face_swapper\n            get_face_analyser()\n            get_face_swapper()\n            _open_webcam_preview(camera_index)\n        else:\n            modules.globals.source_target_map = []\n            _open_live_mapper_dialog(camera_index, modules.globals.source_target_map)\n\n    def closeEvent(self, event):\n        # Treat OS-level close as Destroy click\n        self._destroy_cb()\n        event.accept()\n\n\ndef _update_tumbler(var: str, value: bool) -> None:\n    modules.globals.fp_ui[var] = value\n    save_switch_states()\n    # If we\'re currently in a live preview, refresh frame processors so\n    # toggling enhancers takes effect immediately.\n    if _WEBCAM_PREVIEW is not None and _WEBCAM_PREVIEW.isVisible():\n        get_frame_processors_modules(modules.globals.frame_processors)\n\n\n# ─── preview window (still-image / video scrub) ──────────────────────────\n\n\nclass PreviewWindow(QWidget):\n    def __init__(self):\n        super().__init__()\n        self.setWindowTitle(_("Preview"))\n        self.resize(PREVIEW_DEFAULT_WIDTH, PREVIEW_DEFAULT_HEIGHT)\n        layout = QVBoxLayout(self)\n        layout.setContentsMargins(0, 0, 0, 0)\n\n        self._image_label = QLabel()\n        self._image_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n        self._image_label.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)\n        layout.addWidget(self._image_label, 1)\n\n        self._slider = QSlider(Qt.Orientation.Horizontal)\n        self._slider.setRange(0, 0)\n        self._slider.valueChanged.connect(self.refresh_frame)\n        layout.addWidget(self._slider)\n\n    def init_for_target(self) -> None:\n        if is_image(modules.globals.target_path):\n            self._slider.hide()\n        elif is_video(modules.globals.target_path):\n            total = get_video_frame_total(modules.globals.target_path)\n            self._slider.setRange(0, max(0, total - 1))\n            self._slider.setValue(0)\n            self._slider.show()\n\n    def refresh_frame(self, frame_number: int = 0) -> None:\n        if not (modules.globals.source_path and modules.globals.target_path):\n            return\n        update_status("Processing...")\n        temp_frame = get_video_frame(modules.globals.target_path, frame_number)\n        if modules.globals.nsfw_filter and check_and_ignore_nsfw(temp_frame):\n            return\n        from modules.processors.frame.core import get_frame_processors_modules as _gfpm\n        for fp in _gfpm(modules.globals.frame_processors):\n            temp_frame = fp.process_frame(\n                get_one_face(imread_unicode(modules.globals.source_path)), temp_frame\n            )\n        # Fit to current widget size while preserving aspect ratio.\n        h, w = temp_frame.shape[:2]\n        bound_w = min(PREVIEW_MAX_WIDTH, max(self.width(), PREVIEW_DEFAULT_WIDTH))\n        bound_h = min(PREVIEW_MAX_HEIGHT, max(self.height(), PREVIEW_DEFAULT_HEIGHT))\n        ratio = min(bound_w / w, bound_h / h)\n        new_size = (max(1, int(w * ratio)), max(1, int(h * ratio)))\n        temp_frame = cv2.resize(temp_frame, new_size, interpolation=cv2.INTER_LANCZOS4)\n        self._image_label.setPixmap(_bgr_to_qpixmap(temp_frame))\n        update_status("Processing succeed!")\n\n\n# ─── webcam preview window ───────────────────────────────────────────────\n\n\nclass _CaptureWorker(QThread):\n    """Reads frames from the camera into a bounded queue. Drops on overflow."""\n\n    def __init__(self, cap, capture_queue: queue.Queue, stop_event: threading.Event):\n        super().__init__()\n        self._cap = cap\n        self._queue = capture_queue\n        self._stop = stop_event\n\n    def run(self) -> None:\n        while not self._stop.is_set():\n            ret, frame = self._cap.read()\n            if not ret:\n                self._stop.set()\n                break\n            try:\n                self._queue.put_nowait(frame)\n            except queue.Full:\n                try:\n                    self._queue.get_nowait()\n                except queue.Empty:\n                    pass\n                try:\n                    self._queue.put_nowait(frame)\n                except queue.Full:\n                    pass\n\n\nclass _ProcessingWorker(QThread):\n    """Pulls raw frames, runs detect/swap/enhance, pushes processed frames."""\n\n    def __init__(self, capture_queue, processed_queue, stop_event, camera_fps: float):\n        super().__init__()\n        self._cq = capture_queue\n        self._pq = processed_queue\n        self._stop = stop_event\n        self._fps = camera_fps\n\n    def run(self) -> None:\n        frame_processors = get_frame_processors_modules(modules.globals.frame_processors)\n        source_image = None\n        last_source_path = None\n        prev_time = time.time()\n        fps_update_interval = 0.5\n        frame_count = 0\n        fps = 0.0\n        det_count = 0\n        cached_target_face = None\n        cached_many_faces = None\n        det_interval = max(1, round(self._fps * 0.08))\n\n        while not self._stop.is_set():\n            try:\n                frame = self._cq.get(timeout=0.05)\n            except queue.Empty:\n                continue\n\n            temp_frame = frame\n            if modules.globals.live_mirror:\n                temp_frame = gpu_flip(temp_frame, 1)\n\n            if not modules.globals.map_faces:\n                if (\n                    modules.globals.source_path\n                    and modules.globals.source_path != last_source_path\n                ):\n                    last_source_path = modules.globals.source_path\n                    source_image = get_one_face(imread_unicode(modules.globals.source_path))\n\n                det_count += 1\n                if det_count % det_interval == 0:\n                    if modules.globals.many_faces:\n                        cached_target_face = None\n                        cached_many_faces = detect_many_faces_fast(temp_frame)\n                    else:\n                        cached_target_face = detect_one_face_fast(temp_frame)\n                        cached_many_faces = None\n\n                cached_faces = None\n                if cached_many_faces:\n                    cached_faces = cached_many_faces\n                elif cached_target_face is not None:\n                    cached_faces = [cached_target_face]\n\n                # Fast detection skips the 2d106 landmark model, but the mouth\n                # mask needs it. Attach landmarks on demand (computed once per\n                # detection cycle — the helper no-ops if already present).\n                if modules.globals.mouth_mask and cached_faces:\n                    ensure_landmarks(temp_frame, cached_faces)\n\n                for fp in frame_processors:\n                    if fp.NAME == "DLC.FACE-ENHANCER":\n                        if modules.globals.fp_ui["face_enhancer"]:\n                            temp_frame = fp.process_frame(\n                                None, temp_frame, detected_faces=cached_faces\n                            )\n                    elif fp.NAME == "DLC.FACE-ENHANCER-GPEN256":\n                        if modules.globals.fp_ui.get("face_enhancer_gpen256", False):\n                            temp_frame = fp.process_frame(\n                                None, temp_frame, detected_faces=cached_faces\n                            )\n                    elif fp.NAME == "DLC.FACE-ENHANCER-GPEN512":\n                        if modules.globals.fp_ui.get("face_enhancer_gpen512", False):\n                            temp_frame = fp.process_frame(\n                                None, temp_frame, detected_faces=cached_faces\n                            )\n                    elif fp.NAME == "DLC.FACE-SWAPPER":\n                        swapped_bboxes = []\n                        if modules.globals.many_faces and cached_many_faces:\n                            result = temp_frame.copy()\n                            for t_face in cached_many_faces:\n                                result = fp.swap_face(source_image, t_face, result)\n                                if hasattr(t_face, "bbox") and t_face.bbox is not None:\n                                    swapped_bboxes.append(t_face.bbox.astype(int))\n                            temp_frame = result\n                        elif cached_target_face is not None:\n                            temp_frame = fp.swap_face(\n                                source_image, cached_target_face, temp_frame\n                            )\n                            if (\n                                hasattr(cached_target_face, "bbox")\n                                and cached_target_face.bbox is not None\n                            ):\n                                swapped_bboxes.append(cached_target_face.bbox.astype(int))\n                        temp_frame = fp.apply_post_processing(temp_frame, swapped_bboxes)\n                    else:\n                        temp_frame = fp.process_frame(source_image, temp_frame)\n            else:\n                modules.globals.target_path = None\n                for fp in frame_processors:\n                    if fp.NAME == "DLC.FACE-ENHANCER":\n                        if modules.globals.fp_ui["face_enhancer"]:\n                            temp_frame = fp.process_frame_v2(temp_frame)\n                    elif fp.NAME in ("DLC.FACE-ENHANCER-GPEN256", "DLC.FACE-ENHANCER-GPEN512"):\n                        fp_key = fp.NAME.split(".")[-1].lower().replace("-", "_")\n                        if modules.globals.fp_ui.get(fp_key, False):\n                            temp_frame = fp.process_frame_v2(temp_frame)\n                    else:\n                        temp_frame = fp.process_frame_v2(temp_frame)\n\n            current_time = time.time()\n            frame_count += 1\n            if current_time - prev_time >= fps_update_interval:\n                fps = frame_count / (current_time - prev_time)\n                frame_count = 0\n                prev_time = current_time\n\n            if modules.globals.show_fps:\n                cv2.putText(\n                    temp_frame, f"FPS: {fps:.1f}", (10, 30),\n                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2,\n                )\n\n            try:\n                self._pq.put_nowait(temp_frame)\n            except queue.Full:\n                try:\n                    self._pq.get_nowait()\n                except queue.Empty:\n                    pass\n                try:\n                    self._pq.put_nowait(temp_frame)\n                except queue.Full:\n                    pass\n\n\nclass WebcamPreviewWindow(QWidget):\n    def __init__(self, camera_index: int):\n        super().__init__()\n        self.setWindowTitle("Live Preview")\n        self.resize(PREVIEW_DEFAULT_WIDTH, PREVIEW_DEFAULT_HEIGHT)\n        layout = QVBoxLayout(self)\n        layout.setContentsMargins(0, 0, 0, 0)\n        self._image_label = QLabel()\n        self._image_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n        self._image_label.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)\n        layout.addWidget(self._image_label, 1)\n\n        self._cap = VideoCapturer(camera_index)\n        if not self._cap.start(PREVIEW_DEFAULT_WIDTH, PREVIEW_DEFAULT_HEIGHT, 60):\n            update_status("Failed to start camera")\n            QTimer.singleShot(0, self.close)\n            return\n\n        camera_fps = self._cap.actual_fps\n        print(\n            f"[webcam] Camera running at {self._cap.actual_width}x"\n            f"{self._cap.actual_height}@{camera_fps:.0f}fps"\n        )\n\n        self._capture_queue: queue.Queue = queue.Queue(maxsize=2)\n        self._processed_queue: queue.Queue = queue.Queue(maxsize=2)\n        self._stop_event = threading.Event()\n\n        self._capture_worker = _CaptureWorker(\n            self._cap, self._capture_queue, self._stop_event\n        )\n        self._processing_worker = _ProcessingWorker(\n            self._capture_queue, self._processed_queue, self._stop_event, camera_fps\n        )\n        self._capture_worker.start()\n        self._processing_worker.start()\n\n        # Poll at ~2x camera fps so we never block but also don\'t burn CPU.\n        poll_ms = max(1, min(16, int(500 / max(camera_fps, 1))))\n        self._timer = QTimer(self)\n        self._timer.timeout.connect(self._tick)\n        self._timer.start(poll_ms)\n\n    def _tick(self) -> None:\n        if self._stop_event.is_set():\n            self.close()\n            return\n        try:\n            bgr_frame = self._processed_queue.get_nowait()\n        except queue.Empty:\n            return\n        bgr_frame = fit_image_to_size(bgr_frame, self.width(), self.height())\n        self._image_label.setPixmap(_bgr_to_qpixmap(bgr_frame))\n\n    def closeEvent(self, event) -> None:\n        self._stop_event.set()\n        try:\n            self._timer.stop()\n        except Exception:\n            pass\n        for worker in (self._capture_worker, self._processing_worker):\n            try:\n                worker.wait(2000)\n            except Exception:\n                pass\n        try:\n            self._cap.release()\n        except Exception:\n            pass\n        global _WEBCAM_PREVIEW\n        if _WEBCAM_PREVIEW is self:\n            _WEBCAM_PREVIEW = None\n        event.accept()\n\n\ndef _open_webcam_preview(camera_index: int) -> None:\n    global _WEBCAM_PREVIEW\n    if _WEBCAM_PREVIEW is not None:\n        _WEBCAM_PREVIEW.close()\n    _WEBCAM_PREVIEW = WebcamPreviewWindow(camera_index)\n    _WEBCAM_PREVIEW.show()\n\n\n# ─── mapper dialogs (image/video + live) ────────────────────────────────\n\n\ndef _make_thumb(cv2_img: np.ndarray) -> QPixmap:\n    rgb = gpu_cvt_color(cv2_img, cv2.COLOR_BGR2RGB)\n    image = Image.fromarray(rgb).resize(\n        (MAPPER_PREVIEW_SIZE, MAPPER_PREVIEW_SIZE), Image.LANCZOS\n    )\n    return _pil_to_qpixmap(image)\n\n\nclass MapperDialog(QDialog):\n    """Source × Target mapper for image / video processing."""\n\n    def __init__(self, start_cb: Callable, mapping: list):\n        super().__init__(_MAIN)\n        self._start_cb = start_cb\n        self._map = mapping\n        self.setWindowTitle(_("Source x Target Mapper"))\n        self.resize(POPUP_WIDTH, POPUP_HEIGHT)\n        layout = QVBoxLayout(self)\n\n        self._scroll = QScrollArea()\n        self._scroll.setWidgetResizable(True)\n        layout.addWidget(self._scroll, 1)\n\n        self._status = QLabel("")\n        self._status.setAlignment(Qt.AlignmentFlag.AlignCenter)\n        layout.addWidget(self._status)\n\n        btn_submit = QPushButton(_("Submit"))\n        btn_submit.clicked.connect(self._on_submit)\n        layout.addWidget(btn_submit, alignment=Qt.AlignmentFlag.AlignCenter)\n\n        self._rebuild()\n\n    def set_status(self, text: str) -> None:\n        self._status.setText(_(text))\n\n    def _rebuild(self) -> None:\n        body = QWidget()\n        grid = QGridLayout(body)\n        grid.setHorizontalSpacing(10)\n        grid.setVerticalSpacing(10)\n        for item in self._map:\n            row = item["id"]\n            btn = QPushButton(_("Select source image"))\n            btn.setFixedWidth(200)\n            btn.clicked.connect(lambda _c, n=row: self._select_source(n))\n            grid.addWidget(btn, row, 0)\n\n            src_label = QLabel(f"S-{row}")\n            src_label.setFixedSize(MAPPER_PREVIEW_SIZE, MAPPER_PREVIEW_SIZE)\n            src_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n            src_label.setStyleSheet("border: 1px dashed #555;")\n            grid.addWidget(src_label, row, 1)\n            if "source" in item:\n                src_label.setPixmap(_make_thumb(item["source"]["cv2"]))\n                src_label.setText("")\n\n            x_label = QLabel("×")\n            x_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n            grid.addWidget(x_label, row, 2)\n\n            tgt_label = QLabel(f"T-{row}")\n            tgt_label.setFixedSize(MAPPER_PREVIEW_SIZE, MAPPER_PREVIEW_SIZE)\n            tgt_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n            tgt_label.setStyleSheet("border: 1px solid #555;")\n            grid.addWidget(tgt_label, row, 3)\n            if "target" in item:\n                tgt_label.setPixmap(_make_thumb(item["target"]["cv2"]))\n                tgt_label.setText("")\n\n        grid.setRowStretch(grid.rowCount(), 1)\n        self._scroll.setWidget(body)\n\n    def _select_source(self, row: int) -> None:\n        path, _f = QFileDialog.getOpenFileName(\n            self, _("select an source image"),\n            _RECENT_SOURCE_DIR or "",\n            "Images (*.png *.jpg *.jpeg *.gif *.bmp)",\n        )\n        if not path:\n            return\n        cv2_img = imread_unicode(path)\n        face = get_one_face(cv2_img)\n        if face is None:\n            self.set_status("Face could not be detected in last upload!")\n            return\n        x_min, y_min, x_max, y_max = face["bbox"]\n        self._map[row]["source"] = {\n            "cv2": cv2_img[int(y_min):int(y_max), int(x_min):int(x_max)],\n            "face": face,\n        }\n        self._rebuild()\n\n    def _on_submit(self) -> None:\n        if has_valid_map():\n            self.accept()\n            _MAIN._select_output_and_start()\n        else:\n            self.set_status("Atleast 1 source with target is required!")\n\n\nclass LiveMapperDialog(QDialog):\n    """Source × Target mapper for live webcam mode."""\n\n    def __init__(self, camera_index: int, mapping: list):\n        super().__init__(_MAIN)\n        self._camera_index = camera_index\n        self._map = mapping\n        self.setWindowTitle(_("Source x Target Mapper"))\n        self.resize(POPUP_LIVE_WIDTH, POPUP_LIVE_HEIGHT)\n        layout = QVBoxLayout(self)\n\n        self._scroll = QScrollArea()\n        self._scroll.setWidgetResizable(True)\n        layout.addWidget(self._scroll, 1)\n\n        self._status = QLabel("")\n        self._status.setAlignment(Qt.AlignmentFlag.AlignCenter)\n        layout.addWidget(self._status)\n\n        btn_row = QHBoxLayout()\n        for text, slot in (\n            (_("Add"), self._on_add),\n            (_("Clear"), self._on_clear),\n            (_("Submit"), self._on_submit),\n        ):\n            b = QPushButton(text)\n            b.clicked.connect(slot)\n            btn_row.addWidget(b)\n        layout.addLayout(btn_row)\n\n        self._rebuild()\n\n    def set_status(self, text: str) -> None:\n        self._status.setText(_(text))\n\n    def _rebuild(self) -> None:\n        body = QWidget()\n        grid = QGridLayout(body)\n        grid.setHorizontalSpacing(10)\n        grid.setVerticalSpacing(10)\n        for item in self._map:\n            row = item["id"]\n            btn_s = QPushButton(_("Select source image"))\n            btn_s.setFixedWidth(200)\n            btn_s.clicked.connect(lambda _c, n=row: self._select_face(n, "source"))\n            grid.addWidget(btn_s, row, 0)\n\n            src_label = QLabel(f"S-{row}")\n            src_label.setFixedSize(MAPPER_PREVIEW_SIZE, MAPPER_PREVIEW_SIZE)\n            src_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n            src_label.setStyleSheet("border: 1px dashed #555;")\n            grid.addWidget(src_label, row, 1)\n            if "source" in item:\n                src_label.setPixmap(_make_thumb(item["source"]["cv2"]))\n                src_label.setText("")\n\n            x_label = QLabel("×")\n            x_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n            grid.addWidget(x_label, row, 2)\n\n            btn_t = QPushButton(_("Select target image"))\n            btn_t.setFixedWidth(200)\n            btn_t.clicked.connect(lambda _c, n=row: self._select_face(n, "target"))\n            grid.addWidget(btn_t, row, 3)\n\n            tgt_label = QLabel(f"T-{row}")\n            tgt_label.setFixedSize(MAPPER_PREVIEW_SIZE, MAPPER_PREVIEW_SIZE)\n            tgt_label.setAlignment(Qt.AlignmentFlag.AlignCenter)\n            tgt_label.setStyleSheet("border: 1px dashed #555;")\n            grid.addWidget(tgt_label, row, 4)\n            if "target" in item:\n                tgt_label.setPixmap(_make_thumb(item["target"]["cv2"]))\n                tgt_label.setText("")\n\n        grid.setRowStretch(grid.rowCount(), 1)\n        self._scroll.setWidget(body)\n\n    def _select_face(self, row: int, kind: str) -> None:\n        path, _f = QFileDialog.getOpenFileName(\n            self, _("select an source image"),\n            _RECENT_SOURCE_DIR or "",\n            "Images (*.png *.jpg *.jpeg *.gif *.bmp)",\n        )\n        if not path:\n            return\n        cv2_img = imread_unicode(path)\n        face = get_one_face(cv2_img)\n        if face is None:\n            self.set_status("Face could not be detected in last upload!")\n            return\n        x_min, y_min, x_max, y_max = face["bbox"]\n        self._map[row][kind] = {\n            "cv2": cv2_img[int(y_min):int(y_max), int(x_min):int(x_max)],\n            "face": face,\n        }\n        self._rebuild()\n\n    def _on_add(self) -> None:\n        add_blank_map()\n        self._rebuild()\n        self.set_status("Please provide mapping!")\n\n    def _on_clear(self) -> None:\n        for item in self._map:\n            item.pop("source", None)\n            item.pop("target", None)\n        self._rebuild()\n        self.set_status("All mappings cleared!")\n\n    def _on_submit(self) -> None:\n        if has_valid_map():\n            simplify_maps()\n            self.set_status("Mappings successfully submitted!")\n            self.accept()\n            _open_webcam_preview(self._camera_index)\n        else:\n            self.set_status("At least 1 source with target is required!")\n\n\ndef _open_mapper_dialog(start_cb: Callable, mapping: list) -> None:\n    global _MAPPER\n    close_mapper_window()\n    _MAPPER = MapperDialog(start_cb, mapping)\n    _MAPPER.show()\n\n\ndef _open_live_mapper_dialog(camera_index: int, mapping: list) -> None:\n    global _LIVE_MAPPER\n    close_mapper_window()\n    _LIVE_MAPPER = LiveMapperDialog(camera_index, mapping)\n    _LIVE_MAPPER.show()\n\n\ndef close_mapper_window() -> None:\n    global _MAPPER, _LIVE_MAPPER\n    if _MAPPER is not None:\n        _MAPPER.close()\n        _MAPPER = None\n    if _LIVE_MAPPER is not None:\n        _LIVE_MAPPER.close()\n        _LIVE_MAPPER = None\n\n\n# ─── entry point ─────────────────────────────────────────────────────────\n\n\nclass _Window:\n    """Thin wrapper exposing .mainloop() for core.py compatibility."""\n\n    def __init__(self, app: QApplication, main_window: MainWindow):\n        self._app = app\n        self._main = main_window\n\n    def mainloop(self) -> None:\n        self._main.show()\n        self._app.exec()\n\n\ndef init(\n    start: Callable[[], None], destroy: Callable[[], None], lang: str\n) -> _Window:\n    global _APP, _MAIN, _PREVIEW, _LANG, _BRIDGE\n\n    _LANG = LanguageManager(lang)\n    if QApplication.instance() is None:\n        _APP = QApplication(sys.argv)\n    else:\n        _APP = QApplication.instance()\n    _APP.setStyleSheet(QSS)\n\n    _BRIDGE = _UIBridge()\n    _MAIN = MainWindow(start, destroy)\n    _PREVIEW = PreviewWindow()\n\n    # Route status updates onto the UI thread regardless of caller.\n    _BRIDGE.statusChanged.connect(_MAIN.set_status)\n\n    return _Window(_APP, _MAIN)\n',
    'modules/ui_tooltip.py': '"""Lightweight hover tooltip for CustomTkinter widgets."""\n\nimport customtkinter as ctk\n\n\nclass ToolTip:\n    """Show a floating tooltip popup when the user hovers over a widget.\n\n    Usage:\n        ToolTip(my_button, "Helpful description text")\n    """\n\n    def __init__(self, widget: ctk.CTkBaseClass, text: str, delay: int = 500):\n        self._widget = widget\n        self._text = text\n        self._delay = delay\n        self._tooltip_window = None\n        self._after_id = None\n\n        widget.bind("<Enter>", self._schedule_show, add="+")\n        widget.bind("<Leave>", self._hide, add="+")\n\n    def _schedule_show(self, event=None):\n        self._cancel()\n        self._after_id = self._widget.after(self._delay, self._show)\n\n    def _show(self):\n        if self._tooltip_window is not None:\n            return\n\n        x = self._widget.winfo_rootx() + 20\n        y = self._widget.winfo_rooty() + self._widget.winfo_height() + 5\n\n        self._tooltip_window = tw = ctk.CTkToplevel(self._widget)\n        tw.withdraw()\n        tw.overrideredirect(True)\n\n        label = ctk.CTkLabel(\n            tw,\n            text=self._text,\n            fg_color="#333333",\n            text_color="#EEEEEE",\n            corner_radius=6,\n            padx=8,\n            pady=4,\n        )\n        label.pack()\n\n        tw.update_idletasks()\n\n        # Clamp to screen bounds\n        screen_w = tw.winfo_screenwidth()\n        screen_h = tw.winfo_screenheight()\n        tip_w = tw.winfo_reqwidth()\n        tip_h = tw.winfo_reqheight()\n\n        if x + tip_w > screen_w:\n            x = screen_w - tip_w - 5\n        if y + tip_h > screen_h:\n            y = self._widget.winfo_rooty() - tip_h - 5\n\n        tw.geometry(f"+{x}+{y}")\n        tw.deiconify()\n\n    def _hide(self, event=None):\n        self._cancel()\n        if self._tooltip_window is not None:\n            self._tooltip_window.destroy()\n            self._tooltip_window = None\n\n    def _cancel(self):\n        if self._after_id is not None:\n            self._widget.after_cancel(self._after_id)\n            self._after_id = None\n',
    'modules/utilities.py': 'import glob\nimport mimetypes\nimport os\nimport platform\nimport shutil\nimport ssl\nimport subprocess\nimport urllib\nfrom pathlib import Path\nfrom typing import List, Any\nfrom tqdm import tqdm\n\nimport modules.globals\n\nTEMP_FILE = "temp.mp4"\nTEMP_DIRECTORY = "temp"\n\n\ndef run_ffmpeg(args: List[str]) -> bool:\n    """Run ffmpeg with hardware acceleration and optimized settings."""\n    commands = [\n        "ffmpeg",\n        "-hide_banner",\n        "-hwaccel", "auto",  # Auto-detect hardware acceleration\n        "-hwaccel_output_format", "auto",  # Use hardware format when possible\n        "-threads", str(modules.globals.execution_threads or 0),  # 0 = auto-detect optimal thread count\n        "-loglevel", modules.globals.log_level,\n    ]\n    commands.extend(args)\n    try:\n        subprocess.check_output(commands, stderr=subprocess.STDOUT)\n        return True\n    except subprocess.CalledProcessError as error:\n        output = error.output.decode(errors="ignore").strip()\n        if output:\n            print(output)\n    except Exception as error:\n        print(f"ffmpeg execution failed: {error}")\n    return False\n\n\ndef detect_fps(target_path: str) -> float:\n    command = [\n        "ffprobe",\n        "-v",\n        "error",\n        "-select_streams",\n        "v:0",\n        "-show_entries",\n        "stream=r_frame_rate",\n        "-of",\n        "default=noprint_wrappers=1:nokey=1",\n        target_path,\n    ]\n    output = subprocess.check_output(command).decode().strip().split("/")\n    try:\n        numerator, denominator = map(int, output)\n        return numerator / denominator\n    except Exception:\n        pass\n    return 30.0\n\n\ndef extract_frames(target_path: str) -> None:\n    """Extract frames with hardware acceleration and optimized settings."""\n    temp_directory_path = get_temp_directory_path(target_path)\n    \n    # Write a contiguous image sequence so the later "%04d.png" input pattern\n    # used during encoding can consume every frame reliably.\n    run_ffmpeg(\n        [\n            "-i", target_path,\n            "-vf", "format=rgb24",  # Use video filter for format conversion (faster)\n            "-vsync", "0",  # Prevent frame duplication\n            os.path.join(temp_directory_path, "%04d.png"),\n        ]\n    )\n\n\ndef create_video(target_path: str, fps: float = 30.0) -> bool:\n    """Create video with hardware-accelerated encoding and optimized settings."""\n    temp_output_path = get_temp_output_path(target_path)\n    temp_directory_path = get_temp_directory_path(target_path)\n    \n    # Determine optimal encoder based on available hardware\n    encoder = modules.globals.video_encoder\n    encoder_options = []\n    \n    # GPU-accelerated encoding options\n    if \'CUDAExecutionProvider\' in modules.globals.execution_providers:\n        # NVIDIA GPU encoding\n        if encoder == \'libx264\':\n            encoder = \'h264_nvenc\'\n            encoder_options = [\n                "-preset", "p7",  # Highest quality preset for NVENC\n                "-tune", "hq",  # High quality tuning\n                "-rc", "vbr",  # Variable bitrate\n                "-cq", str(modules.globals.video_quality),  # Quality level\n                "-b:v", "0",  # Let CQ control bitrate\n                "-multipass", "fullres",  # Two-pass encoding for better quality\n            ]\n        elif encoder == \'libx265\':\n            encoder = \'hevc_nvenc\'\n            encoder_options = [\n                "-preset", "p7",\n                "-tune", "hq",\n                "-rc", "vbr",\n                "-cq", str(modules.globals.video_quality),\n                "-b:v", "0",\n            ]\n    elif \'DmlExecutionProvider\' in modules.globals.execution_providers:\n        # AMD/Intel GPU encoding (DirectML on Windows)\n        if encoder == \'libx264\':\n            # Try AMD AMF encoder\n            encoder = \'h264_amf\'\n            encoder_options = [\n                "-quality", "quality",  # Quality mode\n                "-rc", "vbr_latency",\n                "-qp_i", str(modules.globals.video_quality),\n                "-qp_p", str(modules.globals.video_quality),\n            ]\n        elif encoder == \'libx265\':\n            encoder = \'hevc_amf\'\n            encoder_options = [\n                "-quality", "quality",\n                "-rc", "vbr_latency",\n                "-qp_i", str(modules.globals.video_quality),\n                "-qp_p", str(modules.globals.video_quality),\n            ]\n    else:\n        # CPU encoding with optimized settings\n        if encoder == \'libx264\':\n            encoder_options = [\n                "-preset", "medium",  # Balance speed/quality\n                "-crf", str(modules.globals.video_quality),\n                "-tune", "film",  # Optimize for film content\n            ]\n        elif encoder == \'libx265\':\n            encoder_options = [\n                "-preset", "medium",\n                "-crf", str(modules.globals.video_quality),\n                "-x265-params", "log-level=error",\n            ]\n        elif encoder == \'libvpx-vp9\':\n            encoder_options = [\n                "-crf", str(modules.globals.video_quality),\n                "-b:v", "0",  # Constant quality mode\n                "-cpu-used", "2",  # Speed vs quality (0-5, lower=slower/better)\n            ]\n    \n    # Build ffmpeg command\n    ffmpeg_args = [\n        "-r", str(fps),\n        "-i", os.path.join(temp_directory_path, "%04d.png"),\n        "-c:v", encoder,\n    ]\n    \n    # Add encoder-specific options\n    ffmpeg_args.extend(encoder_options)\n    \n    # Add common options\n    ffmpeg_args.extend([\n        "-pix_fmt", "yuv420p",\n        "-movflags", "+faststart",  # Enable fast start for web playback\n        "-vf", "colorspace=bt709:iall=bt601-6-625:fast=1",\n        "-y",\n        temp_output_path,\n    ])\n    \n    # Try with hardware encoder first, fallback to software if it fails\n    success = run_ffmpeg(ffmpeg_args)\n    \n    if not success and encoder in [\'h264_nvenc\', \'hevc_nvenc\', \'h264_amf\', \'hevc_amf\']:\n        # Fallback to software encoding\n        print(f"Hardware encoding with {encoder} failed, falling back to software encoding...")\n        fallback_encoder = \'libx264\' if \'h264\' in encoder else \'libx265\'\n        ffmpeg_args_fallback = [\n            "-r", str(fps),\n            "-i", os.path.join(temp_directory_path, "%04d.png"),\n            "-c:v", fallback_encoder,\n            "-preset", "medium",\n            "-crf", str(modules.globals.video_quality),\n            "-pix_fmt", "yuv420p",\n            "-movflags", "+faststart",\n            "-vf", "colorspace=bt709:iall=bt601-6-625:fast=1",\n            "-y",\n            temp_output_path,\n        ]\n        success = run_ffmpeg(ffmpeg_args_fallback)\n    return success and os.path.isfile(temp_output_path)\n\n\ndef restore_audio(target_path: str, output_path: str) -> None:\n    temp_output_path = get_temp_output_path(target_path)\n    done = run_ffmpeg(\n        [\n            "-i",\n            temp_output_path,\n            "-i",\n            target_path,\n            "-c:v",\n            "copy",\n            "-map",\n            "0:v:0",\n            "-map",\n            "1:a:0",\n            "-y",\n            output_path,\n        ]\n    )\n    if not done:\n        move_temp(target_path, output_path)\n\n\ndef get_temp_frame_paths(target_path: str) -> List[str]:\n    temp_directory_path = get_temp_directory_path(target_path)\n    return glob.glob((os.path.join(glob.escape(temp_directory_path), "*.png")))\n\n\ndef get_temp_directory_path(target_path: str) -> str:\n    target_name, _ = os.path.splitext(os.path.basename(target_path))\n    target_directory_path = os.path.dirname(target_path)\n    return os.path.join(target_directory_path, TEMP_DIRECTORY, target_name)\n\n\ndef get_temp_output_path(target_path: str) -> str:\n    temp_directory_path = get_temp_directory_path(target_path)\n    return os.path.join(temp_directory_path, TEMP_FILE)\n\n\ndef normalize_output_path(source_path: str, target_path: str, output_path: str) -> Any:\n    if source_path and target_path:\n        source_name, _ = os.path.splitext(os.path.basename(source_path))\n        target_name, target_extension = os.path.splitext(os.path.basename(target_path))\n        if os.path.isdir(output_path):\n            return os.path.join(\n                output_path, source_name + "-" + target_name + target_extension\n            )\n    return output_path\n\n\ndef create_temp(target_path: str) -> None:\n    temp_directory_path = get_temp_directory_path(target_path)\n    Path(temp_directory_path).mkdir(parents=True, exist_ok=True)\n\n\ndef move_temp(target_path: str, output_path: str) -> None:\n    temp_output_path = get_temp_output_path(target_path)\n    if os.path.isfile(temp_output_path):\n        if os.path.isfile(output_path):\n            os.remove(output_path)\n        shutil.move(temp_output_path, output_path)\n\n\ndef clean_temp(target_path: str) -> None:\n    temp_directory_path = get_temp_directory_path(target_path)\n    parent_directory_path = os.path.dirname(temp_directory_path)\n    if not modules.globals.keep_frames and os.path.isdir(temp_directory_path):\n        shutil.rmtree(temp_directory_path)\n    if os.path.exists(parent_directory_path) and not os.listdir(parent_directory_path):\n        os.rmdir(parent_directory_path)\n\n\ndef has_image_extension(image_path: str) -> bool:\n    return image_path.lower().endswith(("png", "jpg", "jpeg"))\n\n\ndef is_image(image_path: str) -> bool:\n    if image_path and os.path.isfile(image_path):\n        mimetype, _ = mimetypes.guess_type(image_path)\n        return bool(mimetype and mimetype.startswith("image/"))\n    return False\n\n\ndef is_video(video_path: str) -> bool:\n    if video_path and os.path.isfile(video_path):\n        mimetype, _ = mimetypes.guess_type(video_path)\n        return bool(mimetype and mimetype.startswith("video/"))\n    return False\n\n\ndef conditional_download(download_directory_path: str, urls: List[str]) -> None:\n    if not os.path.exists(download_directory_path):\n        os.makedirs(download_directory_path)\n    for url in urls:\n        download_file_path = os.path.join(\n            download_directory_path, os.path.basename(url)\n        )\n        if not os.path.exists(download_file_path):\n            request = urllib.request.Request(url)\n            \n            # Create a specific SSL context for macOS to avoid globally disabling verification\n            ctx = None\n            if platform.system().lower() == "darwin":\n                ctx = ssl._create_unverified_context()\n                \n            response = urllib.request.urlopen(request, context=ctx)\n            total = int(response.headers.get("Content-Length", 0))\n            with tqdm(\n                total=total,\n                desc="Downloading",\n                unit="B",\n                unit_scale=True,\n                unit_divisor=1024,\n            ) as progress:\n                with open(download_file_path, "wb") as f:\n                    while True:\n                        buffer = response.read(8192)\n                        if not buffer:\n                            break\n                        f.write(buffer)\n                        progress.update(len(buffer))\n\n\ndef resolve_relative_path(path: str) -> str:\n    return os.path.abspath(os.path.join(os.path.dirname(__file__), path))\n\n\ndef get_video_dimensions(target_path: str) -> tuple:\n    """Get video width and height using ffprobe."""\n    command = [\n        "ffprobe", "-v", "error",\n        "-select_streams", "v:0",\n        "-show_entries", "stream=width,height",\n        "-of", "csv=p=0:s=x",\n        target_path,\n    ]\n    output = subprocess.check_output(command).decode().strip()\n    width, height = map(int, output.split("x"))\n    return width, height\n\n\ndef estimate_frame_count(target_path: str, fps: float = None) -> int:\n    """Estimate total frame count from video duration and fps."""\n    if fps is None:\n        fps = detect_fps(target_path)\n    command = [\n        "ffprobe", "-v", "error",\n        "-show_entries", "format=duration",\n        "-of", "csv=p=0",\n        target_path,\n    ]\n    try:\n        output = subprocess.check_output(command).decode().strip()\n        duration = float(output)\n        return int(duration * fps)\n    except Exception:\n        return 0\n',
    'modules/video_capture.py': 'import cv2\nimport numpy as np\nimport time\nfrom typing import Optional, Tuple, Callable\nimport platform\nimport threading\n\n# Only import Windows-specific library if on Windows\nif platform.system() == "Windows":\n    from pygrabber.dshow_graph import FilterGraph\n\n\nclass VideoCapturer:\n    def __init__(self, device_index: int):\n        self.device_index = device_index\n        self.frame_callback = None\n        self._current_frame = None\n        self._frame_ready = threading.Event()\n        self.is_running = False\n        self.cap = None\n        # Actual values reported by the camera after configuration\n        self.actual_width: int = 0\n        self.actual_height: int = 0\n        self.actual_fps: float = 0.0\n\n        # Initialize Windows-specific components if on Windows\n        if platform.system() == "Windows":\n            self.graph = FilterGraph()\n            # Verify device exists\n            devices = self.graph.get_input_devices()\n            if self.device_index >= len(devices):\n                raise ValueError(\n                    f"Invalid device index {device_index}. Available devices: {len(devices)}"\n                )\n\n    def start(self, width: int = 960, height: int = 540, fps: int = 60) -> bool:\n        """Initialize and start video capture"""\n        try:\n            if platform.system() == "Windows":\n                # device_index comes from pygrabber.FilterGraph (DirectShow\n                # enumeration), so open with DSHOW first to preserve mapping.\n                # MSMF and DirectShow enumerate cameras in different orders, so\n                # opening MSMF with a DSHOW index silently selects the wrong\n                # camera. MSMF/ANY remain as fallbacks for cameras DSHOW can\'t\n                # open.\n                #\n                # Pass codec + resolution + fps as construction params (OpenCV\n                # 4.6+). DSHOW locks the pixel format at open time and ignores\n                # later cap.set(CAP_PROP_FOURCC, ...) — without this, DSHOW\n                # falls back to uncompressed YUYV at 1080p, which is USB-\n                # bandwidth-limited to ~5 fps. Setting MJPG at construction\n                # negotiates compressed frames from the first read.\n                mjpg = cv2.VideoWriter_fourcc(*\'MJPG\')\n                open_params = [\n                    cv2.CAP_PROP_FOURCC, mjpg,\n                    cv2.CAP_PROP_FRAME_WIDTH, width,\n                    cv2.CAP_PROP_FRAME_HEIGHT, height,\n                    cv2.CAP_PROP_FPS, fps,\n                ]\n                capture_methods = [\n                    (self.device_index, cv2.CAP_DSHOW),\n                    (self.device_index, cv2.CAP_MSMF),\n                    (self.device_index, cv2.CAP_ANY),\n                ]\n\n                for dev_id, backend in capture_methods:\n                    try:\n                        self.cap = cv2.VideoCapture(dev_id, backend, open_params)\n                        if self.cap.isOpened():\n                            break\n                        self.cap.release()\n                    except Exception:\n                        continue\n            else:\n                # Unix-like systems (Linux/Mac) capture method\n                self.cap = cv2.VideoCapture(self.device_index)\n\n            if not self.cap or not self.cap.isOpened():\n                raise RuntimeError("Failed to open camera")\n\n            # Belt-and-braces: also set via cap.set() for backends that honor\n            # post-open changes (MSMF, V4L2). DSHOW ignores these, but the\n            # construction params above already handled it.\n            if platform.system() != "Windows":\n                self.cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*\'MJPG\'))\n                self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)\n                self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)\n                self.cap.set(cv2.CAP_PROP_FPS, fps)\n\n            # Read back resolution (usually reliable)\n            self.actual_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n            self.actual_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n\n            # CAP_PROP_FPS is unreliable on DirectShow — often reports 30\n            # even when the camera delivers 60.  Measure empirically by\n            # timing a burst of frames.\n            reported_fps = self.cap.get(cv2.CAP_PROP_FPS)\n            self.actual_fps = self._measure_fps(warmup=10, sample=30,\n                                                fallback=reported_fps or fps)\n\n            print(f"[VideoCapturer] {self.actual_width}x{self.actual_height} "\n                  f"@ {self.actual_fps:.1f}fps (reported={reported_fps:.0f})",\n                  flush=True)\n\n            self.is_running = True\n            return True\n\n        except Exception as e:\n            print(f"Failed to start capture: {str(e)}")\n            if self.cap:\n                self.cap.release()\n            return False\n\n    def read(self) -> Tuple[bool, Optional[np.ndarray]]:\n        """Read a frame from the camera"""\n        if not self.is_running or self.cap is None:\n            return False, None\n\n        ret, frame = self.cap.read()\n        if ret:\n            self._current_frame = frame\n            if self.frame_callback:\n                self.frame_callback(frame)\n            return True, frame\n        return False, None\n\n    def release(self) -> None:\n        """Stop capture and release resources"""\n        if self.is_running and self.cap is not None:\n            self.cap.release()\n            self.is_running = False\n            self.cap = None\n\n    def _measure_fps(self, warmup: int = 10, sample: int = 30,\n                     fallback: float = 30.0) -> float:\n        """Read warmup+sample frames and return measured FPS.\n\n        This is more reliable than CAP_PROP_FPS which often lies on\n        DirectShow.  Takes ~0.5-1s at startup but gives a ground-truth\n        number for adaptive polling/detection intervals.\n        """\n        try:\n            for _ in range(warmup):\n                self.cap.read()\n            t0 = time.perf_counter()\n            for _ in range(sample):\n                ret, _ = self.cap.read()\n                if not ret:\n                    return fallback\n            elapsed = time.perf_counter() - t0\n            if elapsed <= 0:\n                return fallback\n            return sample / elapsed\n        except Exception:\n            return fallback\n\n    def set_frame_callback(self, callback: Callable[[np.ndarray], None]) -> None:\n        """Set callback for frame processing"""\n        self.frame_callback = callback\n',
}


In [ ]:
# @title Install and initialize
# IPYNB_GENERATED_B64_FROM_CELL target=_RUNTIME_BUNDLE_B64 source_id=runtime-bundle codec=zlib
_RUNTIME_BUNDLE_B64 = "..."
import base64, os, subprocess, sys, zlib
from pathlib import Path
exec(zlib.decompress(base64.b64decode(_RUNTIME_BUNDLE_B64)).decode('utf-8'))
WORK_DIR = Path("/content/Deep-Live-Cam-Remote")
WORK_DIR.mkdir(parents=True, exist_ok=True)
for relative_path, source in _RUNTIME_FILES.items():
    target = WORK_DIR / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2", "opencv-python==4.10.0.84", "insightface==0.7.3", "onnx==1.18.0", "onnxruntime-gpu==1.23.2", "scikit-learn", "tqdm", "pillow", "psutil", "protobuf==4.25.1", "PySide6>=6.7,<7", "cv2_enumerate_cameras==1.1.15"], check=True)
os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))
subprocess.run(["nvidia-smi"], check=False)
print("Runtime ready:", WORK_DIR)


## 2. Configure Colab paths and processing options

In [ ]:
# @title Batch configuration
SOURCE_FACE = "/content/source_face.png"
INPUT_DIR = "/content/in"
OUTPUT_DIR = "/content/out"
ZIP_PATH = "/content/face_swapped_outputs.zip"
SS = 0.0
DURATION = None  # None processes the remainder
MAX_FPS = 30.0
MAX_WIDTH = 420
MANY_FACES = False
OPACITY = 1.0
SHARPNESS = 0.0
MOUTH_MASK_SIZE = 0.0
POISSON_BLEND = False
COLOR_CORRECTION = False
INTERPOLATION_WEIGHT = 0.0
ENHANCER = "none"  # none, gfpgan, gpen256, gpen512
MAPPING_JSON = None  # e.g. "/content/mapping/face_mapping.json"


## 3. Optional: scan identities and edit mapping JSON
Run this before processing only when different target identities need different source faces. Set each generated `source_path`, then set `MAPPING_JSON` above.

In [ ]:
# @title Scan identity gallery (optional)
from colab_batch import main
MAPPING_DIR = "/content/mapping"
main(["scan", "--input-dir", INPUT_DIR, "--mapping-dir", MAPPING_DIR])


## 4. Process folder and create ZIP

In [ ]:
# @title Run batch processor
from colab_batch import main
args = ["process", "--input-dir", INPUT_DIR, "--output-dir", OUTPUT_DIR, "--zip-output", ZIP_PATH, "--ss", str(SS), "--max-fps", str(MAX_FPS), "--max-width", str(MAX_WIDTH), "--opacity", str(OPACITY), "--sharpness", str(SHARPNESS), "--mouth-mask-size", str(MOUTH_MASK_SIZE), "--interpolation-weight", str(INTERPOLATION_WEIGHT), "--enhancer", ENHANCER]
if SOURCE_FACE: args += ["--source-face", SOURCE_FACE]
if DURATION is not None: args += ["--duration", str(DURATION)]
if MAPPING_JSON: args += ["--map-config", MAPPING_JSON]
if MANY_FACES: args += ["--many-faces"]
if POISSON_BLEND: args += ["--poisson-blend"]
if COLOR_CORRECTION: args += ["--color-correction"]
exit_code = main(args)
print("Batch exit code:", exit_code)


## 5. Download ZIP

In [ ]:
# @title Download result archive
from google.colab import files
files.download(ZIP_PATH)
